
# 12 — Injected Feature Engineering and Validation

## Overview

This notebook reconstructs the behavioral feature dataset for **Mubsir: AI-Based Detection of Channel-Rotating Low-and-Slow Insider Data Exfiltration**, using the validated count-injected User-Day dataset and the original Data Engineering feature definitions.

It integrates the 30 engineered behavioral features from the previous Clear Attack Simulation and the 27 New Rotation features documented in the ML Feature Generation Contract.

All features must be derived from observable user activity and approved historical behavioral references, without using attack scenario metadata or Ground Truth labels as feature inputs.

## Objectives

1. Identify and validate the original Data Engineering feature sources and definitions.
2. Reconstruct affected behavioral features from the final injected User-Day dataset.
3. Preserve immutable historical baselines from 2010.
4. Reimplement the 30 previously engineered Clear features using their documented formulas.
5. Implement the 27 New Rotation features using the ML team's Feature Generation Contract.
6. Validate workday eligibility, rolling windows, baseline handling, and feature consistency.
7. Generate the binary classification target `y` by matching `(user, day)` with the approved User-Day Ground Truth.
8. Verify dataset integrity, feature correctness, temporal consistency, and absence of target leakage.
9. Export the final model-ready behavioral dataset with its data dictionary, QA reports, and SHA-256 manifest.

## Input Sources

- **Original Data Engineering:** Behavioral feature data, calculation definitions, and historical baseline references.
- **Notebook 10:** `user_day_injected_counts_r62.parquet`
- **Notebook 11:** `ground_truth_user_days_r62.parquet`
- **Clear Attack Simulation:** Definitions of 30 additional behavioral features.
- **ML Feature Generation Contract:** Definitions of 27 New Rotation features.

## Key Constraints

- No modification of original or previously exported datasets.
- No additional attack injection.
- No historical baseline recalculation using 2011 activity.
- No use of scenario identifiers, scheduled channels, stealth settings, or Ground Truth metadata in feature calculations.
- Preserve the documented definitions and edge cases of the original, Clear, and ML feature groups.
- Use only current and past observations when calculating behavioral features.
- Keep `y` strictly separate from predictive inputs during model training.
- Apply the target label only after feature construction.

## Expected Outcome

A validated, model-ready behavioral feature dataset containing the reconstructed original features, approved engineered features, and binary classification target, with complete documentation and quality verification.



## 12.1 — Source Inventory & Feature Contract Review

### Objective

Identify the authoritative feature-engineering inputs and their schemas before reconstructing the injected behavioral dataset.

### Tasks

1. Resolve the project root and approved Notebook 10/11 outputs.
2. Verify the presence and structure of the injected User-Day and Ground Truth datasets.
3. Discover original Data Engineering behavioral feature files.
4. Locate existing baseline references and feature dictionaries.
5. Identify the previous Clear feature dictionary containing the 30 engineered features.
6. Inspect available feature schemas and detect candidate source files.
7. Establish the inputs required by the ML team's 27 New Rotation features.

### Feature Groups

- Original Data Engineering features.
- 30 Clear Attack Simulation features.
- 27 ML New Rotation features.

### Restrictions

- Read-only inventory.
- No feature computation.
- No baseline recalculation.
- No classification label generation.
- No exports.

### Expected Outcome

A verified source inventory identifying the datasets and definitions required for the subsequent feature reconstruction.


In [35]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.1
# SOURCE INVENTORY & FEATURE CONTRACT REVIEW
# READ-ONLY — NO EXPORTS
# ============================================================

from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Resolve project root
# ------------------------------------------------------------

def find_root_121(start):
    current = Path(start).resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "reports"
             / "10_Attack_Injection_Pipeline").is_dir()
            and
            (candidate / "reports"
             / "11_Ground_Truth_and_Quality_Validation").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Cannot locate New_Attack_Simulation root."
    )


ROOT_12 = find_root_121(Path.cwd())
REPORTS_12 = ROOT_12 / "reports"

REPORT_10_12 = (
    REPORTS_12 / "10_Attack_Injection_Pipeline"
)

REPORT_11_12 = (
    REPORTS_12 / "11_Ground_Truth_and_Quality_Validation"
)


def require_121(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.1 FAILED: {message}")


# ------------------------------------------------------------
# STEP 2 — Verify approved exported inputs
# ------------------------------------------------------------

required_inputs_121 = {
    "Injected User-Day":
        REPORT_10_12
        / "user_day_injected_counts_r62.parquet",

    "User-Day Ground Truth":
        REPORT_11_12
        / "ground_truth_user_days_r62.parquet",

    "Channel-Day Ground Truth":
        REPORT_11_12
        / "ground_truth_channel_days_r62.parquet",
}

expected_shapes_121 = {
    "Injected User-Day": (1_394_010, 34),
    "User-Day Ground Truth": (9_080, 42),
    "Channel-Day Ground Truth": (10_124, 27),
}

input_rows_121 = []
input_schemas_121 = {}

for name, path in required_inputs_121.items():

    require_121(
        path.is_file(),
        f"Required input missing: {path}"
    )

    metadata = pq.read_metadata(path)
    schema = pq.read_schema(path)

    shape = (
        metadata.num_rows,
        metadata.num_columns,
    )

    require_121(
        shape == expected_shapes_121[name],
        f"Unexpected dimensions: {name}"
    )

    input_schemas_121[name] = schema.names

    input_rows_121.append({
        "source": name,
        "rows": metadata.num_rows,
        "columns": metadata.num_columns,
        "status": "PASS",
    })

input_inventory_12 = pd.DataFrame(input_rows_121)


# ------------------------------------------------------------
# STEP 3 — Verify required event-count fields
# ------------------------------------------------------------

daily_count_fields_121 = {
    "user",
    "day",
    "is_weekend",
    "logon_count",
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
}

require_121(
    daily_count_fields_121.issubset(
        input_schemas_121["Injected User-Day"]
    ),
    "Missing required daily activity fields."
)

require_121(
    {"user", "day"}.issubset(
        input_schemas_121["User-Day Ground Truth"]
    ),
    "Ground Truth linking keys are missing."
)


# ------------------------------------------------------------
# STEP 4 — Locate original Data Engineering and Clear
# feature resources
#
# Discovery only: do not assume which candidate is approved.
# ------------------------------------------------------------

WORKSPACE_12 = ROOT_12.parent

search_patterns_121 = [
    "*behavior_features*.parquet",
    "*feature_dictionary*.csv",
    "*new_features_dictionary*.csv",
    "*baseline*.parquet",
]

candidate_paths_121 = set()

for pattern in search_patterns_121:
    candidate_paths_121.update(
        WORKSPACE_12.rglob(pattern)
    )

candidate_paths_121 = sorted(
    p for p in candidate_paths_121
    if ".git" not in p.parts
    and ".venv" not in p.parts
    and "site-packages" not in p.parts
)

candidate_rows_121 = []

for path in candidate_paths_121:

    try:
        if path.suffix.lower() == ".parquet":
            metadata = pq.read_metadata(path)
            rows = metadata.num_rows
            columns = metadata.num_columns
        else:
            # Header-only inspection for CSV dictionaries.
            header = pd.read_csv(
                path,
                nrows=0,
                sep=None,
                engine="python",
            )
            rows = None
            columns = len(header.columns)

        candidate_rows_121.append({
            "filename": path.name,
            "parent": path.parent.name,
            "rows": rows,
            "columns": columns,
            "path": str(path),
        })

    except Exception as exc:
        candidate_rows_121.append({
            "filename": path.name,
            "parent": path.parent.name,
            "rows": None,
            "columns": None,
            "path": str(path),
        })

feature_candidates_12 = pd.DataFrame(
    candidate_rows_121,
    columns=[
        "filename",
        "parent",
        "rows",
        "columns",
        "path",
    ],
)


# ------------------------------------------------------------
# STEP 5 — Identify available historical baseline fields
# ------------------------------------------------------------

baseline_columns_121 = [
    "email_count_p50",
    "email_count_p90",
    "email_count_p95",
    "usb_copy_p50",
    "usb_copy_p90",
    "usb_copy_p95",
    "http_upload_p50",
    "http_upload_p90",
    "http_upload_p95",
]

baseline_inventory_121 = []

parquet_candidates_121 = [
    p for p in candidate_paths_121
    if p.suffix.lower() == ".parquet"
]

for path in parquet_candidates_121:

    schema_columns = set(
        pq.read_schema(path).names
    )

    available = [
        col for col in baseline_columns_121
        if col in schema_columns
    ]

    if available:
        baseline_inventory_121.append({
            "filename": path.name,
            "baseline_fields_found": len(available),
            "baseline_fields_missing":
                len(baseline_columns_121) - len(available),
            "rows": pq.read_metadata(path).num_rows,
        })

baseline_inventory_12 = pd.DataFrame(
    baseline_inventory_121,
    columns=[
        "filename",
        "baseline_fields_found",
        "baseline_fields_missing",
        "rows",
    ],
)


# ------------------------------------------------------------
# STEP 6 — Inspect the approved source manifests
# ------------------------------------------------------------

manifest_status_121 = []

for notebook, folder in [
    ("Notebook 10", REPORT_10_12),
    ("Notebook 11", REPORT_11_12),
]:

    path = folder / "export_manifest.json"

    require_121(
        path.is_file(),
        f"{notebook} export manifest missing."
    )

    manifest = json.loads(
        path.read_text(encoding="utf-8")
    )

    require_121(
        manifest.get("status")
        == "INDEPENDENT_VALIDATION_APPROVED",
        f"{notebook} is not approved."
    )

    manifest_status_121.append({
        "notebook": notebook,
        "status": "APPROVED",
    })


# ------------------------------------------------------------
# STEP 7 — Concise inventory results
# ------------------------------------------------------------

print("\n12.1 — APPROVED INPUT DATASETS")
display(input_inventory_12)

print("\nFEATURE SOURCE & DICTIONARY CANDIDATES")
display(feature_candidates_12)

print("\nHISTORICAL BASELINE FIELD AVAILABILITY")
display(baseline_inventory_12)

print("\nSOURCE MANIFEST STATUS")
display(pd.DataFrame(manifest_status_121))

print("\n" + "=" * 58)
print("12.1 — SOURCE INVENTORY")
print("=" * 58)
print("Project root          :", ROOT_12)
print("Injected User-Day     : VERIFIED")
print("Ground Truth sources  : VERIFIED")
print("Notebook 10/11 status : APPROVED")
print("Candidate source files:", len(feature_candidates_12))
print("Features calculated   : NO")
print("Baselines recalculated: NO")
print("Classification y added: NO")
print("Files exported        : NO")
print("=" * 58)



12.1 — APPROVED INPUT DATASETS


,source,rows,columns,status
0,Injected User-Day,1394010,34,PASS
1,User-Day Ground Truth,9080,42,PASS
2,Channel-Day Ground Truth,10124,27,PASS



FEATURE SOURCE & DICTIONARY CANDIDATES


,filename,parent,rows,columns,path
0,behavior_features_injected_final.parquet,data,69634.0,115,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
1,behavior_features_r62_ml_final.parquet,final,387162.0,116,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
2,behavior_features_injected_final.parquet,stages_output,69634.0,115,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
3,feature_dictionary_attack_simulation_r62.csv,dictionary,NaN,12,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
4,new_features_dictionary_30_r62.csv,dictionary,NaN,12,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
5,behavior_features_r62_v1.parquet,features,387162.0,85,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
6,malicious_pre_attack_baseline_r62.parquet,features,5.0,36,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
7,user_baseline_r62_v1.parquet,features,3995.0,36,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
8,feature_dictionary_r62.csv,results,NaN,12,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
9,behavior_features_r62_ml_final_new.parquet,12_Feature_Engineering_and_Validation,387162.0,143,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...



HISTORICAL BASELINE FIELD AVAILABILITY


,filename,baseline_fields_found,baseline_fields_missing,rows
0,behavior_features_injected_final.parquet,9,0,69634
1,behavior_features_r62_ml_final.parquet,9,0,387162
2,behavior_features_injected_final.parquet,9,0,69634
3,behavior_features_r62_v1.parquet,9,0,387162
4,malicious_pre_attack_baseline_r62.parquet,9,0,5
5,user_baseline_r62_v1.parquet,9,0,3995
6,behavior_features_r62_ml_final_new.parquet,9,0,387162
7,injected_behavior_features_checkpoint_v2.parquet,1,8,9819
8,injected_behavior_features_checkpoint_v3.parquet,1,8,9819



SOURCE MANIFEST STATUS


,notebook,status
0,Notebook 10,APPROVED
1,Notebook 11,APPROVED



12.1 — SOURCE INVENTORY
Project root          : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation
Injected User-Day     : VERIFIED
Ground Truth sources  : VERIFIED
Notebook 10/11 status : APPROVED
Candidate source files: 13
Features calculated   : NO
Baselines recalculated: NO
Classification y added: NO
Files exported        : NO



## 12.2 — Feature Schema & Definition Alignment

### Objective

Establish the exact feature schema and source definitions for the new Mubsir behavioral dataset before implementing feature reconstruction.

### Authoritative Sources

- **Data Engineering:** `behavior_features_r62_v1.parquet` — 85 original columns.
- **Historical Baseline:** `user_baseline_r62_v1.parquet`.
- **Clear Attack Simulation:** `new_features_dictionary_30_r62.csv` — 30 engineered features.
- **ML Feature Contract:** `FEATURE_GENERATION_CONTRACT.md` — 27 New Rotation features.

### Validation Scope

1. Verify the original Data Engineering schema and unique User-Day keys.
2. Load and validate all 30 documented Clear feature names.
3. Enumerate the 27 New Rotation features from the approved ML contract.
4. Identify exact column-name overlaps across feature groups.
5. Inspect the historical baseline schema and user coverage.
6. Identify the required daily event-count and historical reference fields.
7. Compare the previous Clear final schema against the reconstructed feature inventory.
8. Identify potential schema conflicts before implementation.

### Methodological Constraints

- Preserve documented feature definitions.
- Do not infer feature equivalence from similar names.
- Preserve historical baselines from 2010.
- Do not include scenario metadata or Ground Truth in feature calculations.
- Do not compute or export features in this section.

### Expected Outcome

A verified feature inventory and schema-alignment report supporting implementation decisions in Section 12.3.


In [36]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.2
# FEATURE SCHEMA & DEFINITION ALIGNMENT
# READ-ONLY — NO FEATURE CALCULATION OR EXPORTS
# ============================================================

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Validation helpers and source resolution
# ------------------------------------------------------------

def require_122(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.2 FAILED: {message}")


def find_unique_122(filename):
    matches = sorted({
        Path(p).resolve()
        for p in candidate_paths_121
        if Path(p).name == filename
    })

    require_122(
        len(matches) == 1,
        f"Expected one source named {filename}; "
        f"found {len(matches)}."
    )

    return matches[0]


original_path_122 = find_unique_122(
    "behavior_features_r62_v1.parquet"
)

baseline_path_122 = find_unique_122(
    "user_baseline_r62_v1.parquet"
)

clear_dictionary_path_122 = find_unique_122(
    "new_features_dictionary_30_r62.csv"
)

clear_final_path_122 = find_unique_122(
    "behavior_features_r62_ml_final.parquet"
)


# ------------------------------------------------------------
# STEP 2 — Load authoritative schemas
# ------------------------------------------------------------

original_columns_122 = pq.read_schema(
    original_path_122
).names

baseline_columns_122 = pq.read_schema(
    baseline_path_122
).names

clear_final_columns_122 = pq.read_schema(
    clear_final_path_122
).names

require_122(
    len(original_columns_122) == 85
    and len(set(original_columns_122)) == 85,
    "Original Data Engineering schema must have 85 unique columns."
)

require_122(
    len(clear_final_columns_122) == 116
    and len(set(clear_final_columns_122)) == 116,
    "Previous Clear final schema must have 116 unique columns."
)

require_122(
    {"user", "day"}.issubset(original_columns_122),
    "Original features are missing User-Day keys."
)

require_122(
    "user" in baseline_columns_122,
    "Baseline dataset is missing user identifiers."
)


# ------------------------------------------------------------
# STEP 3 — Load the documented 30 Clear features
# ------------------------------------------------------------

clear_dictionary_122 = pd.read_csv(
    clear_dictionary_path_122
)

require_122(
    "feature_name" in clear_dictionary_122.columns,
    "Clear dictionary is missing feature_name."
)

clear_features_122 = (
    clear_dictionary_122["feature_name"]
    .astype("string")
    .tolist()
)

require_122(
    len(clear_features_122) == 30
    and len(set(clear_features_122)) == 30
    and all(pd.notna(x) and str(x).strip()
            for x in clear_features_122),
    "Clear feature dictionary must contain 30 unique names."
)


# ------------------------------------------------------------
# STEP 4 — Define the 27 ML New Rotation feature names
#
# These are the exact naming patterns documented in
# FEATURE_GENERATION_CONTRACT.md.
# No formulas are implemented in this section.
# ------------------------------------------------------------

NR_GROUPS_122 = [
    "nr_m",
    "nr_entropy",
    "nr_excl_days",
    "nr_overlap_days",
    "nr_alt",
    "nr_handoff",
    "nr_stealth_acc",
    "nr_gap_alt_2",
]

NR_WINDOWS_122 = (5, 10, 20)

nr_features_122 = [
    f"{group}_{window}wd"
    for group in NR_GROUPS_122
    for window in NR_WINDOWS_122
]

nr_features_122 += [
    "email_count_to_p90_ratio",
    "usb_copy_to_p90_ratio",
    "http_upload_to_p90_ratio",
]

require_122(
    len(nr_features_122) == 27
    and len(set(nr_features_122)) == 27,
    "New Rotation contract must contain 27 distinct features."
)


# ------------------------------------------------------------
# STEP 5 — Detect schema overlaps
# ------------------------------------------------------------

original_set_122 = set(original_columns_122)
clear_set_122 = set(clear_features_122)
nr_set_122 = set(nr_features_122)

overlap_original_clear_122 = sorted(
    original_set_122 & clear_set_122
)

overlap_original_nr_122 = sorted(
    original_set_122 & nr_set_122
)

overlap_clear_nr_122 = sorted(
    clear_set_122 & nr_set_122
)

all_unique_columns_122 = (
    original_set_122 | clear_set_122 | nr_set_122
)

# This is the theoretical combined schema.
# It is not yet an approved final model schema.
theoretical_columns_122 = len(
    all_unique_columns_122
) + int("y" not in all_unique_columns_122)


# ------------------------------------------------------------
# STEP 6 — Inspect historical baseline coverage
# ------------------------------------------------------------

original_keys_122 = pd.read_parquet(
    original_path_122,
    columns=["user", "day"],
)

baseline_users_122 = pd.read_parquet(
    baseline_path_122,
    columns=["user"],
)

original_keys_122["user"] = (
    original_keys_122["user"].astype("string")
)

original_keys_122["day"] = pd.to_datetime(
    original_keys_122["day"]
).dt.normalize()

baseline_users_122["user"] = (
    baseline_users_122["user"].astype("string")
)

require_122(
    len(original_keys_122) == 387_162
    and not original_keys_122.duplicated(
        ["user", "day"]
    ).any(),
    "Original features have invalid User-Day keys."
)

require_122(
    not baseline_users_122["user"].isna().any()
    and not baseline_users_122["user"].duplicated().any(),
    "Historical baseline user keys are invalid."
)

original_users_122 = set(
    original_keys_122["user"].dropna()
)

historical_users_122 = set(
    baseline_users_122["user"].dropna()
)

missing_baseline_users_122 = sorted(
    original_users_122 - historical_users_122
)

require_122(
    not missing_baseline_users_122,
    "Some original ML feature users have no baseline reference."
)

required_baselines_122 = [
    "email_count_p50",
    "email_count_p90",
    "email_count_p95",
    "usb_copy_p50",
    "usb_copy_p90",
    "usb_copy_p95",
    "http_upload_p50",
    "http_upload_p90",
    "http_upload_p95",
]

require_122(
    set(required_baselines_122).issubset(
        baseline_columns_122
    ),
    "Historical baseline is missing required P50/P90/P95 fields."
)


# ------------------------------------------------------------
# STEP 7 — Compare previous Clear final schema
# ------------------------------------------------------------

clear_final_set_122 = set(
    clear_final_columns_122
)

clear_missing_122 = sorted(
    clear_set_122 - clear_final_set_122
)

original_missing_122 = sorted(
    original_set_122 - clear_final_set_122
)

require_122(
    not clear_missing_122 and not original_missing_122,
    "Previous Clear final schema is missing documented columns."
)

clear_y_present_122 = (
    "y" in clear_final_set_122
)


# ------------------------------------------------------------
# STEP 8 — Concise schema report
# ------------------------------------------------------------

feature_inventory_12 = pd.DataFrame([
    {
        "group": "Original Data Engineering",
        "defined_columns": len(original_columns_122),
        "source": original_path_122.name,
    },
    {
        "group": "Clear engineered features",
        "defined_columns": len(clear_features_122),
        "source": clear_dictionary_path_122.name,
    },
    {
        "group": "ML New Rotation features",
        "defined_columns": len(nr_features_122),
        "source": "FEATURE_GENERATION_CONTRACT.md",
    },
])

overlap_inventory_12 = pd.DataFrame([
    {
        "comparison": "Original / Clear",
        "shared_names": len(overlap_original_clear_122),
        "columns": ", ".join(overlap_original_clear_122) or "None",
    },
    {
        "comparison": "Original / New Rotation",
        "shared_names": len(overlap_original_nr_122),
        "columns": ", ".join(overlap_original_nr_122) or "None",
    },
    {
        "comparison": "Clear / New Rotation",
        "shared_names": len(overlap_clear_nr_122),
        "columns": ", ".join(overlap_clear_nr_122) or "None",
    },
])

print("\n12.2 — FEATURE SOURCE INVENTORY")
display(feature_inventory_12)

print("\n12.2 — SCHEMA OVERLAP AUDIT")
display(overlap_inventory_12)

print("\n" + "=" * 60)
print("12.2 — FEATURE SCHEMA ALIGNMENT")
print("=" * 60)
print("Original columns          :", len(original_columns_122))
print("Clear engineered features :", len(clear_features_122))
print("New Rotation features     :", len(nr_features_122))
print("Unique combined names     :", len(all_unique_columns_122))
print("Theoretical columns + y   :", theoretical_columns_122)
print("Previous Clear columns    :", len(clear_final_columns_122))
print("Previous Clear includes y :", clear_y_present_122)
print("Original ML users         :", len(original_users_122))
print("Baseline users            :", len(historical_users_122))
print("Missing ML user baselines :", len(missing_baseline_users_122))
print("Required baseline fields  : 9/9 AVAILABLE")
print("Features calculated       : NO")
print("Baseline recalculated     : NO")
print("Files exported            : NO")
print("Status                    : SCHEMA REVIEW COMPLETE")
print("=" * 60)



12.2 — FEATURE SOURCE INVENTORY


,group,defined_columns,source
0,Original Data Engineering,85,behavior_features_r62_v1.parquet
1,Clear engineered features,30,new_features_dictionary_30_r62.csv
2,ML New Rotation features,27,FEATURE_GENERATION_CONTRACT.md



12.2 — SCHEMA OVERLAP AUDIT


,comparison,shared_names,columns
0,Original / Clear,0,None
1,Original / New Rotation,0,None
2,Clear / New Rotation,0,None



12.2 — FEATURE SCHEMA ALIGNMENT
Original columns          : 85
Clear engineered features : 30
New Rotation features     : 27
Unique combined names     : 142
Theoretical columns + y   : 143
Previous Clear columns    : 116
Previous Clear includes y : True
Original ML users         : 3737
Baseline users            : 3995
Missing ML user baselines : 0
Required baseline fields  : 9/9 AVAILABLE
Features calculated       : NO
Baseline recalculated     : NO
Files exported            : NO
Status                    : SCHEMA REVIEW COMPLETE


In [37]:

# First actual injection day — 400 approved attackers
gt = pd.read_parquet(
    required_inputs_121["User-Day Ground Truth"],
    columns=["user", "day"]
)

gt["day"] = pd.to_datetime(gt["day"]).dt.normalize()

first_attack = gt.groupby("user")["day"].min()

# First 5, 10, 20, and 30 weekdays from Jan 3, 2011
workdays = pd.bdate_range("2011-01-03", "2011-06-01")

assert first_attack.size == 400

results = pd.DataFrame([
    {
        "First N weekdays": n,
        "Through": workdays[n - 1].date(),
        "Attackers started": int(
            (first_attack <= workdays[n - 1]).sum()
        ),
        "Percentage": round(
            (first_attack <= workdays[n - 1]).mean() * 100, 2
        ),
    }
    for n in (5, 10, 20, 30)
])

display(results)


,First N weekdays,Through,Attackers started,Percentage
0,5,2011-01-07,69,17.25
1,10,2011-01-14,100,25.00
2,20,2011-01-28,156,39.00
3,30,2011-02-11,210,52.50



## 12.3 — Baseline Preservation & Partial Workday Preparation

### Objective

Prepare the 2011 eligible-workday sequence for partial-window feature engineering while preserving the original historical baseline without modification.

### Approved Window Policy

All original, Clear, and New Rotation rolling features will use partial windows starting from the first eligible user workday in 2011.

- Window sizes remain feature-specific: 5, 10, 20, or 30 workdays.
- `min_periods=1` is the general rolling-window policy.
- Each window includes the current workday and available preceding eligible workdays.
- No 2010 activity rows will be used as rolling-window context.
- Historical P50, P90, P95, and other baseline references remain unchanged.
- Feature-specific formulas and undefined-value handling will be validated independently.

### Tasks

1. Load the approved historical baseline and original ML reference keys.
2. Verify that historical baseline values match the original ML dataset.
3. Load injected 2011 User-Day activity without changing the source.
4. Evaluate the strict eligible-workday rule.
5. Verify coverage of the original 387,162 ML User-Day keys.
6. Assign chronological workday indices for each user.
7. Determine available history for 5-, 10-, 20-, and 30-workday partial windows.
8. Report any eligibility or population mismatch before feature construction.

### Restrictions

- No baseline recalculation.
- No 2010 activity in rolling windows.
- No feature computation.
- No Ground Truth or target generation.
- No exports.

### Expected Outcome

An approved 2011-only workday sequence and immutable historical baseline references, ready for feature reconstruction using the new partial-window policy.


In [38]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.3
# BASELINE PRESERVATION & PARTIAL WORKDAY PREPARATION
# APPROVED POLICY: 2011 ONLY, PARTIAL WINDOWS
# READ-ONLY — NO EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


def require_123(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.3 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Load immutable historical baseline
# ------------------------------------------------------------

BASELINE_FIELDS_123 = [
    "email_count_p50",
    "email_count_p90",
    "email_count_p95",
    "usb_copy_p50",
    "usb_copy_p90",
    "usb_copy_p95",
    "http_upload_p50",
    "http_upload_p90",
    "http_upload_p95",
]

ACTIVITY_FIELDS_123 = [
    "user",
    "day",
    "is_weekend",
    "logon_count",
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

baseline_12 = pd.read_parquet(
    baseline_path_122,
    columns=["user", *BASELINE_FIELDS_123]
)

baseline_12["user"] = baseline_12["user"].astype("string")

require_123(
    len(baseline_12) == 3995
    and baseline_12["user"].notna().all()
    and not baseline_12["user"].duplicated().any(),
    "Historical baseline user keys are invalid."
)


# ------------------------------------------------------------
# STEP 2 — Verify unchanged baseline values
# ------------------------------------------------------------

original_reference_123 = pd.read_parquet(
    original_path_122,
    columns=["user", "day", *BASELINE_FIELDS_123]
)

original_reference_123["user"] = (
    original_reference_123["user"].astype("string")
)
original_reference_123["day"] = pd.to_datetime(
    original_reference_123["day"]
).dt.normalize()

require_123(
    len(original_reference_123) == 387162
    and not original_reference_123.duplicated(
        ["user", "day"]
    ).any(),
    "Original ML reference keys are invalid."
)

baseline_check_123 = original_reference_123.merge(
    baseline_12,
    on="user",
    how="left",
    suffixes=("_original", "_historical"),
    validate="many_to_one",
    indicator=True,
)

require_123(
    baseline_check_123["_merge"].eq("both").all(),
    "An original ML user lacks historical baseline."
)

baseline_audit_123 = []

for field in BASELINE_FIELDS_123:
    old_values = baseline_check_123[
        f"{field}_original"
    ].to_numpy(dtype="float64")

    historical_values = baseline_check_123[
        f"{field}_historical"
    ].to_numpy(dtype="float64")

    equal = np.isclose(
        old_values,
        historical_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )

    baseline_audit_123.append({
        "field": field,
        "mismatches": int((~equal).sum()),
        "status": "PASS" if equal.all() else "FAIL",
    })

baseline_audit_12 = pd.DataFrame(
    baseline_audit_123
)

require_123(
    baseline_audit_12["status"].eq("PASS").all(),
    "Historical baseline does not match the original ML reference."
)


# ------------------------------------------------------------
# STEP 3 — Load 2011 injected activity only
# ------------------------------------------------------------

daily_123 = pd.read_parquet(
    required_inputs_121["Injected User-Day"],
    columns=ACTIVITY_FIELDS_123,
)

daily_123["user"] = daily_123["user"].astype("string")
daily_123["day"] = pd.to_datetime(
    daily_123["day"]
).dt.normalize()

require_123(
    not daily_123[["user", "day"]].isna().any().any()
    and not daily_123.duplicated(
        ["user", "day"]
    ).any(),
    "Injected User-Day contains invalid keys."
)

future_activity_12 = daily_123.loc[
    daily_123["day"].between(
        "2011-01-01", "2011-06-01"
    )
].copy()

del daily_123

require_123(
    len(future_activity_12) == 395562,
    "Unexpected future-period User-Day population."
)


# ------------------------------------------------------------
# STEP 4 — Build chronological eligible-workday sequence
# ------------------------------------------------------------

require_123(
    future_activity_12[
        ["is_weekend", "logon_count"]
    ].notna().all().all(),
    "Workday eligibility inputs contain missing values."
)

future_activity_12["eligible_workday"] = (
    future_activity_12["is_weekend"].eq(0)
    & future_activity_12["logon_count"].gt(0)
)

workdays_12 = (
    future_activity_12.loc[
        future_activity_12["eligible_workday"]
    ]
    .sort_values(["user", "day"])
    .reset_index(drop=True)
    .copy()
)

workdays_12["workday_number"] = (
    workdays_12.groupby("user").cumcount() + 1
)


# ------------------------------------------------------------
# STEP 5 — Preserve the original ML target population
# ------------------------------------------------------------

ml_target_keys_12 = original_reference_123[
    ["user", "day"]
].copy()

target_check_123 = ml_target_keys_12.merge(
    workdays_12[
        ["user", "day", "workday_number"]
    ],
    on=["user", "day"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

missing_target_123 = target_check_123.loc[
    target_check_123["_merge"].ne("both")
]

extra_workdays_123 = workdays_12[
    ["user", "day"]
].merge(
    ml_target_keys_12,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

extra_workday_count_123 = int(
    extra_workdays_123["_merge"].eq("left_only").sum()
)

population_audit_12 = pd.DataFrame([
    {
        "check": "Original ML target rows",
        "count": len(ml_target_keys_12),
    },
    {
        "check": "2011 eligible workdays",
        "count": len(workdays_12),
    },
    {
        "check": "Missing original ML keys",
        "count": len(missing_target_123),
    },
    {
        "check": "Extra eligible workdays",
        "count": extra_workday_count_123,
    },
])


# ------------------------------------------------------------
# STEP 6 — Inspect partial-window availability
#
# This does not compute behavioral features.
# It only determines how many days are available.
# ------------------------------------------------------------

for window in (5, 10, 20, 30):
    workdays_12[f"available_{window}wd"] = (
        workdays_12["workday_number"].clip(
            upper=window
        )
    )

window_audit_12 = pd.DataFrame([
    {
        "window": f"{window}wd",
        "full_window_rows": int(
            workdays_12["workday_number"].ge(window).sum()
        ),
        "partial_window_rows": int(
            workdays_12["workday_number"].lt(window).sum()
        ),
        "minimum_available_days": int(
            workdays_12[f"available_{window}wd"].min()
        ),
    }
    for window in (5, 10, 20, 30)
])


# ------------------------------------------------------------
# STEP 7 — Validate chronological preparation
# ------------------------------------------------------------

require_123(
    workdays_12["day"].ge(
        pd.Timestamp("2011-01-01")
    ).all(),
    "2010 activity entered the rolling-workday sequence."
)

require_123(
    not workdays_12.duplicated(
        ["user", "day"]
    ).any(),
    "Duplicate eligible workday keys."
)

require_123(
    workdays_12["workday_number"].ge(1).all(),
    "Invalid workday indices."
)

require_123(
    all(
        workdays_12[f"available_{window}wd"]
        .between(1, window).all()
        for window in (5, 10, 20, 30)
    ),
    "Invalid partial-window availability."
)


# ------------------------------------------------------------
# STEP 8 — Report and fail closed on population mismatch
# ------------------------------------------------------------

print("\n12.3 — HISTORICAL BASELINE VERIFICATION")
display(baseline_audit_12)

print("\n12.3 — ML POPULATION RECONCILIATION")
display(population_audit_12)

print("\n12.3 — PARTIAL-WINDOW AVAILABILITY")
display(window_audit_12)

print("\n" + "=" * 58)
print("12.3 — PARTIAL-WINDOW SOURCE PREPARATION")
print("=" * 58)
print("Window policy         : PARTIAL")
print("Rolling history begins: 2011-01-01")
print("Minimum periods       : 1")
print("Baseline recalculated : NO")
print("2010 activity used    : NO")
print("Feature computation   : NO")
print("Ground Truth used     : NO")
print("Files exported        : NO")
print("ML target rows        :", len(ml_target_keys_12))
print("Eligible workdays     :", len(workdays_12))
print("Missing ML target keys:", len(missing_target_123))
print("Extra eligible days   :", extra_workday_count_123)
print("=" * 58)

require_123(
    len(missing_target_123) == 0,
    "Some original ML target keys are not eligible workdays."
)

print("12.3 STATUS: PASS — READY FOR FEATURE RECONSTRUCTION")




12.3 — HISTORICAL BASELINE VERIFICATION


,field,mismatches,status
0,email_count_p50,0,PASS
1,email_count_p90,0,PASS
2,email_count_p95,0,PASS
3,usb_copy_p50,0,PASS
4,usb_copy_p90,0,PASS
5,usb_copy_p95,0,PASS
6,http_upload_p50,0,PASS
7,http_upload_p90,0,PASS
8,http_upload_p95,0,PASS



12.3 — ML POPULATION RECONCILIATION


,check,count
0,Original ML target rows,387162
1,2011 eligible workdays,387437
2,Missing original ML keys,0
3,Extra eligible workdays,275



12.3 — PARTIAL-WINDOW AVAILABILITY


,window,full_window_rows,partial_window_rows,minimum_available_days
0,5wd,372480,14957,1
1,10wd,353804,33633,1
2,20wd,316514,70923,1
3,30wd,279334,108103,1



12.3 — PARTIAL-WINDOW SOURCE PREPARATION
Window policy         : PARTIAL
Rolling history begins: 2011-01-01
Minimum periods       : 1
Baseline recalculated : NO
2010 activity used    : NO
Feature computation   : NO
Ground Truth used     : NO
Files exported        : NO
ML target rows        : 387162
Eligible workdays     : 387437
Missing ML target keys: 0
Extra eligible days   : 275
12.3 STATUS: PASS — READY FOR FEATURE RECONSTRUCTION



## 12.4 — Original Behavioral Feature Reconstruction & Independent Validation

### Objective

Reconstruct the original 85-column Data Engineering dataset using the validated injected event counts while preserving the original historical baselines, user population, and unaffected attributes.

### Approved Methodology

- Use the original 387,162 User-Day keys.
- Replace only the eight approved injected count columns.
- Recalculate all dependent daily indicators and rolling features.
- Use partial rolling windows (`min_periods=1`) beginning in 2011.
- Calculate windows independently for each user.
- Preserve historical baseline values without modification.
- Preserve the original 85-column schema and order.

### Independent Validation

First reconstruct the original derived features using the original, non-injected daily counts and compare them against the stored Data Engineering values.

Then reconstruct the injected version using the exact same calculations.

Verify:

1. Original reconstruction matches Data Engineering.
2. All 85 columns retain the original schema.
3. Historical baseline fields remain identical.
4. All 51 unaffected columns remain unchanged.
5. Changes occur only within the 34 permitted columns.
6. The 387,162 User-Day keys are preserved.
7. No excluded malicious users reappear.
8. Rolling windows are causal and use only eligible days from 2011.
9. No Ground Truth or target information enters feature calculations.

### Output

An in-memory reconstructed 85-column dataset and comprehensive per-column validation reports.

No files are exported.


In [39]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.4
# ORIGINAL FEATURES RECONSTRUCTION + INDEPENDENT QA
# POLICY: PARTIAL WINDOWS FROM 2011
# READ-ONLY SOURCES — NO EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


def require_124(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.4 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Approved column dependencies
# ------------------------------------------------------------

COUNT_FIELDS_124 = [
    "external_email_count",
    "email_send_count",
    "email_event_count",
    "usb_copy_count",
    "file_copy_count",
    "file_event_count",
    "www_upload_count",
    "http_event_count",
]

DAILY_FIELDS_124 = [
    "email_active_today",
    "usb_active_today",
    "http_upload_active_today",
    "active_channel_count",
    "email_count_above_p95",
    "email_bytes_above_p95",
    "usb_copy_above_p95",
    "http_upload_above_p95",
    "email_count_to_p95_ratio",
    "email_bytes_to_p95_ratio",
    "usb_copy_to_p95_ratio",
    "http_upload_to_p95_ratio",
]

ROLLING_SOURCES_124 = {
    "email_active_days": "email_active_today",
    "usb_active_days": "usb_active_today",
    "http_active_days": "http_upload_active_today",
}

ROLLING_FIELDS_124 = []

for prefix in ROLLING_SOURCES_124:
    for w in (5, 10, 20):
        ROLLING_FIELDS_124.append(f"{prefix}_{w}wd")

for prefix in (
    "external_email_count",
    "usb_copy_count",
    "http_upload_count",
):
    for w in (5, 10):
        ROLLING_FIELDS_124.append(f"{prefix}_{w}wd")

ROLLING_FIELDS_124.append("multi_channel_days_10wd")

AFFECTED_DAILY_124 = sorted(
    set(DAILY_FIELDS_124)
    - {
        "email_bytes_above_p95",
        "email_bytes_to_p95_ratio",
    }
)

PERMITTED_CHANGES_124 = (
    set(COUNT_FIELDS_124)
    | set(AFFECTED_DAILY_124)
    | set(ROLLING_FIELDS_124)
)

ALL_ORIGINAL_COLUMNS_124 = list(original_columns_122)

UNCHANGED_FIELDS_124 = sorted(
    set(ALL_ORIGINAL_COLUMNS_124)
    - PERMITTED_CHANGES_124
)

require_124(
    len(COUNT_FIELDS_124) == 8
    and len(AFFECTED_DAILY_124) == 10
    and len(ROLLING_FIELDS_124) == 16
    and len(PERMITTED_CHANGES_124) == 34
    and len(UNCHANGED_FIELDS_124) == 51,
    "Expected 34 affected and 51 unaffected columns."
)

require_124(
    set(COUNT_FIELDS_124 + DAILY_FIELDS_124 + ROLLING_FIELDS_124)
    .issubset(ALL_ORIGINAL_COLUMNS_124),
    "Some documented feature names are missing."
)


# ------------------------------------------------------------
# STEP 2 — Load original and approved injected inputs
# ------------------------------------------------------------

original_124 = pd.read_parquet(original_path_122)

injected_counts_124 = pd.read_parquet(
    required_inputs_121["Injected User-Day"],
    columns=["user", "day", *COUNT_FIELDS_124],
)

for df in (original_124, injected_counts_124):
    df["user"] = df["user"].astype("string")
    df["day"] = pd.to_datetime(df["day"]).dt.normalize()

require_124(
    original_124.shape == (387162, 85),
    "Original ML dataset has unexpected dimensions."
)

require_124(
    not original_124[["user", "day"]].isna().any().any()
    and not original_124.duplicated(["user", "day"]).any()
    and not injected_counts_124.duplicated(["user", "day"]).any(),
    "Source User-Day keys are invalid."
)

original_124 = (
    original_124.sort_values(["user", "day"])
    .reset_index(drop=True)
)

joined_124 = original_124[["user", "day"]].merge(
    injected_counts_124,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

require_124(
    joined_124["_merge"].eq("both").all(),
    "Some ML keys are missing from injected User-Day."
)

require_124(
    joined_124[COUNT_FIELDS_124].notna().all().all(),
    "Injected event counts contain NULLs on ML rows."
)

for field in COUNT_FIELDS_124:
    values = joined_124[field].to_numpy(dtype="float64")
    require_124(
        np.isfinite(values).all()
        and (values >= 0).all()
        and np.equal(values, np.floor(values)).all(),
        f"Invalid injected event counts: {field}"
    )


# ------------------------------------------------------------
# STEP 3 — Reusable feature reconstruction function
#
# Used independently on original and injected counts.
# ------------------------------------------------------------

def reconstruct_features_124(source):

    df = source.copy()
    df = df.sort_values(["user", "day"]).reset_index(drop=True)

    require_124(
        df["day"].between(
            pd.Timestamp("2011-01-01"),
            pd.Timestamp("2011-06-01")
        ).all(),
        "Rolling input includes dates outside the 2011 period."
    )

    require_124(
        df["is_weekend"].eq(0).all()
        and df["logon_count"].gt(0).all(),
        "Rolling input includes ineligible workdays."
    )

    # Daily channel activity
    df["email_active_today"] = (
        df["external_email_count"] > 0
    ).astype("int64")

    df["usb_active_today"] = (
        df["usb_copy_count"] > 0
    ).astype("int64")

    df["http_upload_active_today"] = (
        df["www_upload_count"] > 0
    ).astype("int64")

    df["active_channel_count"] = (
        df["email_active_today"]
        + df["usb_active_today"]
        + df["http_upload_active_today"]
    )

    # Historical P95 comparisons
    comparisons = [
        ("email_count", "external_email_count",
         "email_count_p95"),
        ("email_bytes", "external_email_bytes",
         "email_bytes_p95"),
        ("usb_copy", "usb_copy_count",
         "usb_copy_p95"),
        ("http_upload", "www_upload_count",
         "http_upload_p95"),
    ]

    for prefix, count_field, p95_field in comparisons:

        df[f"{prefix}_above_p95"] = (
            df[p95_field].notna()
            & df[count_field].gt(df[p95_field])
        ).astype("int64")

        denominator = df[p95_field].where(
            df[p95_field].gt(0)
        )

        df[f"{prefix}_to_p95_ratio"] = (
            df[count_field] / denominator
        )

    # 2011-only partial rolling workdays
    def roll_sum(field, window):
        return (
            df.groupby("user", sort=False)[field]
            .transform(
                lambda s: s.rolling(
                    window=window,
                    min_periods=1
                ).sum()
            )
        )

    for prefix, source_field in ROLLING_SOURCES_124.items():
        for window in (5, 10, 20):
            df[f"{prefix}_{window}wd"] = roll_sum(
                source_field, window
            )

    for prefix, source_field in [
        ("external_email_count", "external_email_count"),
        ("usb_copy_count", "usb_copy_count"),
        ("http_upload_count", "www_upload_count"),
    ]:
        for window in (5, 10):
            df[f"{prefix}_{window}wd"] = roll_sum(
                source_field, window
            )

    df["multi_channel_today_124"] = (
        df["active_channel_count"] >= 2
    ).astype("int64")

    df["multi_channel_days_10wd"] = roll_sum(
        "multi_channel_today_124", 10
    )

    df = df.drop(columns=["multi_channel_today_124"])

    return df[ALL_ORIGINAL_COLUMNS_124]


# ------------------------------------------------------------
# STEP 4 — Independent pre-injection reconstruction
#
# Prove that our implementation reproduces Ritajs
# stored Data Engineering features before injection.
# ------------------------------------------------------------

original_recalculated_124 = reconstruct_features_124(
    original_124
)


def equal_values_124(left, right):
    a = pd.to_numeric(left, errors="raise").to_numpy(
        dtype="float64"
    )
    b = pd.to_numeric(right, errors="raise").to_numpy(
        dtype="float64"
    )
    return np.isclose(
        a, b,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True
    )


original_formula_qa_124 = pd.DataFrame([
    {
        "feature": field,
        "mismatches": int(
            (~equal_values_124(
                original_124[field],
                original_recalculated_124[field]
            )).sum()
        ),
    }
    for field in DAILY_FIELDS_124 + ROLLING_FIELDS_124
])

print("\nORIGINAL DATA ENGINEERING FORMULA VERIFICATION")
display(original_formula_qa_124)

require_124(
    original_formula_qa_124["mismatches"].eq(0).all(),
    "Our formulas do not reproduce the original Data "
    "Engineering results. Stop before injection."
)


# ------------------------------------------------------------
# STEP 5 — Reconstruct using approved injected counts
# ------------------------------------------------------------

injected_source_124 = original_124.copy()

for field in COUNT_FIELDS_124:
    injected_source_124[field] = joined_124[field].to_numpy()

reconstructed_original_12 = reconstruct_features_124(
    injected_source_124
)

require_124(
    reconstructed_original_12.shape == (387162, 85),
    "Reconstructed dimensions are invalid."
)

require_124(
    list(reconstructed_original_12.columns)
    == ALL_ORIGINAL_COLUMNS_124,
    "Original column schema or order changed."
)

require_124(
    reconstructed_original_12[["user", "day"]]
    .equals(original_124[["user", "day"]]),
    "Original ML User-Day keys changed."
)


# ------------------------------------------------------------
# STEP 6 — Compare every one of the 85 columns
# ------------------------------------------------------------

difference_rows_124 = []

for field in ALL_ORIGINAL_COLUMNS_124:

    if field in ("user", "day"):
        matched = (
            original_124[field].eq(
                reconstructed_original_12[field]
            )
            | (
                original_124[field].isna()
                & reconstructed_original_12[field].isna()
            )
        ).to_numpy()

    elif (
        pd.api.types.is_numeric_dtype(original_124[field])
        and pd.api.types.is_numeric_dtype(
            reconstructed_original_12[field]
        )
    ):
        matched = equal_values_124(
            original_124[field],
            reconstructed_original_12[field]
        )

    else:
        matched = (
            original_124[field].eq(
                reconstructed_original_12[field]
            )
            | (
                original_124[field].isna()
                & reconstructed_original_12[field].isna()
            )
        ).to_numpy()

    difference_rows_124.append({
        "column": field,
        "group": (
            "Permitted"
            if field in PERMITTED_CHANGES_124
            else "Unchanged"
        ),
        "changed_rows": int((~matched).sum()),
        "status": (
            "PASS"
            if field in PERMITTED_CHANGES_124
            or matched.all()
            else "FAIL"
        ),
    })

column_difference_audit_12 = pd.DataFrame(
    difference_rows_124
)

require_124(
    column_difference_audit_12["status"].eq("PASS").all(),
    "An unauthorized column changed."
)


# ------------------------------------------------------------
# STEP 7 — Strict historical baseline preservation
# ------------------------------------------------------------

BASELINE_COLUMNS_124 = [
    col for col in ALL_ORIGINAL_COLUMNS_124
    if col in baseline_columns_122
    and col != "user"
]

baseline_qa_124 = []

for field in BASELINE_COLUMNS_124:

    before = original_124[field]
    after = reconstructed_original_12[field]

    require_124(
        before.equals(after),
        f"Historical baseline modified: {field}"
    )

    baseline_qa_124.append({
        "baseline_field": field,
        "mismatches": 0,
        "status": "PASS",
    })

baseline_qa_12 = pd.DataFrame(baseline_qa_124)


# ------------------------------------------------------------
# STEP 8 — Excluded-user and population validation
# ------------------------------------------------------------

EXCLUDED_USERS_124 = {
    "CDE1846",
    "MBG3183",
    "CMP2946",
}

require_124(
    not reconstructed_original_12["user"].isin(
        EXCLUDED_USERS_124
    ).any(),
    "Excluded users reappeared."
)

require_124(
    reconstructed_original_12["user"].nunique() == 3737,
    "Original ML user population changed."
)

require_124(
    not reconstructed_original_12.duplicated(
        ["user", "day"]
    ).any(),
    "Duplicate reconstructed keys."
)

require_124(
    "y" not in reconstructed_original_12.columns,
    "Classification target appeared before labeling."
)


# ------------------------------------------------------------
# STEP 9 — Final audit results
# ------------------------------------------------------------

print("\n12.4 — BASELINE PRESERVATION")
display(baseline_qa_12)

print("\n12.4 — ACTUAL CHANGES IN ALL 85 COLUMNS")
display(column_difference_audit_12)

changed_permitted_124 = int(
    (
        (column_difference_audit_12["group"] == "Permitted")
        & (column_difference_audit_12["changed_rows"] > 0)
    ).sum()
)

changed_unexpected_124 = int(
    (
        (column_difference_audit_12["group"] == "Unchanged")
        & (column_difference_audit_12["changed_rows"] > 0)
    ).sum()
)

summary_124 = pd.DataFrame([
    {"check": "Final rows", "result": 387162},
    {"check": "Final columns", "result": 85},
    {"check": "Users", "result": 3737},
    {"check": "Permitted change columns", "result": 34},
    {"check": "Protected columns", "result": 51},
    {"check": "Actually changed columns",
     "result": changed_permitted_124},
    {"check": "Unexpected changed columns",
     "result": changed_unexpected_124},
    {"check": "Original formula mismatches",
     "result": int(original_formula_qa_124["mismatches"].sum())},
    {"check": "Baseline modifications", "result": 0},
    {"check": "Files exported", "result": 0},
])

print("\n12.4 — FINAL VALIDATION SUMMARY")
display(summary_124)

print("\n12.4 STATUS: PASS — RECONSTRUCTION APPROVED")



ORIGINAL DATA ENGINEERING FORMULA VERIFICATION


,feature,mismatches
0,email_active_today,0
1,usb_active_today,0
2,http_upload_active_today,0
3,active_channel_count,0
4,email_count_above_p95,0
5,email_bytes_above_p95,0
6,usb_copy_above_p95,0
7,http_upload_above_p95,0
8,email_count_to_p95_ratio,0
9,email_bytes_to_p95_ratio,0



12.4 — BASELINE PRESERVATION


,baseline_field,mismatches,status
0,baseline_workdays,0,PASS
1,email_active_days,0,PASS
2,email_active_rate,0,PASS
3,email_count_p50,0,PASS
4,email_count_p90,0,PASS
5,email_count_p95,0,PASS
6,email_count_mad,0,PASS
7,email_bytes_p50,0,PASS
8,email_bytes_p90,0,PASS
9,email_bytes_p95,0,PASS



12.4 — ACTUAL CHANGES IN ALL 85 COLUMNS


,column,group,changed_rows,status
0,user,Unchanged,0,PASS
1,day,Unchanged,0,PASS
2,day_of_week,Unchanged,0,PASS
3,is_weekend,Unchanged,0,PASS
4,email_event_count,Permitted,6703,PASS
...,...,...,...,...
80,http_upload_count_5wd,Permitted,5210,PASS
81,external_email_count_10wd,Permitted,13015,PASS
82,usb_copy_count_10wd,Permitted,7465,PASS
83,http_upload_count_10wd,Permitted,7548,PASS



12.4 — FINAL VALIDATION SUMMARY


,check,result
0,Final rows,387162
1,Final columns,85
2,Users,3737
3,Permitted change columns,34
4,Protected columns,51
5,Actually changed columns,20
6,Unexpected changed columns,0
7,Original formula mismatches,0
8,Baseline modifications,0
9,Files exported,0



12.4 STATUS: PASS — RECONSTRUCTION APPROVED



## 12.4.1 — Baseline Protection & Actual Change Investigation

### Objective

Independently investigate why only 20 of the 34 injection-dependent columns changed, and verify that historical baseline protection was correctly scoped.

### Audit Scope

1. Inspect the columns included in the previous baseline protection test.
2. Identify any overlap between baseline-protected columns and injection-dependent columns.
3. Independently verify preservation of the nine authoritative historical P50/P90/P95 fields.
4. Identify all 20 changed and 14 unchanged injection-dependent columns.
5. Examine whether the unchanged columns received changed input counts but produced unchanged feature values.
6. Verify that all 51 protected columns remain unchanged.
7. Preserve all datasets in memory without recomputation or export.

### Acceptance Criteria

- No changes to historical reference fields.
- No changes outside the 34 permitted columns.
- All 34 columns accounted for.
- Observed changes explained by input dependencies and feature behavior.
- No source files modified.


In [40]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.4.1
# BASELINE PROTECTION & 34-COLUMN CHANGE INVESTIGATION
# READ-ONLY — NO RECOMPUTATION OR EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


def require_1241(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.4.1 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Verify the previous baseline audit scope
# ------------------------------------------------------------

previous_baseline_scope_1241 = sorted(
    set(BASELINE_COLUMNS_124)
)

historical_fields_1241 = list(BASELINE_FIELDS_123)

baseline_overlap_1241 = sorted(
    set(previous_baseline_scope_1241)
    & PERMITTED_CHANGES_124
)

scope_audit_1241 = pd.DataFrame([
    {
        "check": "Previously protected shared columns",
        "count": len(previous_baseline_scope_1241),
    },
    {
        "check": "Authoritative P50/P90/P95 fields",
        "count": len(historical_fields_1241),
    },
    {
        "check": "Protected / permitted overlap",
        "count": len(baseline_overlap_1241),
    },
])

print("\nA — BASELINE AUDIT SCOPE")
display(scope_audit_1241)

print("Overlapping columns:", baseline_overlap_1241)

require_1241(
    not baseline_overlap_1241,
    "Previous baseline protection includes injection-dependent "
    "columns. The audit scope must be corrected."
)


# ------------------------------------------------------------
# STEP 2 — Independent historical baseline verification
# ------------------------------------------------------------

historical_qa_1241 = []

for field in historical_fields_1241:

    before = original_124[field]
    after = reconstructed_original_12[field]

    same = before.equals(after)

    historical_qa_1241.append({
        "field": field,
        "unchanged": same,
        "status": "PASS" if same else "FAIL",
    })

historical_qa_1241 = pd.DataFrame(
    historical_qa_1241
)

print("\nB — IMMUTABLE HISTORICAL BASELINE")
display(historical_qa_1241)

require_1241(
    historical_qa_1241["status"].eq("PASS").all(),
    "Historical baseline reference values changed."
)


# ------------------------------------------------------------
# STEP 3 — Identify exact actual changes in all 34 columns
# ------------------------------------------------------------

require_1241(
    set(column_difference_audit_12["column"])
    == set(original_columns_122),
    "The previous column audit did not cover all 85 columns."
)

require_1241(
    set(PERMITTED_CHANGES_124).issubset(
        set(original_columns_122)
    ),
    "Permitted change columns are missing."
)

affected_audit_1241 = (
    column_difference_audit_12.loc[
        column_difference_audit_12["column"].isin(
            PERMITTED_CHANGES_124
        ),
        ["column", "changed_rows"]
    ]
    .copy()
)

affected_audit_1241["changed"] = (
    affected_audit_1241["changed_rows"] > 0
)

require_1241(
    len(affected_audit_1241) == 34,
    "Expected exactly 34 injection-dependent columns."
)


# ------------------------------------------------------------
# STEP 4 — Trace each feature to its injected input counts
# ------------------------------------------------------------

count_dependencies_1241 = {
    field: [field]
    for field in COUNT_FIELDS_124
}

count_dependencies_1241.update({
    "email_active_today": ["external_email_count"],
    "usb_active_today": ["usb_copy_count"],
    "http_upload_active_today": ["www_upload_count"],
    "active_channel_count": [
        "external_email_count",
        "usb_copy_count",
        "www_upload_count",
    ],
    "email_count_above_p95": ["external_email_count"],
    "usb_copy_above_p95": ["usb_copy_count"],
    "http_upload_above_p95": ["www_upload_count"],
    "email_count_to_p95_ratio": ["external_email_count"],
    "usb_copy_to_p95_ratio": ["usb_copy_count"],
    "http_upload_to_p95_ratio": ["www_upload_count"],
})

for channel, count in [
    ("email", "external_email_count"),
    ("usb", "usb_copy_count"),
    ("http", "www_upload_count"),
]:
    for w in (5, 10, 20):
        count_dependencies_1241[
            f"{channel}_active_days_{w}wd"
        ] = [count]

for name, count in [
    ("external_email", "external_email_count"),
    ("usb_copy", "usb_copy_count"),
    ("http_upload", "www_upload_count"),
]:
    for w in (5, 10):
        count_dependencies_1241[
            f"{name}_count_{w}wd"
        ] = [count]

count_dependencies_1241[
    "multi_channel_days_10wd"
] = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

require_1241(
    set(count_dependencies_1241)
    == PERMITTED_CHANGES_124,
    "Not all 34 features have documented dependencies."
)


# ------------------------------------------------------------
# STEP 5 — Count changes in each source activity field
# ------------------------------------------------------------

source_change_masks_1241 = {}

for count in COUNT_FIELDS_124:

    source_change_masks_1241[count] = (
        ~equal_values_124(
            original_124[count],
            reconstructed_original_12[count]
        )
    )

source_change_qa_1241 = pd.DataFrame([
    {
        "count_field": count,
        "changed_user_days": int(mask.sum()),
    }
    for count, mask in source_change_masks_1241.items()
])

print("\nC — DIRECTLY INJECTED COUNT CHANGES")
display(source_change_qa_1241)


# ------------------------------------------------------------
# STEP 6 — Explain unchanged feature columns
# ------------------------------------------------------------

def feature_group_1241(name):
    if name in COUNT_FIELDS_124:
        return "Direct injected count"
    if name in AFFECTED_DAILY_124:
        return "Daily behavioral indicator"
    if name in ROLLING_FIELDS_124:
        return "Rolling behavioral feature"
    return "Unknown"


def explanation_1241(feature, input_changed):
    if not input_changed:
        return "No changes in underlying count inputs"

    if feature.endswith("_active_today"):
        return "Activity remained active or inactive"

    if feature == "active_channel_count":
        return "Number of active channels remained unchanged"

    if feature.endswith("_above_p95"):
        return "P95 threshold outcome remained unchanged"

    if feature.endswith("_to_p95_ratio"):
        return "Ratio unchanged or undefined for its baseline"

    if "_active_days_" in feature:
        return "Active-day status did not change"

    if feature == "multi_channel_days_10wd":
        return "Multi-channel day status did not change"

    if feature in ROLLING_FIELDS_124:
        return "Rolling result unchanged despite input changes"

    return "Check source change distribution"


investigation_rows_1241 = []

for row in affected_audit_1241.itertuples(index=False):

    inputs = count_dependencies_1241[row.column]

    input_mask = np.zeros(
        len(original_124),
        dtype=bool
    )

    for count in inputs:
        input_mask |= source_change_masks_1241[count]

    input_change_count = int(input_mask.sum())

    investigation_rows_1241.append({
        "column": row.column,
        "group": feature_group_1241(row.column),
        "changed_rows": int(row.changed_rows),
        "source_changed_rows": input_change_count,
        "status": "CHANGED" if row.changed else "UNCHANGED",
        "possible_explanation": (
            "Feature values changed"
            if row.changed
            else explanation_1241(
                row.column,
                input_change_count > 0
            )
        ),
    })

investigation_1241 = pd.DataFrame(
    investigation_rows_1241
)

changed_features_1241 = investigation_1241.loc[
    investigation_1241["status"] == "CHANGED"
]

unchanged_features_1241 = investigation_1241.loc[
    investigation_1241["status"] == "UNCHANGED"
]

print("\nD — FEATURES THAT ACTUALLY CHANGED")
display(changed_features_1241)

print("\nE — FEATURES THAT DID NOT CHANGE")
display(unchanged_features_1241)


# ------------------------------------------------------------
# STEP 7 — Independently confirm protected columns
# ------------------------------------------------------------

protected_qa_1241 = column_difference_audit_12.loc[
    column_difference_audit_12["column"].isin(
        UNCHANGED_FIELDS_124
    )
]

require_1241(
    len(protected_qa_1241) == 51
    and protected_qa_1241["changed_rows"].eq(0).all(),
    "At least one of the 51 protected columns changed."
)


# ------------------------------------------------------------
# STEP 8 — Final audit summary
# ------------------------------------------------------------

summary_1241 = pd.DataFrame([
    {
        "check": "Historically fixed reference fields",
        "result": len(historical_fields_1241),
    },
    {
        "check": "Baseline mismatches",
        "result": 0,
    },
    {
        "check": "Baseline/permitted overlap",
        "result": len(baseline_overlap_1241),
    },
    {
        "check": "Injection-dependent columns",
        "result": len(investigation_1241),
    },
    {
        "check": "Actually changed columns",
        "result": len(changed_features_1241),
    },
    {
        "check": "Unchanged dependent columns",
        "result": len(unchanged_features_1241),
    },
    {
        "check": "Unexpected protected-column changes",
        "result": int(protected_qa_1241["changed_rows"].sum()),
    },
    {
        "check": "Files exported",
        "result": 0,
    },
])

print("\nF — 12.4.1 FINAL AUDIT SUMMARY")
display(summary_1241)

require_1241(
    len(changed_features_1241) == 20
    and len(unchanged_features_1241) == 14,
    "Actual change distribution differs from previous 12.4 results."
)

print("\n12.4.1 STATUS: PASS — INVESTIGATION COMPLETE")



A — BASELINE AUDIT SCOPE


,check,count
0,Previously protected shared columns,23
1,Authoritative P50/P90/P95 fields,9
2,Protected / permitted overlap,0


Overlapping columns: []

B — IMMUTABLE HISTORICAL BASELINE


,field,unchanged,status
0,email_count_p50,True,PASS
1,email_count_p90,True,PASS
2,email_count_p95,True,PASS
3,usb_copy_p50,True,PASS
4,usb_copy_p90,True,PASS
5,usb_copy_p95,True,PASS
6,http_upload_p50,True,PASS
7,http_upload_p90,True,PASS
8,http_upload_p95,True,PASS



C — DIRECTLY INJECTED COUNT CHANGES


,count_field,changed_user_days
0,external_email_count,6703
1,email_send_count,6703
2,email_event_count,6703
3,usb_copy_count,1935
4,file_copy_count,1935
5,file_event_count,1935
6,www_upload_count,1486
7,http_event_count,1486


... (3 outputs left out)

,column,group,changed_rows,source_changed_rows,status,possible_explanation
8,email_active_today,Daily behavioral indicator,0,6703,UNCHANGED,Activity remained active or inactive
9,usb_active_today,Daily behavioral indicator,0,1935,UNCHANGED,Activity remained active or inactive
10,http_upload_active_today,Daily behavioral indicator,0,1486,UNCHANGED,Activity remained active or inactive
11,active_channel_count,Daily behavioral indicator,0,9080,UNCHANGED,Number of active channels remained unchanged
18,email_active_days_5wd,Rolling behavioral feature,0,6703,UNCHANGED,Active-day status did not change
19,email_active_days_10wd,Rolling behavioral feature,0,6703,UNCHANGED,Active-day status did not change
20,email_active_days_20wd,Rolling behavioral feature,0,6703,UNCHANGED,Active-day status did not change
21,usb_active_days_5wd,Rolling behavioral feature,0,1935,UNCHANGED,Active-day status did not change
22,usb_active_days_10wd,Rolling behavioral feature,0,1935,UNCHANGED,Active-day status did not change
23,usb_active_days_20wd,Rolling behavioral feature,0,1935,UNCHANGED,Active-day status did not change



F — 12.4.1 FINAL AUDIT SUMMARY


,check,result
0,Historically fixed reference fields,9
1,Baseline mismatches,0
2,Baseline/permitted overlap,0
3,Injection-dependent columns,34
4,Actually changed columns,20
5,Unchanged dependent columns,14
6,Unexpected protected-column changes,0
7,Files exported,0



12.4.1 STATUS: PASS — INVESTIGATION COMPLETE



## 12.5.1 — Clear Feature Inputs & Formula Readiness

### Objective

Verify all inputs required to reconstruct the 30 Clear behavioral features using the approved injected User-Day dataset.

### Feature Groups

1. Cumulative Deviation — 4 features.
2. Combined Channel Intensity — 4 features.
3. Dominant Channel Switches — 4 features.
4. Channel Distribution Shift — 4 features.
5. Consecutive Elevated Days — 4 features.
6. Cross-Channel Deviation Days — 4 features.
7. Additional Rolling Event Counts — 6 features.

### Approved Policy

- Use the reconstructed 85-column dataset from Section 12.4.
- Calculate partial windows of 5, 10, 20, and 30 eligible workdays.
- Start all rolling calculations in 2011.
- Include the current day.
- Preserve the documented Clear formulas, including baseline validity and dominant-channel tie handling.
- Use the original historical baseline without recalculation.
- Do not use attack labels, scheduled channels, or Ground Truth.

### Validation

Check feature names, daily count fields, baseline reference fields, historical distribution fields, and source population.

### Expected Outcome

A validated input contract for implementing the 30 Clear features.

No feature computation or file exports.


In [41]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.5.1
# CLEAR 30 — INPUT & FORMULA READINESS
# READ-ONLY — NO EXPORTS
# ============================================================

import pandas as pd
from IPython.display import display


def require_1251(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.5.1 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Validate reconstructed source
# ------------------------------------------------------------

require_1251(
    reconstructed_original_12.shape == (387162, 85),
    "Section 12.4 reconstructed dataset is invalid."
)

require_1251(
    "y" not in reconstructed_original_12.columns,
    "Classification label must not enter feature engineering."
)

require_1251(
    not reconstructed_original_12.duplicated(
        ["user", "day"]
    ).any(),
    "Duplicate User-Day keys."
)


# ------------------------------------------------------------
# STEP 2 — Inspect official Clear feature dictionary
# ------------------------------------------------------------

clear_dict_1251 = pd.read_csv(
    clear_dictionary_path_122
)

require_1251(
    "feature_name" in clear_dict_1251.columns,
    "Clear dictionary is missing feature_name."
)

official_names_1251 = (
    clear_dict_1251["feature_name"]
    .astype(str)
    .tolist()
)

require_1251(
    len(official_names_1251) == 30
    and len(set(official_names_1251)) == 30,
    "Expected exactly 30 unique Clear feature names."
)

feature_groups_1251 = [
    "cumulative_deviation",
    "combined_channel_intensity",
    "dominant_channel_switches",
    "channel_distribution_shift",
    "consecutive_elevated_days",
    "cross_channel_deviation_days",
]

expected_names_1251 = [
    f"{name}_{w}wd"
    for name in feature_groups_1251
    for w in (5, 10, 20, 30)
]

expected_names_1251 += [
    f"{channel}_{w}wd"
    for channel in (
        "external_email_count",
        "usb_copy_count",
        "http_upload_count",
    )
    for w in (20, 30)
]

require_1251(
    set(expected_names_1251) == set(official_names_1251),
    "Clear feature names differ from the documented contract."
)

require_1251(
    not set(official_names_1251).intersection(
        reconstructed_original_12.columns
    ),
    "Clear features overlap with the original 85 columns."
)


# ------------------------------------------------------------
# STEP 3 — Verify daily counts and baseline percentiles
# ------------------------------------------------------------

channels_1251 = {
    "EMAIL": {
        "count": "external_email_count",
        "p50": "email_count_p50",
        "p95": "email_count_p95",
        "active_rate": "email_active_rate",
    },
    "USB": {
        "count": "usb_copy_count",
        "p50": "usb_copy_p50",
        "p95": "usb_copy_p95",
        "active_rate": "usb_active_rate",
    },
    "HTTP": {
        "count": "www_upload_count",
        "p50": "http_upload_p50",
        "p95": "http_upload_p95",
        "active_rate": "http_upload_active_rate",
    },
}

available_features_1251 = set(
    reconstructed_original_12.columns
)

core_rows_1251 = []

for channel, names in channels_1251.items():
    for role, field in names.items():
        core_rows_1251.append({
            "channel": channel,
            "role": role,
            "field": field,
            "available": field in available_features_1251,
        })

core_audit_1251 = pd.DataFrame(core_rows_1251)

require_1251(
    core_audit_1251["available"].all(),
    "Missing required daily count or historical percentile fields."
)


# ------------------------------------------------------------
# STEP 4 — Discover original historical mean-active fields
#
# channel_distribution_shift requires historical:
# active_rate * mean_active_count / P95
#
# We inspect column names only.
# No new historical statistics are calculated.
# ------------------------------------------------------------

baseline_schema_1251 = list(
    baseline_columns_122
)

historical_candidate_fields_1251 = sorted({
    name
    for name in (
        list(reconstructed_original_12.columns)
        + baseline_schema_1251
    )
    if (
        "mean" in name.lower()
        or "avg" in name.lower()
        or "active_rate" in name.lower()
    )
})

mean_field_audit_1251 = pd.DataFrame([
    {
        "field": name,
        "in_original_features":
            name in available_features_1251,
        "in_historical_baseline":
            name in baseline_schema_1251,
    }
    for name in historical_candidate_fields_1251
])


# ------------------------------------------------------------
# STEP 5 — Inspect baseline source coverage
# ------------------------------------------------------------

require_1251(
    baseline_12["user"].nunique() == 3995,
    "Unexpected historical baseline population."
)

ml_users_1251 = set(
    reconstructed_original_12["user"].astype("string")
)

baseline_users_1251 = set(
    baseline_12["user"].astype("string")
)

missing_baseline_users_1251 = (
    ml_users_1251 - baseline_users_1251
)

require_1251(
    not missing_baseline_users_1251,
    "Some ML users are missing historical baselines."
)


# ------------------------------------------------------------
# STEP 6 — Report only
# ------------------------------------------------------------

print("\nA — CLEAR FEATURE GROUPS")
display(
    clear_dict_1251.groupby(
        "category",
        dropna=False
    ).size().reset_index(name="features")
)

print("\nB — REQUIRED CHANNEL INPUTS")
display(core_audit_1251)

print("\nC — HISTORICAL MEAN / ACTIVE-RATE CANDIDATES")
display(mean_field_audit_1251)

print("\n" + "=" * 58)
print("12.5.1 — INPUT READINESS")
print("=" * 58)
print("Original feature rows     :", len(reconstructed_original_12))
print("Original feature columns  :", reconstructed_original_12.shape[1])
print("Clear features documented :", len(official_names_1251))
print("Clear feature name matches: 30/30")
print("Channel inputs available  :", int(core_audit_1251["available"].sum()), "/12")
print("Missing baseline users    :", len(missing_baseline_users_1251))
print("Window policy            : PARTIAL FROM 2011")
print("Historical baseline       : UNCHANGED")
print("Feature calculation       : NOT STARTED")
print("Files exported            : NO")
print("=" * 58)

print("12.5.1 STATUS: INPUT INVENTORY COMPLETE")



A — CLEAR FEATURE GROUPS


,category,features
0,Behavioral Channel Rotation,4
1,Cross-Channel Co-elevation,4
2,Cross-Channel Distribution,4
3,Cross-Channel Intensity,4
4,Temporal Channel Volume,6
5,Temporal Deviation,4
6,Temporal Persistence,4



B — REQUIRED CHANNEL INPUTS


,channel,role,field,available
0,EMAIL,count,external_email_count,True
1,EMAIL,p50,email_count_p50,True
2,EMAIL,p95,email_count_p95,True
3,EMAIL,active_rate,email_active_rate,True
4,USB,count,usb_copy_count,True
5,USB,p50,usb_copy_p50,True
6,USB,p95,usb_copy_p95,True
7,USB,active_rate,usb_active_rate,True
8,HTTP,count,www_upload_count,True
9,HTTP,p50,http_upload_p50,True



C — HISTORICAL MEAN / ACTIVE-RATE CANDIDATES


,field,in_original_features,in_historical_baseline
0,email_active_rate,True,True
1,email_bytes_mean_active,False,True
2,email_count_mean_active,False,True
3,http_upload_active_rate,True,True
4,http_upload_mean_active,False,True
5,usb_active_rate,True,True
6,usb_copy_mean_active,False,True



12.5.1 — INPUT READINESS
Original feature rows     : 387162
Original feature columns  : 85
Clear features documented : 30
Clear feature name matches: 30/30
Channel inputs available  : 12 /12
Missing baseline users    : 0
Window policy            : PARTIAL FROM 2011
Historical baseline       : UNCHANGED
Feature calculation       : NOT STARTED
Files exported            : NO
12.5.1 STATUS: INPUT INVENTORY COMPLETE



## 12.5.2 — Clear 30 Feature Calculation

### Objective

Calculate the 30 documented Clear behavioral features using the validated, count-injected 85-column behavioral dataset.

### Approved Calculation Policy

- Source: `reconstructed_original_12`.
- Population: 387,162 approved User-Day rows.
- Rolling history: eligible workdays from 2011 only.
- Windows: 5, 10, 20, and 30 workdays.
- Partial windows: use all currently available preceding workdays, including the current day.
- Historical baselines: immutable 2010 values.
- Channel order: EMAIL, USB, HTTP.
- No attack-scenario metadata, scheduled channels, or Ground Truth in calculations.

### Feature Definitions

1. **Cumulative Deviation:** Sum positive normalized deviations above valid historical P50 references.
2. **Combined Channel Intensity:** Mean daily channel activity normalized by positive historical P95 references.
3. **Dominant Channel Switches:** Count changes among uniquely dominant observed channels, ignoring inactive days and tied maxima.
4. **Channel Distribution Shift:** Total Variation Distance between historical and observed normalized channel distributions.
5. **Consecutive Elevated Days:** Longest consecutive run of eligible workdays with elevated activity in at least one historically valid channel.
6. **Cross-Channel Deviation Days:** Count days where two or more historically valid channels exceed their P50 references.
7. **Rolling Event Counts:** Sum observed Email, USB, and HTTP event counts over 20 and 30 workdays.

### Validation Requirements

- Exactly 30 new feature columns.
- No overlap with the original 85 columns.
- No changes to original columns or historical baselines.
- Correct partial-window boundaries.
- No future observations.
- Correct handling of invalid baseline denominators, tied maxima, and undefined distributions.
- No exports.

### Expected Output

`behavior_features_clear30_12` — 387,162 rows × 115 columns, consisting of the original 85 columns and the 30 Clear features.

Independent formula verification follows in Section 12.5.3.


In [42]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.5.2
# CLEAR 30 FEATURE CALCULATION — CORRECTED
#
# Original Clear formulas and tie handling
# Approved policy: PARTIAL WINDOWS FROM 2011
# Historical 2010 baseline: IMMUTABLE
# Read-only sources — NO EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from IPython.display import display


def require_1252(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.5.2 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Verify approved inputs
# ------------------------------------------------------------

require_1252(
    reconstructed_original_12.shape == (387162, 85),
    "Section 12.4 output has unexpected dimensions."
)

require_1252(
    len(official_names_1251) == 30
    and len(set(official_names_1251)) == 30,
    "Clear feature dictionary is incomplete."
)

BASE_1252 = (
    reconstructed_original_12
    .sort_values(["user", "day"])
    .reset_index(drop=True)
    .copy()
)

require_1252(
    BASE_1252["day"].between(
        "2011-01-01", "2011-06-01"
    ).all(),
    "Feature population includes dates outside 2011."
)

require_1252(
    BASE_1252["is_weekend"].eq(0).all()
    and BASE_1252["logon_count"].gt(0).all(),
    "Feature population includes ineligible workdays."
)

require_1252(
    not BASE_1252.duplicated(["user", "day"]).any(),
    "Duplicate User-Day keys."
)

require_1252(
    "y" not in BASE_1252.columns,
    "Target labels must not enter feature calculations."
)


# ------------------------------------------------------------
# STEP 2 — Load immutable historical mean-active fields
#
# Read from the original 2010 baseline.
# No recalculation using 2011 activity.
# ------------------------------------------------------------

MEAN_FIELDS_1252 = [
    "email_count_mean_active",
    "usb_copy_mean_active",
    "http_upload_mean_active",
]

RATE_FIELDS_1252 = [
    "email_active_rate",
    "usb_active_rate",
    "http_upload_active_rate",
]

require_1252(
    set(MEAN_FIELDS_1252).issubset(baseline_columns_122),
    "Historical mean-active fields are missing."
)

historical_means_1252 = pd.read_parquet(
    baseline_path_122,
    columns=["user", *MEAN_FIELDS_1252],
)

historical_means_1252["user"] = (
    historical_means_1252["user"].astype("string")
)

require_1252(
    historical_means_1252["user"].notna().all()
    and not historical_means_1252["user"].duplicated().any(),
    "Historical baseline contains invalid user keys."
)

source_1252 = BASE_1252.merge(
    historical_means_1252,
    on="user",
    how="left",
    validate="many_to_one",
    indicator=True,
)

require_1252(
    source_1252["_merge"].eq("both").all(),
    "Some ML users lack historical baseline references."
)

source_1252.drop(columns=["_merge"], inplace=True)

require_1252(
    source_1252[["user", "day"]].equals(
        BASE_1252[["user", "day"]]
    ),
    "Baseline join changed User-Day ordering."
)


# ------------------------------------------------------------
# STEP 3 — Validate activity and baseline references
# ------------------------------------------------------------

COUNT_NAMES_1252 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

P50_NAMES_1252 = [
    "email_count_p50",
    "usb_copy_p50",
    "http_upload_p50",
]

P95_NAMES_1252 = [
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95",
]

counts_1252 = source_1252[
    COUNT_NAMES_1252
].to_numpy(dtype="float64")

require_1252(
    np.isfinite(counts_1252).all()
    and (counts_1252 >= 0).all(),
    "Activity counts contain missing, negative, or infinite values."
)

reference_fields_1252 = (
    P50_NAMES_1252
    + P95_NAMES_1252
    + RATE_FIELDS_1252
    + MEAN_FIELDS_1252
)

reference_nunique_1252 = (
    source_1252
    .groupby("user")[reference_fields_1252]
    .nunique(dropna=False)
)

require_1252(
    reference_nunique_1252.le(1).all().all(),
    "Historical baseline references vary within a user."
)


# ------------------------------------------------------------
# STEP 4 — Prepare output and rolling-window functions
# ------------------------------------------------------------

WINDOWS_1252 = (5, 10, 20, 30)

features_1252 = {
    name: np.full(
        len(source_1252),
        np.nan,
        dtype="float64"
    )
    for name in official_names_1251
}

users_array_1252 = source_1252["user"].to_numpy()

boundaries_1252 = np.r_[
    0,
    np.flatnonzero(
        users_array_1252[1:] != users_array_1252[:-1]
    ) + 1,
    len(source_1252),
]


def rolling_sum_1252(values, window):
    """
    Causal partial-window sum.

    Includes the current day and up to window-1
    preceding eligible workdays of the same user.
    """
    values = np.asarray(values, dtype="float64")

    n = len(values)

    ends = np.arange(1, n + 1)
    starts = np.maximum(0, ends - window)

    prefix = np.r_[0.0, np.cumsum(values)]

    return prefix[ends] - prefix[starts]


def longest_run_1252(flags, window):
    """
    Longest consecutive True run within each
    partial window.

    Missing pre-history is treated as False.
    """
    flags = np.asarray(flags, dtype=bool)

    padded = np.pad(
        flags,
        (window - 1, 0),
        constant_values=False,
    )

    windows = sliding_window_view(
        padded,
        window
    )

    current = np.zeros(len(flags), dtype="int64")
    longest = np.zeros(len(flags), dtype="int64")

    for column in range(window):
        current = np.where(
            windows[:, column],
            current + 1,
            0,
        )

        longest = np.maximum(
            longest,
            current
        )

    return longest.astype("float64")


def dominant_switches_1252(labels, window):
    """
    Original Clear switching logic adapted
    to partial windows.

    Ignore inactive days and tied maxima.
    Count switches among the remaining
    observed dominant-channel labels.
    """
    labels = np.asarray(labels, dtype="int64")

    n = len(labels)
    result = np.zeros(n, dtype="float64")

    valid_idx = np.flatnonzero(labels >= 0)

    if len(valid_idx) < 2:
        return result

    valid_labels = labels[valid_idx]

    transitions = (
        valid_labels[1:] != valid_labels[:-1]
    ).astype("int64")

    transition_prefix = np.r_[
        0,
        np.cumsum(transitions)
    ]

    for day in range(n):
        left_day = max(0, day - window + 1)

        left = np.searchsorted(
            valid_idx,
            left_day,
            side="left",
        )

        right = np.searchsorted(
            valid_idx,
            day,
            side="right",
        )

        if right - left >= 2:
            result[day] = (
                transition_prefix[right - 1]
                - transition_prefix[left]
            )

    return result


# ------------------------------------------------------------
# STEP 5 — Calculate the 30 features per user
# ------------------------------------------------------------

undefined_distribution_1252 = 0
no_valid_p95_1252 = 0

for group_index in range(len(boundaries_1252) - 1):

    start = boundaries_1252[group_index]
    end = boundaries_1252[group_index + 1]

    user_frame = source_1252.iloc[start:end]
    n = end - start

    x = counts_1252[start:end]

    p50 = user_frame[
        P50_NAMES_1252
    ].iloc[0].to_numpy(dtype="float64")

    p95 = user_frame[
        P95_NAMES_1252
    ].iloc[0].to_numpy(dtype="float64")

    active_rates = user_frame[
        RATE_FIELDS_1252
    ].iloc[0].to_numpy(dtype="float64")

    mean_active = user_frame[
        MEAN_FIELDS_1252
    ].iloc[0].to_numpy(dtype="float64")

    # --------------------------------------------------------
    # A — Historical baseline validity
    #
    # Preserve original Clear conditions:
    # P50 > 0, P95 > 0, P95 >= P50.
    # --------------------------------------------------------

    valid_deviation = (
        np.isfinite(p50)
        & np.isfinite(p95)
        & (p50 > 0)
        & (p95 > 0)
        & (p95 >= p50)
    )

    E = np.zeros((n, 3), dtype="float64")

    for c in range(3):

        if valid_deviation[c]:

            denominator = max(
                1.0,
                p95[c] - p50[c]
            )

            E[:, c] = (
                np.maximum(
                    0.0,
                    x[:, c] - p50[c]
                )
                / denominator
            )

    daily_deviation = E.sum(axis=1)

    elevated_channels = E > 0

    elevated_today = elevated_channels.any(axis=1)

    cross_elevated_today = (
        elevated_channels.sum(axis=1) >= 2
    )

    # --------------------------------------------------------
    # B — Normalize observed activity using historical P95
    # --------------------------------------------------------

    valid_p95 = (
        np.isfinite(p95)
        & (p95 > 0)
    )

    normalized = np.zeros(
        (n, 3),
        dtype="float64"
    )

    for c in range(3):

        if valid_p95[c]:
            normalized[:, c] = (
                x[:, c] / p95[c]
            )

    valid_p95_count = int(valid_p95.sum())

    if valid_p95_count > 0:

        daily_intensity = (
            normalized.sum(axis=1)
            / valid_p95_count
        )

    else:

        daily_intensity = np.full(
            n,
            np.nan,
            dtype="float64"
        )

        no_valid_p95_1252 += n

    # --------------------------------------------------------
    # C — Observed dominant channel
    #
    # CORRECTED:
    # Use original Clear np.isclose tie handling.
    #
    # Email = 0, USB = 1, HTTP = 2
    # No unique dominant channel = -1
    # --------------------------------------------------------

    daily_max = normalized.max(axis=1)

    ties = np.isclose(
        normalized,
        daily_max[:, None],
        rtol=1e-9,
        atol=1e-12,
    )

    unique_maximum = (
        (ties.sum(axis=1) == 1)
        & (daily_max > 0)
    )

    dominant = np.full(
        n,
        -1,
        dtype="int64"
    )

    dominant[unique_maximum] = (
        normalized[unique_maximum].argmax(axis=1)
    )

    # --------------------------------------------------------
    # D — Fixed historical channel distribution
    # --------------------------------------------------------

    historical_weights = np.zeros(
        3,
        dtype="float64"
    )

    for c in range(3):

        if not valid_p95[c]:
            continue

        require_1252(
            np.isfinite(active_rates[c])
            and np.isfinite(mean_active[c])
            and active_rates[c] >= 0
            and mean_active[c] >= 0,
            "Invalid historical active-rate or mean "
            f"reference for user "
            f"{user_frame['user'].iloc[0]}."
        )

        historical_weights[c] = (
            active_rates[c]
            * mean_active[c]
            / p95[c]
        )

    historical_total = historical_weights.sum()

    if historical_total > 0:

        historical_share = (
            historical_weights
            / historical_total
        )

    else:

        historical_share = None

    # --------------------------------------------------------
    # E — Calculate each partial window
    # --------------------------------------------------------

    for w in WINDOWS_1252:

        # 1 — Cumulative Deviation

        features_1252[
            f"cumulative_deviation_{w}wd"
        ][start:end] = rolling_sum_1252(
            daily_deviation,
            w
        )

        # 2 — Combined Channel Intensity

        if valid_p95_count > 0:

            available_days = np.minimum(
                np.arange(1, n + 1),
                w
            )

            features_1252[
                f"combined_channel_intensity_{w}wd"
            ][start:end] = (
                rolling_sum_1252(
                    daily_intensity,
                    w
                )
                / available_days
            )

        # 3 — Dominant Channel Switches
        #
        # Uses the corrected Clear dominant labels.

        features_1252[
            f"dominant_channel_switches_{w}wd"
        ][start:end] = dominant_switches_1252(
            dominant,
            w
        )

        # 4 — Channel Distribution Shift (TVD)

        observed_rolling = np.column_stack([
            rolling_sum_1252(
                normalized[:, c],
                w
            )
            for c in range(3)
        ])

        observed_total = observed_rolling.sum(axis=1)

        shift = np.full(
            n,
            np.nan,
            dtype="float64"
        )

        if historical_share is not None:

            defined = observed_total > 0

            observed_share = (
                observed_rolling[defined]
                / observed_total[defined, None]
            )

            # CORRECTED:
            # Match original Clear numerical clipping.

            shift[defined] = np.clip(
                0.5 * np.abs(
                    observed_share
                    - historical_share
                ).sum(axis=1),
                0.0,
                1.0,
            )

        undefined_distribution_1252 += int(
            np.isnan(shift).sum()
        )

        features_1252[
            f"channel_distribution_shift_{w}wd"
        ][start:end] = shift

        # 5 — Consecutive Elevated Days

        features_1252[
            f"consecutive_elevated_days_{w}wd"
        ][start:end] = longest_run_1252(
            elevated_today,
            w
        )

        # 6 — Cross-Channel Deviation Days

        features_1252[
            f"cross_channel_deviation_days_{w}wd"
        ][start:end] = rolling_sum_1252(
            cross_elevated_today.astype(float),
            w
        )

        # 7 — Rolling Count Totals (20 and 30 workdays)

        if w in (20, 30):

            for name, channel_index in [
                ("external_email_count", 0),
                ("usb_copy_count", 1),
                ("http_upload_count", 2),
            ]:

                features_1252[
                    f"{name}_{w}wd"
                ][start:end] = rolling_sum_1252(
                    x[:, channel_index],
                    w
                )


# ------------------------------------------------------------
# STEP 6 — Assemble final in-memory Clear dataset
# ------------------------------------------------------------

clear_features_only_12 = pd.DataFrame(
    {
        name: features_1252[name]
        for name in official_names_1251
    },
    index=BASE_1252.index,
)

behavior_features_clear30_12 = pd.concat(
    [
        BASE_1252,
        clear_features_only_12
    ],
    axis=1,
)

require_1252(
    behavior_features_clear30_12.shape
    == (387162, 115),
    "Unexpected output dimensions."
)

require_1252(
    behavior_features_clear30_12[
        original_columns_122
    ].equals(
        BASE_1252[original_columns_122]
    ),
    "An original Data Engineering column changed."
)

require_1252(
    not behavior_features_clear30_12.columns.duplicated().any(),
    "Duplicate feature columns detected."
)

require_1252(
    "y" not in behavior_features_clear30_12.columns,
    "Classification target appeared unexpectedly."
)


# ------------------------------------------------------------
# STEP 7 — Mathematical bounds validation
# ------------------------------------------------------------

bounds_1252 = []

for w in WINDOWS_1252:

    shift = clear_features_only_12[
        f"channel_distribution_shift_{w}wd"
    ]

    switches = clear_features_only_12[
        f"dominant_channel_switches_{w}wd"
    ]

    consecutive = clear_features_only_12[
        f"consecutive_elevated_days_{w}wd"
    ]

    cross = clear_features_only_12[
        f"cross_channel_deviation_days_{w}wd"
    ]

    bounds_1252.append({
        "window": f"{w}wd",

        "distribution_shift":
            shift.dropna().between(0, 1).all(),

        "switches":
            switches.ge(0).all()
            and switches.le(w - 1).all(),

        "consecutive":
            consecutive.ge(0).all()
            and consecutive.le(w).all(),

        "cross_channel":
            cross.ge(0).all()
            and cross.le(w).all(),
    })

bounds_qa_1252 = pd.DataFrame(bounds_1252)

require_1252(
    bounds_qa_1252.drop(
        columns=["window"]
    ).all().all(),
    "Mathematical bounds validation failed."
)

feature_values_1252 = (
    clear_features_only_12.to_numpy(dtype="float64")
)

require_1252(
    np.isfinite(
        feature_values_1252[
            ~np.isnan(feature_values_1252)
        ]
    ).all(),
    "Infinite feature values detected."
)


# ------------------------------------------------------------
# STEP 8 — Partial-window count validation
# ------------------------------------------------------------

first_day_mask_1252 = (
    BASE_1252.groupby("user").cumcount() == 0
)

for name, source_name in [
    ("external_email_count", "external_email_count"),
    ("usb_copy_count", "usb_copy_count"),
    ("http_upload_count", "www_upload_count"),
]:

    count20 = clear_features_only_12[
        f"{name}_20wd"
    ]

    count30 = clear_features_only_12[
        f"{name}_30wd"
    ]

    require_1252(
        count30.ge(count20 - 1e-9).all(),
        f"30-day rolling count is smaller than "
        f"20-day rolling count: {name}"
    )

    require_1252(
        np.allclose(
            count20.loc[first_day_mask_1252],
            BASE_1252.loc[
                first_day_mask_1252,
                source_name
            ],
            rtol=1e-9,
            atol=1e-9,
        ),
        f"First-day partial-window check failed: {name}"
    )


# ------------------------------------------------------------
# STEP 9 — Initial QA reports
# ------------------------------------------------------------

null_audit_1252 = (
    clear_features_only_12
    .isna()
    .sum()
    .rename("null_rows")
    .reset_index()
    .rename(columns={"index": "feature"})
)

print("\nA — CLEAR FEATURE GROUP COUNT")
print("Original columns :", len(original_columns_122))
print("Clear columns    :", len(official_names_1251))
print("Combined columns :", behavior_features_clear30_12.shape[1])
print("Rows             :", behavior_features_clear30_12.shape[0])

print("\nB — MATHEMATICAL BOUNDS QA")
display(bounds_qa_1252)

print("\nC — NULL VALUES BY CLEAR FEATURE")
display(null_audit_1252)

print("\n" + "=" * 60)
print("12.5.2 — CORRECTED CALCULATION SUMMARY")
print("=" * 60)
print("Clear features created       :", len(clear_features_only_12.columns))
print("Output rows                  :", len(behavior_features_clear30_12))
print("Output columns               :", behavior_features_clear30_12.shape[1])
print("Original 85 preserved        : YES")
print("Historical baseline updated  : NO")
print("Rolling start                : 2011 ONLY")
print("Partial windows              : YES")
print("Original Clear tie handling  : YES")
print("Original Clear TVD clipping  : YES")
print("Rows without valid P95       :", no_valid_p95_1252)
print("Undefined distribution cells :", undefined_distribution_1252)
print("Target y added               : NO")
print("Files exported               : NO")
print("=" * 60)

print("12.5.2 STATUS: INITIAL QA PASS — CORRECTED VERSION")



A — CLEAR FEATURE GROUP COUNT
Original columns : 85
Clear columns    : 30
Combined columns : 115
Rows             : 387162

B — MATHEMATICAL BOUNDS QA


,window,distribution_shift,switches,consecutive,cross_channel
0,5wd,True,True,True,True
1,10wd,True,True,True,True
2,20wd,True,True,True,True
3,30wd,True,True,True,True



C — NULL VALUES BY CLEAR FEATURE


,feature,null_rows
0,cumulative_deviation_5wd,0
1,cumulative_deviation_10wd,0
2,cumulative_deviation_20wd,0
3,cumulative_deviation_30wd,0
4,combined_channel_intensity_5wd,0
5,combined_channel_intensity_10wd,0
6,combined_channel_intensity_20wd,0
7,combined_channel_intensity_30wd,0
8,dominant_channel_switches_5wd,0
9,dominant_channel_switches_10wd,0



12.5.2 — CORRECTED CALCULATION SUMMARY
Clear features created       : 30
Output rows                  : 387162
Output columns               : 115
Original 85 preserved        : YES
Historical baseline updated  : NO
Rolling start                : 2011 ONLY
Partial windows              : YES
Original Clear tie handling  : YES
Original Clear TVD clipping  : YES
Rows without valid P95       : 0
Undefined distribution cells : 61997
Target y added               : NO
Files exported               : NO
12.5.2 STATUS: INITIAL QA PASS — CORRECTED VERSION



## 12.5.3 — Independent Clear Feature Verification

### Objective

Independently verify the 30 Clear behavioral features against the documented formulas and the approved partial-window policy.

### Method

1. Select a reproducible sample of users, including users with undefined channel-distribution values.
2. Recompute each feature using simple reference calculations independent of Section 12.5.2.
3. Compare every sampled User-Day and all 30 feature values.
4. Validate the exact placement of NaN values.
5. Verify partial-window boundaries, dominant-channel handling, and historical reference usage.
6. Confirm the original 85 columns remain unchanged.

### Acceptance Criteria

- Zero feature-value mismatches.
- Zero NaN-placement mismatches.
- All 30 features verified.
- Original 85 columns preserved.
- No source modifications or exports.

### Expected Outcome

An independent validation report suitable for approving the Clear feature calculation stage.


In [43]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.5.3
# INDEPENDENT FORMULA VERIFICATION — CLEAR 30
#
# CORRECTED: Original Clear approximate-tie handling
# POLICY: Partial Windows from 2011
# HISTORICAL BASELINE: Immutable
# READ-ONLY — NO EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


def require_1253(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.5.3 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Verify approved inputs
# ------------------------------------------------------------

require_1253(
    behavior_features_clear30_12.shape == (387162, 115),
    "Section 12.5.2 output is invalid."
)

require_1253(
    len(official_names_1251) == 30,
    "Expected 30 documented Clear features."
)

require_1253(
    behavior_features_clear30_12[
        original_columns_122
    ].reset_index(drop=True).equals(
        BASE_1252[
            original_columns_122
        ].reset_index(drop=True)
    ),
    "Original 85 columns were modified."
)


# ------------------------------------------------------------
# STEP 2 — Reproducible user sampling
#
# Include random users and users with undefined TVD.
# ------------------------------------------------------------

all_users_1253 = np.sort(
    behavior_features_clear30_12["user"].unique()
)

rng_1253 = np.random.default_rng(42)

random_users_1253 = rng_1253.choice(
    all_users_1253,
    size=min(25, len(all_users_1253)),
    replace=False,
)

undefined_users_1253 = (
    behavior_features_clear30_12.loc[
        behavior_features_clear30_12[
            "channel_distribution_shift_5wd"
        ].isna(),
        "user"
    ]
    .drop_duplicates()
    .head(10)
    .to_numpy()
)

sample_users_1253 = (
    set(random_users_1253)
    | set(undefined_users_1253)
)

sample_1253 = (
    behavior_features_clear30_12.loc[
        behavior_features_clear30_12[
            "user"
        ].isin(sample_users_1253)
    ]
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

source_sample_1253 = (
    source_1252.loc[
        source_1252["user"].isin(sample_users_1253)
    ]
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

require_1253(
    sample_1253[["user", "day"]].equals(
        source_sample_1253[["user", "day"]]
    ),
    "Sample source keys are misaligned."
)


# ------------------------------------------------------------
# STEP 3 — Independent reference calculation
#
# Uses direct slices and explicit loops.
# Does not reuse 12.5.2 rolling functions.
# ------------------------------------------------------------

def reference_features_1253(frame):

    x = frame[COUNT_NAMES_1252].to_numpy(
        dtype="float64"
    )

    p50 = frame[P50_NAMES_1252].iloc[0].to_numpy(
        dtype="float64"
    )

    p95 = frame[P95_NAMES_1252].iloc[0].to_numpy(
        dtype="float64"
    )

    rates = frame[RATE_FIELDS_1252].iloc[0].to_numpy(
        dtype="float64"
    )

    means = frame[MEAN_FIELDS_1252].iloc[0].to_numpy(
        dtype="float64"
    )

    n = len(frame)

    # Original Clear baseline validity
    valid_dev = (
        np.isfinite(p50)
        & np.isfinite(p95)
        & (p50 > 0)
        & (p95 > 0)
        & (p95 >= p50)
    )

    valid_p95 = (
        np.isfinite(p95)
        & (p95 > 0)
    )

    deviation = np.zeros((n, 3), dtype=float)
    normalized = np.zeros((n, 3), dtype=float)

    for c in range(3):

        if valid_dev[c]:
            for day in range(n):
                deviation[day, c] = (
                    max(0.0, x[day, c] - p50[c])
                    / max(1.0, p95[c] - p50[c])
                )

        if valid_p95[c]:
            for day in range(n):
                normalized[day, c] = (
                    x[day, c] / p95[c]
                )

    # Fixed historical channel distribution
    historical_weights = np.zeros(
        3, dtype=float
    )

    for c in range(3):
        if valid_p95[c]:
            require_1253(
                np.isfinite(rates[c])
                and np.isfinite(means[c])
                and rates[c] >= 0
                and means[c] >= 0,
                "Invalid historical distribution reference."
            )

            historical_weights[c] = (
                rates[c] * means[c] / p95[c]
            )

    historical_total = historical_weights.sum()

    historical_share = (
        historical_weights / historical_total
        if historical_total > 0
        else None
    )

    result = {
        name: []
        for name in official_names_1251
    }

    # Calculate every user's window independently
    for day in range(n):

        for w in (5, 10, 20, 30):

            left = max(0, day - w + 1)

            window_x = x[left:day + 1]
            window_dev = deviation[left:day + 1]
            window_norm = normalized[left:day + 1]

            # ------------------------------------------------
            # 1 — Cumulative Deviation
            # ------------------------------------------------

            result[
                f"cumulative_deviation_{w}wd"
            ].append(
                float(window_dev.sum())
            )

            # ------------------------------------------------
            # 2 — Combined Channel Intensity
            # ------------------------------------------------

            valid_count = int(valid_p95.sum())

            intensity = (
                float(
                    window_norm.sum()
                    / (len(window_x) * valid_count)
                )
                if valid_count > 0
                else np.nan
            )

            result[
                f"combined_channel_intensity_{w}wd"
            ].append(intensity)

            # ------------------------------------------------
            # 3 — Dominant Channel Switches
            #
            # CORRECTED:
            # Match original Clear np.isclose tie handling.
            # ------------------------------------------------

            dominant_sequence = []

            for vector in window_norm:

                maximum = vector.max()

                if maximum <= 0:
                    continue

                ties = np.isclose(
                    vector,
                    maximum,
                    rtol=1e-9,
                    atol=1e-12,
                )

                # Ignore days with tied dominant channels
                if int(ties.sum()) != 1:
                    continue

                dominant_sequence.append(
                    int(np.argmax(vector))
                )

            switches = sum(
                dominant_sequence[i]
                != dominant_sequence[i - 1]
                for i in range(
                    1, len(dominant_sequence)
                )
            )

            result[
                f"dominant_channel_switches_{w}wd"
            ].append(float(switches))

            # ------------------------------------------------
            # 4 — Channel Distribution Shift (TVD)
            #
            # Match original Clear numerical clipping.
            # ------------------------------------------------

            observed = window_norm.sum(axis=0)
            observed_total = float(observed.sum())

            if (
                historical_share is None
                or observed_total <= 0
            ):
                shift = np.nan

            else:
                observed_share = (
                    observed / observed_total
                )

                shift = float(np.clip(
                    0.5 * np.abs(
                        observed_share - historical_share
                    ).sum(),
                    0.0,
                    1.0,
                ))

            result[
                f"channel_distribution_shift_{w}wd"
            ].append(shift)

            # ------------------------------------------------
            # 5 — Consecutive Elevated Days
            # ------------------------------------------------

            flags = [
                bool((row > 0).any())
                for row in window_dev
            ]

            longest = 0
            current = 0

            for flag in flags:

                if flag:
                    current += 1
                    longest = max(
                        longest, current
                    )
                else:
                    current = 0

            result[
                f"consecutive_elevated_days_{w}wd"
            ].append(float(longest))

            # ------------------------------------------------
            # 6 — Cross-Channel Deviation Days
            # ------------------------------------------------

            overlap = sum(
                int((row > 0).sum() >= 2)
                for row in window_dev
            )

            result[
                f"cross_channel_deviation_days_{w}wd"
            ].append(float(overlap))

            # ------------------------------------------------
            # 7 — Rolling Event Counts
            # ------------------------------------------------

            if w in (20, 30):

                for name, c in [
                    ("external_email_count", 0),
                    ("usb_copy_count", 1),
                    ("http_upload_count", 2),
                ]:

                    result[
                        f"{name}_{w}wd"
                    ].append(
                        float(window_x[:, c].sum())
                    )

    return pd.DataFrame(result)


# ------------------------------------------------------------
# STEP 4 — Independent value-by-value comparison
# ------------------------------------------------------------

mismatches_1253 = {
    name: 0
    for name in official_names_1251
}

nan_mismatches_1253 = {
    name: 0
    for name in official_names_1251
}

checked_1253 = {
    name: 0
    for name in official_names_1251
}


for user, reference_source in source_sample_1253.groupby(
    "user", sort=False
):

    reference_source = (
        reference_source
        .sort_values("day")
    )

    reference = reference_features_1253(
        reference_source
    )

    actual = (
        sample_1253.loc[
            sample_1253["user"].eq(user),
            official_names_1251
        ]
        .reset_index(drop=True)
    )

    require_1253(
        len(reference) == len(actual),
        f"Reference row mismatch for user {user}"
    )

    for name in official_names_1251:

        expected = reference[name].to_numpy(
            dtype=float
        )

        observed = actual[name].to_numpy(
            dtype=float
        )

        nan_mismatches_1253[name] += int(
            np.sum(
                np.isnan(expected)
                != np.isnan(observed)
            )
        )

        same = np.isclose(
            expected,
            observed,
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True,
        )

        mismatches_1253[name] += int(
            (~same).sum()
        )

        checked_1253[name] += len(expected)


# ------------------------------------------------------------
# STEP 5 — Independent QA report
# ------------------------------------------------------------

independent_qa_1253 = pd.DataFrame([
    {
        "feature": name,
        "checked_values": checked_1253[name],
        "value_mismatches": mismatches_1253[name],
        "nan_mismatches": nan_mismatches_1253[name],
        "status": (
            "PASS"
            if mismatches_1253[name] == 0
            and nan_mismatches_1253[name] == 0
            else "FAIL"
        ),
    }
    for name in official_names_1251
])


print("\n12.5.3 — INDEPENDENT FEATURE QA")
display(independent_qa_1253)


print("\n" + "=" * 60)
print("12.5.3 — VALIDATION SUMMARY")
print("=" * 60)

print("Sampled users       :", len(sample_users_1253))
print("Sampled User-Days   :", len(sample_1253))
print("Features checked    :", len(independent_qa_1253))

print(
    "Values checked      :",
    int(independent_qa_1253["checked_values"].sum())
)

print(
    "Value mismatches    :",
    int(independent_qa_1253["value_mismatches"].sum())
)

print(
    "NaN mismatches      :",
    int(independent_qa_1253["nan_mismatches"].sum())
)

print(
    "Passed features     :",
    int(independent_qa_1253["status"].eq("PASS").sum())
)

print("Original Clear ties : np.isclose")
print("Partial windows     : YES — 2011 ONLY")
print("Historical baseline : UNCHANGED")
print("Files exported      : NO")

print("=" * 60)


require_1253(
    independent_qa_1253["status"].eq("PASS").all(),
    "Independent feature verification failed."
)

print(
    "12.5.3 STATUS: PASS — "
    "CLEAR 30 INDEPENDENTLY VERIFIED"
)



12.5.3 — INDEPENDENT FEATURE QA


,feature,checked_values,value_mismatches,nan_mismatches,status
0,cumulative_deviation_5wd,3599,0,0,PASS
1,cumulative_deviation_10wd,3599,0,0,PASS
2,cumulative_deviation_20wd,3599,0,0,PASS
3,cumulative_deviation_30wd,3599,0,0,PASS
4,combined_channel_intensity_5wd,3599,0,0,PASS
5,combined_channel_intensity_10wd,3599,0,0,PASS
6,combined_channel_intensity_20wd,3599,0,0,PASS
7,combined_channel_intensity_30wd,3599,0,0,PASS
8,dominant_channel_switches_5wd,3599,0,0,PASS
9,dominant_channel_switches_10wd,3599,0,0,PASS



12.5.3 — VALIDATION SUMMARY
Sampled users       : 35
Sampled User-Days   : 3599
Features checked    : 30
Values checked      : 107970
Value mismatches    : 0
NaN mismatches      : 0
Passed features     : 30
Original Clear ties : np.isclose
Partial windows     : YES — 2011 ONLY
Historical baseline : UNCHANGED
Files exported      : NO
12.5.3 STATUS: PASS — CLEAR 30 INDEPENDENTLY VERIFIED



## 12.6.1 — New Rotation Feature Calculation

### Objective

Implement the 27 New Rotation features from the ML team's authoritative Feature Generation Contract.

### Input

`behavior_features_clear30_12` — the validated 115-column behavioral feature dataset containing the original 85 Data Engineering columns and the 30 Clear features.

### Approved Methodology

- Preserve the 27 documented feature definitions.
- Use partial windows of 5, 10, and 20 eligible workdays.
- Begin rolling history in 2011 only.
- Preserve fixed historical P50, P90, and P95 values from 2010.
- Calculate deviation according to the ML contract, independently of Clear's historical-channel validity restrictions.
- Break dominant-deviation ties in the fixed order EMAIL, USB, HTTP.
- For `nr_alt`, require adjacent eligible workdays with deviation on both days.
- For `nr_gap_alt_2`, allow at most two intervening quiet eligible workdays.
- For `nr_handoff`, consider the immediately preceding eligible workday, including when it lies outside the current rolling window, but never before the first available 2011 workday.
- Use only observed activity and historical baselines. Ground Truth and scheduling metadata are prohibited.

### Validation Requirements

1. Exactly 27 new columns.
2. Exactly 387,162 User-Day rows and 142 total columns.
3. All previous 115 columns preserved.
4. No duplicate user-day keys or feature names.
5. Correct feature bounds, valid ratios, and missing-value behavior.
6. No 2010 activity within rolling windows.
7. No baseline recalculation, source modifications, or exports.

### Expected Output

`behavior_features_nr27_12` — the 142-column in-memory behavioral dataset.

Independent numerical and edge-case verification will be performed in Section 12.6.2 before approval.


In [44]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.6.1
# NEW ROTATION — 27 FEATURE CALCULATION
#
# Source: FEATURE_GENERATION_CONTRACT.md
# Approved change: PARTIAL WINDOWS FROM 2011
# Historical baseline: IMMUTABLE
# No Ground Truth, no labels, no exports
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


def require_1261(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.6.1 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Validate approved input and feature names
# ------------------------------------------------------------

require_1261(
    behavior_features_clear30_12.shape == (387162, 115),
    "Section 12.5 output has unexpected dimensions."
)

NR_FAMILIES_1261 = [
    "nr_m",
    "nr_entropy",
    "nr_excl_days",
    "nr_overlap_days",
    "nr_alt",
    "nr_handoff",
    "nr_stealth_acc",
    "nr_gap_alt_2",
]

NR_WINDOWS_1261 = (5, 10, 20)

NR_RATIOS_1261 = [
    "email_count_to_p90_ratio",
    "usb_copy_to_p90_ratio",
    "http_upload_to_p90_ratio",
]

NR_NAMES_1261 = [
    f"{family}_{w}wd"
    for family in NR_FAMILIES_1261
    for w in NR_WINDOWS_1261
] + NR_RATIOS_1261

require_1261(
    len(NR_NAMES_1261) == 27
    and len(set(NR_NAMES_1261)) == 27
    and set(NR_NAMES_1261) == set(nr_features_122),
    "The 27 feature names do not match the ML contract."
)

require_1261(
    not set(NR_NAMES_1261).intersection(
        behavior_features_clear30_12.columns
    ),
    "New Rotation feature names overlap existing columns."
)

BASE_1261 = (
    behavior_features_clear30_12
    .sort_values(["user", "day"])
    .reset_index(drop=True)
    .copy()
)

require_1261(
    not BASE_1261[["user", "day"]].isna().any().any()
    and not BASE_1261.duplicated(["user", "day"]).any(),
    "Invalid User-Day keys."
)

require_1261(
    BASE_1261["day"].between(
        "2011-01-01", "2011-06-01"
    ).all(),
    "Rows outside the approved 2011 period."
)

require_1261(
    BASE_1261["is_weekend"].eq(0).all()
    and BASE_1261["logon_count"].gt(0).all(),
    "Ineligible workdays detected."
)

require_1261(
    "y" not in BASE_1261.columns,
    "Target label must not enter feature generation."
)


# ------------------------------------------------------------
# STEP 2 — Required ML contract inputs
# ------------------------------------------------------------

NR_COUNTS_1261 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

NR_P50_1261 = [
    "email_count_p50",
    "usb_copy_p50",
    "http_upload_p50",
]

NR_P90_1261 = [
    "email_count_p90",
    "usb_copy_p90",
    "http_upload_p90",
]

NR_P95_1261 = [
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95",
]

required_1261 = (
    NR_COUNTS_1261
    + NR_P50_1261
    + NR_P90_1261
    + NR_P95_1261
)

require_1261(
    set(required_1261).issubset(BASE_1261.columns),
    "Required ML contract fields are missing."
)

count_matrix_1261 = BASE_1261[
    NR_COUNTS_1261
].to_numpy(dtype="float64")

require_1261(
    np.isfinite(count_matrix_1261).all()
    and (count_matrix_1261 >= 0).all(),
    "Invalid observed event counts."
)

baseline_nunique_1261 = (
    BASE_1261
    .groupby("user")[
        NR_P50_1261 + NR_P90_1261 + NR_P95_1261
    ]
    .nunique(dropna=False)
)

require_1261(
    baseline_nunique_1261.le(1).all().all(),
    "Historical percentiles vary within a user."
)


# ------------------------------------------------------------
# STEP 3 — Prepare in-memory feature arrays
# ------------------------------------------------------------

NR_RESULTS_1261 = {
    name: np.full(
        len(BASE_1261),
        np.nan,
        dtype="float64"
    )
    for name in NR_NAMES_1261
}

users_1261 = BASE_1261["user"].to_numpy()

boundaries_1261 = np.r_[
    0,
    np.flatnonzero(
        users_1261[1:] != users_1261[:-1]
    ) + 1,
    len(BASE_1261),
]


def prefix_sum_1261(values):
    """Cumulative sums with a leading zero."""
    values = np.asarray(values, dtype="float64")

    if values.ndim == 1:
        return np.r_[0.0, np.cumsum(values)]

    return np.vstack([
        np.zeros((1, values.shape[1])),
        np.cumsum(values, axis=0),
    ])


# ------------------------------------------------------------
# STEP 4 — Calculate every user's features
#
# All 2011 eligible workdays are ordered per user.
# No values from 2010 are loaded as rolling history.
# ------------------------------------------------------------

for group_idx in range(len(boundaries_1261) - 1):

    start = boundaries_1261[group_idx]
    end = boundaries_1261[group_idx + 1]

    frame = BASE_1261.iloc[start:end]
    n = end - start

    x = count_matrix_1261[start:end]

    p50 = frame[
        NR_P50_1261
    ].iloc[0].to_numpy(dtype="float64")

    p90 = frame[
        NR_P90_1261
    ].iloc[0].to_numpy(dtype="float64")

    p95 = frame[
        NR_P95_1261
    ].iloc[0].to_numpy(dtype="float64")

    # --------------------------------------------------------
    # A — ML deviation E
    #
    # E = max(0, count-P50) / max(1, P95-P50)
    #
    # Important:
    # P50=0 is allowed when present.
    # Missing count or baseline -> E=0.
    # This differs from Clear's baseline eligibility.
    # --------------------------------------------------------

    E = np.zeros((n, 3), dtype="float64")

    for c in range(3):

        if (
            np.isfinite(p50[c])
            and np.isfinite(p95[c])
        ):
            denominator = max(
                1.0,
                p95[c] - p50[c]
            )

            E[:, c] = (
                np.maximum(
                    0.0,
                    x[:, c] - p50[c]
                )
                / denominator
            )

    dev_flags = E > 0
    dev_day = dev_flags.any(axis=1)

    # Dominant deviation channel:
    # np.argmax breaks ties EMAIL -> USB -> HTTP.
    dominant = np.full(n, -1, dtype=np.int8)
    dominant[dev_day] = E[dev_day].argmax(axis=1)

    # --------------------------------------------------------
    # B — Daily exclusive / overlapping deviations
    # --------------------------------------------------------

    n_deviating = dev_flags.sum(axis=1)

    exclusive = (n_deviating == 1).astype(float)
    overlap = (n_deviating >= 2).astype(float)

    # --------------------------------------------------------
    # C — Handoff from preceding eligible workday
    #
    # First available 2011 workday has no predecessor.
    # Previous day may be outside the current window.
    # --------------------------------------------------------

    previous_flags = np.vstack([
        np.zeros((1, 3), dtype=bool),
        dev_flags[:-1],
    ])

    previous_any = previous_flags.any(axis=1)

    newly_deviating = (
        dev_flags & ~previous_flags
    ).sum(axis=1)

    handoff_daily = np.where(
        previous_any,
        newly_deviating,
        0
    ).astype(float)

    # --------------------------------------------------------
    # D — Stealth accumulation
    #
    # Exclude only channels exceeding their own P95.
    # P95=0 does not qualify positive activity.
    # --------------------------------------------------------

    stealth_daily = np.zeros(n, dtype=float)

    for c in range(3):
        if np.isfinite(p95[c]):
            eligible = (
                (E[:, c] > 0)
                & (x[:, c] <= p95[c])
            )

            stealth_daily += np.where(
                eligible,
                E[:, c],
                0.0,
            )

    # --------------------------------------------------------
    # E — Adjacent deviation-day pairs for nr_alt
    #
    # A quiet day interrupts adjacency.
    # --------------------------------------------------------

    alt_pairs = np.zeros(n, dtype=float)
    alt_changes = np.zeros(n, dtype=float)

    if n > 1:
        adjacent = dev_day[1:] & dev_day[:-1]

        alt_pairs[1:] = adjacent.astype(float)

        alt_changes[1:] = (
            adjacent
            & (dominant[1:] != dominant[:-1])
        ).astype(float)

    # --------------------------------------------------------
    # F — Prefix arrays for fast causal rolling sums
    # --------------------------------------------------------

    E_prefix = prefix_sum_1261(E)
    excl_prefix = prefix_sum_1261(exclusive)
    overlap_prefix = prefix_sum_1261(overlap)
    handoff_prefix = prefix_sum_1261(handoff_daily)
    stealth_prefix = prefix_sum_1261(stealth_daily)
    alt_pairs_prefix = prefix_sum_1261(alt_pairs)
    alt_changes_prefix = prefix_sum_1261(alt_changes)

    # --------------------------------------------------------
    # G — Three daily P90 ratios (no rolling window)
    # --------------------------------------------------------

    for c, name in enumerate(NR_RATIOS_1261):

        if np.isfinite(p90[c]) and p90[c] > 0:
            NR_RESULTS_1261[name][start:end] = (
                x[:, c] / p90[c]
            )

        # Otherwise retain NaN, as specified by Reef.

    # --------------------------------------------------------
    # H — All 5/10/20 partial rolling windows
    # --------------------------------------------------------

    for w in NR_WINDOWS_1261:

        for t in range(n):

            left = max(0, t - w + 1)
            right = t + 1

            # 1 — Number of deviating channels
            S_c = np.maximum(
                E_prefix[right] - E_prefix[left],
                0.0,
            )

            S = float(S_c.sum())

            NR_RESULTS_1261[
                f"nr_m_{w}wd"
            ][start + t] = float(
                np.count_nonzero(S_c > 1e-12)
            )

            # 2 — Normalized deviation entropy
            if S > 1e-12:
                probabilities = (
                    S_c[S_c > 1e-12] / S
                )
                entropy = -float(
                    np.sum(
                        probabilities
                        * np.log(probabilities)
                    )
                    / np.log(3.0)
                )
                entropy = float(
                    np.clip(entropy, 0.0, 1.0)
                )
            else:
                entropy = 0.0

            NR_RESULTS_1261[
                f"nr_entropy_{w}wd"
            ][start + t] = entropy

            # 3 — Exclusive deviation days
            NR_RESULTS_1261[
                f"nr_excl_days_{w}wd"
            ][start + t] = (
                excl_prefix[right] - excl_prefix[left]
            )

            # 4 — Overlapping deviation days
            NR_RESULTS_1261[
                f"nr_overlap_days_{w}wd"
            ][start + t] = (
                overlap_prefix[right] - overlap_prefix[left]
            )

            # 5 — Alternation over adjacent workdays
            # Both endpoints must lie inside the window.
            pair_count = (
                alt_pairs_prefix[right]
                - alt_pairs_prefix[left + 1]
            )

            change_count = (
                alt_changes_prefix[right]
                - alt_changes_prefix[left + 1]
            )

            NR_RESULTS_1261[
                f"nr_alt_{w}wd"
            ][start + t] = (
                change_count / pair_count
                if pair_count > 0
                else 0.0
            )

            # 6 — Handoff
            # Daily handoff was computed using the
            # preceding 2011 workday, even if that day
            # lies outside the current rolling window.
            NR_RESULTS_1261[
                f"nr_handoff_{w}wd"
            ][start + t] = (
                handoff_prefix[right]
                - handoff_prefix[left]
            )

            # 7 — Stealth accumulation
            NR_RESULTS_1261[
                f"nr_stealth_acc_{w}wd"
            ][start + t] = (
                stealth_prefix[right]
                - stealth_prefix[left]
            )

            # 8 — Gap alternation (max 2 quiet days)
            # Use consecutive deviation days inside
            # the current window. Do not bridge gaps
            # greater than two eligible workdays.
            dev_indices = np.flatnonzero(
                dev_day[left:right]
            ) + left

            eligible_pairs = 0
            changed_pairs = 0

            for j in range(1, len(dev_indices)):

                previous = int(dev_indices[j - 1])
                current = int(dev_indices[j])

                quiet_gap = current - previous - 1

                if quiet_gap <= 2:
                    eligible_pairs += 1

                    if dominant[current] != dominant[previous]:
                        changed_pairs += 1

            NR_RESULTS_1261[
                f"nr_gap_alt_2_{w}wd"
            ][start + t] = (
                changed_pairs / eligible_pairs
                if eligible_pairs > 0
                else 0.0
            )


# ------------------------------------------------------------
# STEP 5 — Assemble new in-memory dataset
# ------------------------------------------------------------

nr_features_only_12 = pd.DataFrame(
    {
        name: NR_RESULTS_1261[name]
        for name in NR_NAMES_1261
    },
    index=BASE_1261.index,
)

behavior_features_nr27_12 = pd.concat(
    [BASE_1261, nr_features_only_12],
    axis=1,
)

require_1261(
    behavior_features_nr27_12.shape == (387162, 142),
    "Expected 387,162 rows and 142 columns."
)

require_1261(
    not behavior_features_nr27_12.columns.duplicated().any(),
    "Duplicate feature names."
)

require_1261(
    behavior_features_nr27_12[
        BASE_1261.columns
    ].equals(BASE_1261),
    "Previously approved 115 columns were modified."
)

require_1261(
    "y" not in behavior_features_nr27_12.columns,
    "Ground Truth target appeared during feature generation."
)


# ------------------------------------------------------------
# STEP 6 — Mathematical bounds and missing-value checks
# ------------------------------------------------------------

bounds_rows_1261 = []

for w in NR_WINDOWS_1261:

    fields = {
        "m": nr_features_only_12[f"nr_m_{w}wd"],
        "entropy": nr_features_only_12[
            f"nr_entropy_{w}wd"
        ],
        "exclusive": nr_features_only_12[
            f"nr_excl_days_{w}wd"
        ],
        "overlap": nr_features_only_12[
            f"nr_overlap_days_{w}wd"
        ],
        "alt": nr_features_only_12[f"nr_alt_{w}wd"],
        "handoff": nr_features_only_12[
            f"nr_handoff_{w}wd"
        ],
        "stealth": nr_features_only_12[
            f"nr_stealth_acc_{w}wd"
        ],
        "gap_alt": nr_features_only_12[
            f"nr_gap_alt_2_{w}wd"
        ],
    }

    bounds_rows_1261.append({
        "window": f"{w}wd",
        "m": fields["m"].between(0, 3).all(),
        "entropy": fields["entropy"].between(
            -1e-9, 1 + 1e-9
        ).all(),
        "exclusive": fields["exclusive"].between(
            0, w
        ).all(),
        "overlap": fields["overlap"].between(
            0, w
        ).all(),
        "alt": fields["alt"].between(0, 1).all(),
        "handoff": fields["handoff"].between(
            0, 3 * w
        ).all(),
        "stealth": fields["stealth"].ge(-1e-9).all(),
        "gap_alt": fields["gap_alt"].between(0, 1).all(),
    })

bounds_qa_1261 = pd.DataFrame(bounds_rows_1261)

require_1261(
    bounds_qa_1261.drop(
        columns=["window"]
    ).all().all(),
    "New Rotation mathematical bounds failed."
)

for name in NR_NAMES_1261:
    values = nr_features_only_12[
        name
    ].to_numpy(dtype=float)

    require_1261(
        not np.isinf(values).any(),
        f"Infinite values in {name}."
    )

    if name not in NR_RATIOS_1261:
        require_1261(
            not np.isnan(values).any(),
            f"Unexpected NaN in {name}."
        )


# ------------------------------------------------------------
# STEP 7 — Initial QA reports
# ------------------------------------------------------------

null_qa_1261 = (
    nr_features_only_12
    .isna()
    .sum()
    .rename("null_rows")
    .reset_index()
    .rename(columns={"index": "feature"})
)

print("\nA — NEW ROTATION FEATURE GROUPS")
display(
    pd.DataFrame([
        {
            "family": family,
            "features": 3,
        }
        for family in NR_FAMILIES_1261
    ] + [{
        "family": "P90 ratios",
        "features": 3,
    }])
)

print("\nB — MATHEMATICAL BOUNDS QA")
display(bounds_qa_1261)

print("\nC — NULL VALUES BY NEW ROTATION FEATURE")
display(null_qa_1261)

print("\n" + "=" * 60)
print("12.6.1 — NEW ROTATION CALCULATION SUMMARY")
print("=" * 60)
print("Input rows                :", len(BASE_1261))
print("Input columns             :", BASE_1261.shape[1])
print("New Rotation features     :", len(NR_NAMES_1261))
print("Output rows               :", len(behavior_features_nr27_12))
print("Output columns            :", behavior_features_nr27_12.shape[1])
print("Previous 115 preserved    : YES")
print("Partial windows           : YES — 2011 ONLY")
print("Historical baseline       : UNCHANGED")
print("ML deviation formula      : CONTRACT DEFINITION")
print("Dominant tie priority     : EMAIL > USB > HTTP")
print("Target y added            : NO")
print("Files exported            : NO")
print("=" * 60)

print("12.6.1 STATUS: INITIAL QA PASS")
print("Next: 12.6.2 — Independent New Rotation Verification")



A — NEW ROTATION FEATURE GROUPS


,family,features
0,nr_m,3
1,nr_entropy,3
2,nr_excl_days,3
3,nr_overlap_days,3
4,nr_alt,3
5,nr_handoff,3
6,nr_stealth_acc,3
7,nr_gap_alt_2,3
8,P90 ratios,3



B — MATHEMATICAL BOUNDS QA


,window,m,entropy,exclusive,overlap,alt,handoff,stealth,gap_alt
0,5wd,True,True,True,True,True,True,True,True
1,10wd,True,True,True,True,True,True,True,True
2,20wd,True,True,True,True,True,True,True,True



C — NULL VALUES BY NEW ROTATION FEATURE


,feature,null_rows
0,nr_m_5wd,0
1,nr_m_10wd,0
2,nr_m_20wd,0
3,nr_entropy_5wd,0
4,nr_entropy_10wd,0
5,nr_entropy_20wd,0
6,nr_excl_days_5wd,0
7,nr_excl_days_10wd,0
8,nr_excl_days_20wd,0
9,nr_overlap_days_5wd,0



12.6.1 — NEW ROTATION CALCULATION SUMMARY
Input rows                : 387162
Input columns             : 115
New Rotation features     : 27
Output rows               : 387162
Output columns            : 142
Previous 115 preserved    : YES
Partial windows           : YES — 2011 ONLY
Historical baseline       : UNCHANGED
ML deviation formula      : CONTRACT DEFINITION
Dominant tie priority     : EMAIL > USB > HTTP
Target y added            : NO
Files exported            : NO
12.6.1 STATUS: INITIAL QA PASS
Next: 12.6.2 — Independent New Rotation Verification



## 12.6.1A — P90 Baseline and Missing Ratio Investigation

### Objective

Investigate the high NaN rates in the USB and HTTP P90 ratio features before approving the New Rotation features.

### Validation

1. Separate missing P90 values from zero and positive P90 values.
2. Check whether users with P90 = 0 had any historical channel activity.
3. Distinguish historically inactive users from users with rare activity.
4. Identify users who have positive 2011 activity despite P90 = 0.
5. Verify that the P90 values match the immutable historical baseline.
6. Verify that NaN ratios occur exactly when P90 is missing or nonpositive.
7. Inspect percentile consistency: P50 <= P90 <= P95.

### Restrictions

No historical baseline recalculation, no feature modifications, and no exports.

### Expected Outcome

A diagnostic report identifying whether the missing ratios are mathematically expected or require further investigation.


In [45]:

# ============================================================
# 12.6.1A — P90 BASELINE & MISSING RATIO AUDIT
# READ-ONLY — NO RECALCULATION OR EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

CHANNELS_126A = {
    "EMAIL": {
        "count": "external_email_count",
        "p50": "email_count_p50",
        "p90": "email_count_p90",
        "p95": "email_count_p95",
        "active_days": "email_active_days",
        "ratio": "email_count_to_p90_ratio",
    },
    "USB": {
        "count": "usb_copy_count",
        "p50": "usb_copy_p50",
        "p90": "usb_copy_p90",
        "p95": "usb_copy_p95",
        "active_days": "usb_active_days",
        "ratio": "usb_copy_to_p90_ratio",
    },
    "HTTP": {
        "count": "www_upload_count",
        "p50": "http_upload_p50",
        "p90": "http_upload_p90",
        "p95": "http_upload_p95",
        "active_days": "http_upload_active_days",
        "ratio": "http_upload_to_p90_ratio",
    },
}

df_126A = behavior_features_nr27_12

# Read historical references only.
historical_fields_126A = sorted({
    field
    for spec in CHANNELS_126A.values()
    for field in (
        spec["p50"], spec["p90"],
        spec["p95"], spec["active_days"]
    )
})

historical_126A = pd.read_parquet(
    baseline_path_122,
    columns=["user", *historical_fields_126A]
)

historical_126A["user"] = (
    historical_126A["user"].astype("string")
)

assert historical_126A["user"].is_unique
assert len(df_126A) == 387162

# One reference row per user, with all 2011 counts summarized.
user_daily_126A = df_126A.groupby(
    "user", as_index=False
).agg(
    email_2011=("external_email_count", "sum"),
    usb_2011=("usb_copy_count", "sum"),
    http_2011=("www_upload_count", "sum"),
)

users_126A = historical_126A.merge(
    user_daily_126A,
    on="user",
    how="inner",
    validate="one_to_one"
)

assert len(users_126A) == df_126A["user"].nunique()

summary_rows_126A = []
historical_rows_126A = []
percentile_rows_126A = []
ratio_rows_126A = []

daily_sums_126A = {
    "EMAIL": "email_2011",
    "USB": "usb_2011",
    "HTTP": "http_2011",
}

for channel, spec in CHANNELS_126A.items():

    p90 = pd.to_numeric(df_126A[spec["p90"]])
    count = pd.to_numeric(df_126A[spec["count"]])
    ratio = pd.to_numeric(df_126A[spec["ratio"]])

    missing_p90 = p90.isna()
    zero_p90 = p90.eq(0)
    positive_p90 = p90.gt(0)
    negative_p90 = p90.lt(0)

    # Verify the calculation of the P90 ratio.
    expected = count.div(p90.where(positive_p90))

    matches = np.isclose(
        expected.to_numpy(dtype=float),
        ratio.to_numpy(dtype=float),
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True
    )

    ratio_rows_126A.append({
        "channel": channel,
        "ratio_NaN": int(ratio.isna().sum()),
        "expected_NaN": int((~positive_p90).sum()),
        "ratio_mismatches": int((~matches).sum()),
    })

    summary_rows_126A.append({
        "channel": channel,
        "P90_missing_rows": int(missing_p90.sum()),
        "P90_zero_rows": int(zero_p90.sum()),
        "P90_negative_rows": int(negative_p90.sum()),
        "P90_positive_rows": int(positive_p90.sum()),
        "2011_active_with_P90_zero": int(
            (zero_p90 & count.gt(0)).sum()
        ),
    })

    # Analyze historical activity by UNIQUE USER.
    historical_p90 = users_126A[spec["p90"]]
    active_days = users_126A[spec["active_days"]]
    future_activity = users_126A[daily_sums_126A[channel]]

    zero_users = historical_p90.eq(0)

    historical_rows_126A.append({
        "channel": channel,
        "users_P90_zero": int(zero_users.sum()),
        "zero_P90_no_2010_activity": int(
            (zero_users & active_days.eq(0)).sum()
        ),
        "zero_P90_rare_2010_activity": int(
            (zero_users & active_days.gt(0)).sum()
        ),
        "zero_P90_positive_2011_activity": int(
            (zero_users & future_activity.gt(0)).sum()
        ),
        "P90_missing_users": int(
            historical_p90.isna().sum()
        ),
    })

    # Percentile consistency on historical users.
    p50_user = users_126A[spec["p50"]]
    p95_user = users_126A[spec["p95"]]

    valid = (
        p50_user.notna()
        & historical_p90.notna()
        & p95_user.notna()
    )

    inconsistent = valid & (
        p50_user.gt(historical_p90)
        | historical_p90.gt(p95_user)
    )

    percentile_rows_126A.append({
        "channel": channel,
        "checked_users": int(valid.sum()),
        "percentile_inconsistencies": int(
            inconsistent.sum()
        ),
    })

    # Confirm P90 references match original historical source.
    historical_reference = historical_126A.set_index("user")[
        spec["p90"]
    ].reindex(df_126A["user"])

    original_values = historical_reference.to_numpy(
        dtype=float
    )

    current_values = p90.to_numpy(dtype=float)

    assert np.isclose(
        original_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ).all(), f"{channel}: historical P90 mismatch"


print("\nA — P90 VALUE DISTRIBUTION")
display(pd.DataFrame(summary_rows_126A))

print("\nB — HISTORICAL ACTIVITY (UNIQUE USERS)")
display(pd.DataFrame(historical_rows_126A))

print("\nC — PERCENTILE CONSISTENCY")
display(pd.DataFrame(percentile_rows_126A))

print("\nD — P90 RATIO VERIFICATION")
display(pd.DataFrame(ratio_rows_126A))

print("\n" + "=" * 55)
print("12.6.1A — DIAGNOSTIC SUMMARY")
print("=" * 55)
print("Users checked      :", len(users_126A))
print("User-Days checked  :", len(df_126A))
print("P90 baseline match : PASS")
print("Baseline modified  : NO")
print("Features modified  : NO")
print("Files exported     : NO")
print("=" * 55)
print("12.6.1A STATUS: DIAGNOSTIC COMPLETE")



A — P90 VALUE DISTRIBUTION


,channel,P90_missing_rows,P90_zero_rows,P90_negative_rows,P90_positive_rows,2011_active_with_P90_zero
0,EMAIL,0,0,0,387162,0
1,USB,312211,0,0,74951,0
2,HTTP,328257,0,0,58905,0



B — HISTORICAL ACTIVITY (UNIQUE USERS)


,channel,users_P90_zero,zero_P90_no_2010_activity,zero_P90_rare_2010_activity,zero_P90_positive_2011_activity,P90_missing_users
0,EMAIL,0,0,0,0,0
1,USB,0,0,0,0,3016
2,HTTP,0,0,0,0,3170



C — PERCENTILE CONSISTENCY


,channel,checked_users,percentile_inconsistencies
0,EMAIL,3737,0
1,USB,721,0
2,HTTP,567,0



D — P90 RATIO VERIFICATION


,channel,ratio_NaN,expected_NaN,ratio_mismatches
0,EMAIL,0,0,0
1,USB,312211,312211,0
2,HTTP,328257,328257,0



12.6.1A — DIAGNOSTIC SUMMARY
Users checked      : 3737
User-Days checked  : 387162
P90 baseline match : PASS
Baseline modified  : NO
Features modified  : NO
Files exported     : NO
12.6.1A STATUS: DIAGNOSTIC COMPLETE



## 12.6.1A — P90 Baseline and Missing Ratio Investigation

### Objective

Investigate the high NaN rates in the USB and HTTP P90 ratio features before approving the New Rotation features.

### Validation

1. Separate missing P90 values from zero and positive P90 values.
2. Check whether users with P90 = 0 had any historical channel activity.
3. Distinguish historically inactive users from users with rare activity.
4. Identify users who have positive 2011 activity despite P90 = 0.
5. Verify that the P90 values match the immutable historical baseline.
6. Verify that NaN ratios occur exactly when P90 is missing or nonpositive.
7. Inspect percentile consistency: P50 <= P90 <= P95.

### Restrictions

No historical baseline recalculation, no feature modifications, and no exports.

### Expected Outcome

A diagnostic report identifying whether the missing ratios are mathematically expected or require further investigation.


In [46]:

# ============================================================
# 12.6.1A — P90 BASELINE & MISSING RATIO AUDIT
# READ-ONLY — NO RECALCULATION OR EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

CHANNELS_126A = {
    "EMAIL": {
        "count": "external_email_count",
        "p50": "email_count_p50",
        "p90": "email_count_p90",
        "p95": "email_count_p95",
        "active_days": "email_active_days",
        "ratio": "email_count_to_p90_ratio",
    },
    "USB": {
        "count": "usb_copy_count",
        "p50": "usb_copy_p50",
        "p90": "usb_copy_p90",
        "p95": "usb_copy_p95",
        "active_days": "usb_active_days",
        "ratio": "usb_copy_to_p90_ratio",
    },
    "HTTP": {
        "count": "www_upload_count",
        "p50": "http_upload_p50",
        "p90": "http_upload_p90",
        "p95": "http_upload_p95",
        "active_days": "http_upload_active_days",
        "ratio": "http_upload_to_p90_ratio",
    },
}

df_126A = behavior_features_nr27_12

# Read historical references only.
historical_fields_126A = sorted({
    field
    for spec in CHANNELS_126A.values()
    for field in (
        spec["p50"], spec["p90"],
        spec["p95"], spec["active_days"]
    )
})

historical_126A = pd.read_parquet(
    baseline_path_122,
    columns=["user", *historical_fields_126A]
)

historical_126A["user"] = (
    historical_126A["user"].astype("string")
)

assert historical_126A["user"].is_unique
assert len(df_126A) == 387162

# One reference row per user, with all 2011 counts summarized.
user_daily_126A = df_126A.groupby(
    "user", as_index=False
).agg(
    email_2011=("external_email_count", "sum"),
    usb_2011=("usb_copy_count", "sum"),
    http_2011=("www_upload_count", "sum"),
)

users_126A = historical_126A.merge(
    user_daily_126A,
    on="user",
    how="inner",
    validate="one_to_one"
)

assert len(users_126A) == df_126A["user"].nunique()

summary_rows_126A = []
historical_rows_126A = []
percentile_rows_126A = []
ratio_rows_126A = []

daily_sums_126A = {
    "EMAIL": "email_2011",
    "USB": "usb_2011",
    "HTTP": "http_2011",
}

for channel, spec in CHANNELS_126A.items():

    p90 = pd.to_numeric(df_126A[spec["p90"]])
    count = pd.to_numeric(df_126A[spec["count"]])
    ratio = pd.to_numeric(df_126A[spec["ratio"]])

    missing_p90 = p90.isna()
    zero_p90 = p90.eq(0)
    positive_p90 = p90.gt(0)
    negative_p90 = p90.lt(0)

    # Verify the calculation of the P90 ratio.
    expected = count.div(p90.where(positive_p90))

    matches = np.isclose(
        expected.to_numpy(dtype=float),
        ratio.to_numpy(dtype=float),
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True
    )

    ratio_rows_126A.append({
        "channel": channel,
        "ratio_NaN": int(ratio.isna().sum()),
        "expected_NaN": int((~positive_p90).sum()),
        "ratio_mismatches": int((~matches).sum()),
    })

    summary_rows_126A.append({
        "channel": channel,
        "P90_missing_rows": int(missing_p90.sum()),
        "P90_zero_rows": int(zero_p90.sum()),
        "P90_negative_rows": int(negative_p90.sum()),
        "P90_positive_rows": int(positive_p90.sum()),
        "2011_active_with_P90_zero": int(
            (zero_p90 & count.gt(0)).sum()
        ),
    })

    # Analyze historical activity by UNIQUE USER.
    historical_p90 = users_126A[spec["p90"]]
    active_days = users_126A[spec["active_days"]]
    future_activity = users_126A[daily_sums_126A[channel]]

    zero_users = historical_p90.eq(0)

    historical_rows_126A.append({
        "channel": channel,
        "users_P90_zero": int(zero_users.sum()),
        "zero_P90_no_2010_activity": int(
            (zero_users & active_days.eq(0)).sum()
        ),
        "zero_P90_rare_2010_activity": int(
            (zero_users & active_days.gt(0)).sum()
        ),
        "zero_P90_positive_2011_activity": int(
            (zero_users & future_activity.gt(0)).sum()
        ),
        "P90_missing_users": int(
            historical_p90.isna().sum()
        ),
    })

    # Percentile consistency on historical users.
    p50_user = users_126A[spec["p50"]]
    p95_user = users_126A[spec["p95"]]

    valid = (
        p50_user.notna()
        & historical_p90.notna()
        & p95_user.notna()
    )

    inconsistent = valid & (
        p50_user.gt(historical_p90)
        | historical_p90.gt(p95_user)
    )

    percentile_rows_126A.append({
        "channel": channel,
        "checked_users": int(valid.sum()),
        "percentile_inconsistencies": int(
            inconsistent.sum()
        ),
    })

    # Confirm P90 references match original historical source.
    historical_reference = historical_126A.set_index("user")[
        spec["p90"]
    ].reindex(df_126A["user"])

    original_values = historical_reference.to_numpy(
        dtype=float
    )

    current_values = p90.to_numpy(dtype=float)

    assert np.isclose(
        original_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ).all(), f"{channel}: historical P90 mismatch"


print("\nA — P90 VALUE DISTRIBUTION")
display(pd.DataFrame(summary_rows_126A))

print("\nB — HISTORICAL ACTIVITY (UNIQUE USERS)")
display(pd.DataFrame(historical_rows_126A))

print("\nC — PERCENTILE CONSISTENCY")
display(pd.DataFrame(percentile_rows_126A))

print("\nD — P90 RATIO VERIFICATION")
display(pd.DataFrame(ratio_rows_126A))

print("\n" + "=" * 55)
print("12.6.1A — DIAGNOSTIC SUMMARY")
print("=" * 55)
print("Users checked      :", len(users_126A))
print("User-Days checked  :", len(df_126A))
print("P90 baseline match : PASS")
print("Baseline modified  : NO")
print("Features modified  : NO")
print("Files exported     : NO")
print("=" * 55)
print("12.6.1A STATUS: DIAGNOSTIC COMPLETE")



A — P90 VALUE DISTRIBUTION


,channel,P90_missing_rows,P90_zero_rows,P90_negative_rows,P90_positive_rows,2011_active_with_P90_zero
0,EMAIL,0,0,0,387162,0
1,USB,312211,0,0,74951,0
2,HTTP,328257,0,0,58905,0



B — HISTORICAL ACTIVITY (UNIQUE USERS)


,channel,users_P90_zero,zero_P90_no_2010_activity,zero_P90_rare_2010_activity,zero_P90_positive_2011_activity,P90_missing_users
0,EMAIL,0,0,0,0,0
1,USB,0,0,0,0,3016
2,HTTP,0,0,0,0,3170



C — PERCENTILE CONSISTENCY


,channel,checked_users,percentile_inconsistencies
0,EMAIL,3737,0
1,USB,721,0
2,HTTP,567,0



D — P90 RATIO VERIFICATION


,channel,ratio_NaN,expected_NaN,ratio_mismatches
0,EMAIL,0,0,0
1,USB,312211,312211,0
2,HTTP,328257,328257,0



12.6.1A — DIAGNOSTIC SUMMARY
Users checked      : 3737
User-Days checked  : 387162
P90 baseline match : PASS
Baseline modified  : NO
Features modified  : NO
Files exported     : NO
12.6.1A STATUS: DIAGNOSTIC COMPLETE



## 12.6.1B — Historical Activity Verification for Missing P90

### Objective
Determine whether missing historical P90 values are associated with zero channel activity during the 2010 baseline period.

### Method
- Examine EMAIL, USB, and HTTP separately.
- Use the original historical active-day counts.
- Distinguish zero activity, positive activity, and missing activity statistics.
- Identify inconsistencies between P90 availability and historical activity.
- Preserve all original source files and computed features.

### Expected Outcome
Confirm whether missing P90 values are explained by historical channel inactivity or require additional investigation.

No recalculation, modification, or export is performed.


In [47]:

# ============================================================
# 12.6.1B — HISTORICAL ACTIVITY VERIFICATION
# READ-ONLY / NO FEATURE RECALCULATION
# ============================================================

import pandas as pd
from IPython.display import display

assert "users_126A" in globals(), (
    "Run 12.6.1A before this cell."
)

assert "CHANNELS_126A" in globals(), (
    "Missing channel definitions from 12.6.1A."
)

rows_126B = []
examples_126B = []

for channel, spec in CHANNELS_126A.items():

    p90_col = spec["p90"]
    activity_col = spec["active_days"]

    assert p90_col in users_126A.columns
    assert activity_col in users_126A.columns

    p90 = pd.to_numeric(
        users_126A[p90_col], errors="coerce"
    )

    active_days = pd.to_numeric(
        users_126A[activity_col], errors="coerce"
    )

    missing = p90.isna()
    available = p90.notna()

    no_activity = active_days.eq(0)
    has_activity = active_days.gt(0)
    unknown_activity = active_days.isna()

    rows_126B.append({
        "channel": channel,
        "users": len(users_126A),
        "missing_P90": int(missing.sum()),
        "missing_P90_zero_activity": int(
            (missing & no_activity).sum()
        ),
        "missing_P90_positive_activity": int(
            (missing & has_activity).sum()
        ),
        "missing_P90_unknown_activity": int(
            (missing & unknown_activity).sum()
        ),
        "available_P90_zero_activity": int(
            (available & no_activity).sum()
        ),
    })

    # Inspect unexpected cases rather than hiding them.
    suspicious = (
        (missing & (has_activity | unknown_activity))
        | (available & no_activity)
    )

    if suspicious.any():
        sample = users_126A.loc[
            suspicious,
            ["user", p90_col, activity_col]
        ].head(10).copy()

        sample.insert(0, "channel", channel)
        examples_126B.append(sample)


report_126B = pd.DataFrame(rows_126B)

report_126B["classification_complete"] = (
    report_126B[
        [
            "missing_P90_zero_activity",
            "missing_P90_positive_activity",
            "missing_P90_unknown_activity"
        ]
    ].sum(axis=1)
    == report_126B["missing_P90"]
)

report_126B["expected_inactivity_pattern"] = (
    report_126B["missing_P90_positive_activity"].eq(0)
    & report_126B["missing_P90_unknown_activity"].eq(0)
)

print("\nA — HISTORICAL ACTIVITY AND P90 AVAILABILITY")
display(report_126B)

print("\nB — UNEXPECTED OR UNKNOWN CASES")

if examples_126B:
    display(
        pd.concat(
            examples_126B,
            ignore_index=True
        )
    )
else:
    print("No unexpected or unknown cases found.")

print("\n" + "=" * 58)
print("12.6.1B — HISTORICAL ACTIVITY AUDIT")
print("=" * 58)

if (
    report_126B["classification_complete"].all()
    and report_126B["expected_inactivity_pattern"].all()
):
    print("STATUS: PASS")
    print(
        "All missing P90 values correspond to "
        "zero recorded historical active days."
    )
else:
    print("STATUS: REVIEW REQUIRED")
    print(
        "Some missing P90 values require "
        "further investigation."
    )

print("Historical baseline modified : NO")
print("Feature values modified      : NO")
print("Files exported               : NO")
print("=" * 58)



A — HISTORICAL ACTIVITY AND P90 AVAILABILITY


,channel,users,missing_P90,missing_P90_zero_activity,missing_P90_positive_activity,missing_P90_unknown_activity,available_P90_zero_activity,classification_complete,expected_inactivity_pattern
0,EMAIL,3737,0,0,0,0,0,True,True
1,USB,3737,3016,3016,0,0,0,True,True
2,HTTP,3737,3170,3170,0,0,0,True,True



B — UNEXPECTED OR UNKNOWN CASES
No unexpected or unknown cases found.

12.6.1B — HISTORICAL ACTIVITY AUDIT
STATUS: PASS
All missing P90 values correspond to zero recorded historical active days.
Historical baseline modified : NO
Feature values modified      : NO
Files exported               : NO



## 12.6.2 — Independent New Rotation Feature Verification

### Objective

Independently verify the numerical correctness of the 27 New Rotation features generated in Section 12.6.1.

### Reference

Use the authoritative `FEATURE_GENERATION_CONTRACT.md` definitions for all eight New Rotation feature families and the three daily P90 ratios.

### Methodology

1. Select a reproducible sample of users, including users with different channel-activity profiles.
2. Recalculate the 27 features using a separate, straightforward reference implementation.
3. Use only the observed 2011 eligible workday sequence and the immutable historical baseline.
4. Apply partial windows of 5, 10, and 20 workdays.
5. Verify the exact dominant-deviation tie priority: EMAIL, USB, HTTP.
6. Verify adjacent-day alternation, handoff behavior, and gap-tolerant alternation.
7. Compare numerical values with tolerance 1e-9, including exact agreement on NaN positions.
8. Report mismatches for each feature and inspect any discrepancies.

### Acceptance Criteria

- All 27 features tested.
- Zero numerical mismatches.
- Zero NaN mismatches.
- All sampled user-day keys preserved.
- No modifications to the approved 142-column dataset.
- No Ground Truth, labels, external files, or exports.

### Expected Result

An independent verification report showing whether the New Rotation implementation matches the documented feature contract.

Section 12.6 will not be approved until the independent verification passes.


In [48]:

# ============================================================
# 12.6.2 — INDEPENDENT NEW ROTATION VERIFICATION
#
# Reference: FEATURE_GENERATION_CONTRACT.md
# Window policy: 2011-only partial workday windows
#
# Independent implementation:
# - No reuse of 12.6.1 feature-calculation functions
# - No reuse of 12.6.1 intermediate deviation arrays
# - No Ground Truth or labels
# - No modifications or exports
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

TOL_1262 = 1e-9
WINDOWS_1262 = (5, 10, 20)

EXPECTED_1262 = [
    f"nr_{family}_{w}wd"
    for family in (
        "m",
        "entropy",
        "excl_days",
        "overlap_days",
        "alt",
        "handoff",
        "stealth_acc",
        "gap_alt_2",
    )
    for w in WINDOWS_1262
] + [
    "email_count_to_p90_ratio",
    "usb_copy_to_p90_ratio",
    "http_upload_to_p90_ratio",
]

SOURCE_1262 = behavior_features_nr27_12

assert SOURCE_1262.shape == (387162, 142)
assert len(EXPECTED_1262) == 27
assert len(set(EXPECTED_1262)) == 27
assert set(EXPECTED_1262).issubset(SOURCE_1262.columns)
assert not SOURCE_1262.duplicated(["user", "day"]).any()


# ------------------------------------------------------------
# STEP 1 — Reproducible sampling
#
# Combine population-wide sampling with users who have
# USB/HTTP activity and missing P90 baseline statistics.
# ------------------------------------------------------------

user_profiles_1262 = SOURCE_1262.groupby(
    "user", sort=True
).agg(
    usb_activity=("usb_copy_count", "sum"),
    http_activity=("www_upload_count", "sum"),
    usb_p90=("usb_copy_p90", "first"),
    http_p90=("http_upload_p90", "first"),
)

all_users_1262 = user_profiles_1262.index.to_numpy()

rng_1262 = np.random.default_rng(1262)

random_users_1262 = rng_1262.choice(
    all_users_1262,
    size=min(40, len(all_users_1262)),
    replace=False,
)

targeted_groups_1262 = [
    user_profiles_1262[
        user_profiles_1262["usb_activity"] > 0
    ].index.to_numpy(),

    user_profiles_1262[
        user_profiles_1262["http_activity"] > 0
    ].index.to_numpy(),

    user_profiles_1262[
        user_profiles_1262["usb_p90"].isna()
    ].index.to_numpy(),

    user_profiles_1262[
        user_profiles_1262["http_p90"].isna()
    ].index.to_numpy(),
]

selected_users_1262 = set(random_users_1262.tolist())

for group in targeted_groups_1262:
    if len(group):
        chosen = rng_1262.choice(
            group,
            size=min(10, len(group)),
            replace=False,
        )
        selected_users_1262.update(chosen.tolist())

selected_users_1262 = sorted(selected_users_1262)

sample_1262 = (
    SOURCE_1262[
        SOURCE_1262["user"].isin(selected_users_1262)
    ]
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

assert sample_1262["user"].nunique() == len(selected_users_1262)
assert sample_1262["is_weekend"].eq(0).all()
assert sample_1262["logon_count"].gt(0).all()
assert pd.to_datetime(sample_1262["day"]).dt.year.eq(2011).all()


# ------------------------------------------------------------
# STEP 2 — Independent reference implementation
#
# Simple direct loops and window slices.
# Every feature is recalculated from source counts.
# ------------------------------------------------------------

reference_rows_1262 = []

COUNT_COLS_1262 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

P50_COLS_1262 = [
    "email_count_p50",
    "usb_copy_p50",
    "http_upload_p50",
]

P90_COLS_1262 = [
    "email_count_p90",
    "usb_copy_p90",
    "http_upload_p90",
]

P95_COLS_1262 = [
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95",
]

RATIO_COLS_1262 = [
    "email_count_to_p90_ratio",
    "usb_copy_to_p90_ratio",
    "http_upload_to_p90_ratio",
]


for user, group in sample_1262.groupby("user", sort=False):

    group = group.reset_index(drop=True)
    n = len(group)

    counts = group[COUNT_COLS_1262].to_numpy(dtype=float)
    p50 = group[P50_COLS_1262].to_numpy(dtype=float)
    p90 = group[P90_COLS_1262].to_numpy(dtype=float)
    p95 = group[P95_COLS_1262].to_numpy(dtype=float)

    # Independent daily deviation calculation.
    deviation = np.zeros((n, 3), dtype=float)

    for day_idx in range(n):
        for channel_idx in range(3):

            observed = counts[day_idx, channel_idx]
            median = p50[day_idx, channel_idx]
            percentile95 = p95[day_idx, channel_idx]

            # Missing count or baseline -> no deviation.
            if not (
                np.isfinite(observed)
                and np.isfinite(median)
                and np.isfinite(percentile95)
            ):
                continue

            numerator = max(0.0, observed - median)
            denominator = max(
                1.0,
                percentile95 - median,
            )

            deviation[day_idx, channel_idx] = (
                numerator / denominator
            )

    is_deviation_day = np.any(deviation > 0, axis=1)

    # The smallest index wins ties:
    # EMAIL=0, USB=1, HTTP=2.
    dominant = np.full(n, -1, dtype=int)

    for day_idx in range(n):
        if is_deviation_day[day_idx]:
            dominant[day_idx] = int(
                np.argmax(deviation[day_idx])
            )

    for t in range(n):

        result = {
            "user": user,
            "day": group.loc[t, "day"],
        }

        # ----------------------------------------------------
        # A — Three daily P90 ratios
        # ----------------------------------------------------

        for c, feature_name in enumerate(RATIO_COLS_1262):

            denominator = p90[t, c]

            if np.isfinite(denominator) and denominator > 0:
                result[feature_name] = (
                    counts[t, c] / denominator
                )
            else:
                result[feature_name] = np.nan

        # ----------------------------------------------------
        # B — Eight feature families × three windows
        # ----------------------------------------------------

        for w in WINDOWS_1262:

            left = max(0, t - w + 1)
            indices = list(range(left, t + 1))

            # 1. Channel-wise cumulative deviations.
            channel_totals = [
                sum(
                    float(deviation[d, c])
                    for d in indices
                )
                for c in range(3)
            ]

            overall_total = sum(channel_totals)

            result[f"nr_m_{w}wd"] = sum(
                total > 0
                for total in channel_totals
            )

            # 2. Normalized entropy.
            if overall_total > 0:
                entropy = 0.0

                for total in channel_totals:
                    if total > 0:
                        probability = total / overall_total
                        entropy -= (
                            probability * np.log(probability)
                        )

                entropy /= np.log(3.0)
            else:
                entropy = 0.0

            result[f"nr_entropy_{w}wd"] = entropy

            # 3 and 4. Exclusive and overlapping days.
            exclusive_days = 0
            overlapping_days = 0

            for d in indices:

                active_channels = sum(
                    deviation[d, c] > 0
                    for c in range(3)
                )

                if active_channels == 1:
                    exclusive_days += 1

                if active_channels >= 2:
                    overlapping_days += 1

            result[f"nr_excl_days_{w}wd"] = exclusive_days
            result[f"nr_overlap_days_{w}wd"] = overlapping_days

            # 5. Adjacent-workday alternation.
            valid_adjacent_pairs = 0
            adjacent_changes = 0

            for d in range(left + 1, t + 1):

                if (
                    is_deviation_day[d - 1]
                    and is_deviation_day[d]
                ):
                    valid_adjacent_pairs += 1

                    if dominant[d - 1] != dominant[d]:
                        adjacent_changes += 1

            result[f"nr_alt_{w}wd"] = (
                adjacent_changes / valid_adjacent_pairs
                if valid_adjacent_pairs
                else 0.0
            )

            # 6. Handoff.
            #
            # The predecessor may be outside the window,
            # but must exist in the 2011 workday sequence.
            handoff = 0

            for d in indices:

                if d == 0:
                    continue

                previous_active = any(
                    deviation[d - 1, c] > 0
                    for c in range(3)
                )

                if not previous_active:
                    continue

                for c in range(3):

                    if (
                        deviation[d, c] > 0
                        and deviation[d - 1, c] == 0
                    ):
                        handoff += 1

            result[f"nr_handoff_{w}wd"] = handoff

            # 7. Stealth accumulation.
            stealth = 0.0

            for d in indices:
                for c in range(3):

                    if (
                        deviation[d, c] > 0
                        and np.isfinite(p95[d, c])
                        and counts[d, c] <= p95[d, c]
                    ):
                        stealth += deviation[d, c]

            result[f"nr_stealth_acc_{w}wd"] = stealth

            # 8. Gap-tolerant alternation.
            #
            # Only consecutive deviation days in the
            # filtered list are considered.
            deviation_indices = [
                d for d in indices
                if is_deviation_day[d]
            ]

            valid_gap_pairs = 0
            gap_changes = 0

            for k in range(1, len(deviation_indices)):

                previous = deviation_indices[k - 1]
                current = deviation_indices[k]

                quiet_days = current - previous - 1

                if quiet_days <= 2:
                    valid_gap_pairs += 1

                    if dominant[previous] != dominant[current]:
                        gap_changes += 1

            result[f"nr_gap_alt_2_{w}wd"] = (
                gap_changes / valid_gap_pairs
                if valid_gap_pairs
                else 0.0
            )

        reference_rows_1262.append(result)


# ------------------------------------------------------------
# STEP 3 — Assemble independent results
# ------------------------------------------------------------

reference_1262 = pd.DataFrame(reference_rows_1262)

assert len(reference_1262) == len(sample_1262)
assert set(EXPECTED_1262).issubset(reference_1262.columns)

comparison_1262 = sample_1262[
    ["user", "day", *EXPECTED_1262]
].merge(
    reference_1262,
    on=["user", "day"],
    how="outer",
    validate="one_to_one",
    suffixes=("_actual", "_reference"),
    indicator=True,
)

assert comparison_1262["_merge"].eq("both").all()
assert len(comparison_1262) == len(sample_1262)


# ------------------------------------------------------------
# STEP 4 — Compare every feature value
# ------------------------------------------------------------

qa_rows_1262 = []
mismatch_examples_1262 = []

for feature in EXPECTED_1262:

    actual = comparison_1262[
        f"{feature}_actual"
    ].to_numpy(dtype=float)

    expected = comparison_1262[
        f"{feature}_reference"
    ].to_numpy(dtype=float)

    nan_mismatch = np.isnan(actual) != np.isnan(expected)

    equal = np.isclose(
        actual,
        expected,
        rtol=TOL_1262,
        atol=TOL_1262,
        equal_nan=True,
    )

    mismatch = ~equal

    qa_rows_1262.append({
        "feature": feature,
        "tested_values": len(actual),
        "value_mismatches": int(mismatch.sum()),
        "nan_mismatches": int(nan_mismatch.sum()),
        "status": "PASS" if not mismatch.any() else "FAIL",
    })

    if mismatch.any():

        bad = comparison_1262.loc[
            mismatch,
            [
                "user",
                "day",
                f"{feature}_actual",
                f"{feature}_reference",
            ],
        ].head(5).copy()

        bad.insert(0, "feature", feature)

        mismatch_examples_1262.append(bad)


qa_1262 = pd.DataFrame(qa_rows_1262)

total_tested_1262 = int(
    qa_1262["tested_values"].sum()
)

total_mismatches_1262 = int(
    qa_1262["value_mismatches"].sum()
)

total_nan_mismatches_1262 = int(
    qa_1262["nan_mismatches"].sum()
)


# ------------------------------------------------------------
# STEP 5 — Display independent QA
# ------------------------------------------------------------

print("\nA — INDEPENDENT VERIFICATION BY FEATURE")
display(qa_1262)

print("\nB — MISMATCH EXAMPLES")

if mismatch_examples_1262:
    display(
        pd.concat(
            mismatch_examples_1262,
            ignore_index=True
        )
    )
else:
    print("No numerical or NaN mismatches found.")

print("\n" + "=" * 62)
print("12.6.2 — INDEPENDENT VERIFICATION SUMMARY")
print("=" * 62)
print("Sampled users             :", len(selected_users_1262))
print("Sampled User-Days         :", len(sample_1262))
print("New Rotation features     :", len(EXPECTED_1262))
print("Total values tested       :", total_tested_1262)
print("Numerical mismatches      :", total_mismatches_1262)
print("NaN mismatches            :", total_nan_mismatches_1262)
print("Window policy             : 2011 PARTIAL")
print("Original dataset modified : NO")
print("Files exported            : NO")

if (
    len(qa_1262) == 27
    and total_mismatches_1262 == 0
    and total_nan_mismatches_1262 == 0
):
    print("STATUS: PASS — INDEPENDENT NR VERIFICATION")
else:
    print("STATUS: FAIL — REVIEW DIFFERENCES")

print("=" * 62)



A — INDEPENDENT VERIFICATION BY FEATURE


,feature,tested_values,value_mismatches,nan_mismatches,status
0,nr_m_5wd,8382,0,0,PASS
1,nr_m_10wd,8382,0,0,PASS
2,nr_m_20wd,8382,0,0,PASS
3,nr_entropy_5wd,8382,0,0,PASS
4,nr_entropy_10wd,8382,0,0,PASS
5,nr_entropy_20wd,8382,0,0,PASS
6,nr_excl_days_5wd,8382,0,0,PASS
7,nr_excl_days_10wd,8382,0,0,PASS
8,nr_excl_days_20wd,8382,0,0,PASS
9,nr_overlap_days_5wd,8382,0,0,PASS



B — MISMATCH EXAMPLES
No numerical or NaN mismatches found.

12.6.2 — INDEPENDENT VERIFICATION SUMMARY
Sampled users             : 80
Sampled User-Days         : 8382
New Rotation features     : 27
Total values tested       : 226314
Numerical mismatches      : 0
NaN mismatches            : 0
Window policy             : 2011 PARTIAL
Original dataset modified : NO
Files exported            : NO
STATUS: PASS — INDEPENDENT NR VERIFICATION



## 12.6.3 — Independent Feature Dependency and Source Reconciliation Audit

### Objective
Independently verify that all original Data Engineering features are consistent with the injected User-Day data and immutable historical baseline.

### Independent Sources
- Notebook 10: `user_day_injected_counts_r62.parquet`
- Original Data Engineering baseline: `user_baseline_r62_v1.parquet`
- Current 142-column dataset: `behavior_features_nr27_12`
- Original Data Engineering formulas from `04_build_behavior_features_r62`

### Verification
1. Reconcile all 34 source User-Day columns.
2. Reconcile all 23 historical baseline columns.
3. Independently recompute 12 daily behavioral indicators.
4. Independently recompute 16 rolling features using partial workday windows.
5. Verify primary keys, eligible workdays and schema.
6. Identify numerical and missing-value discrepancies by column.

### Acceptance Criteria
- All 85 original columns covered.
- Zero source mismatches.
- Zero baseline mismatches.
- Zero daily and rolling feature mismatches.
- No duplicate keys or unintended columns.
- No data modification or export.

The 57 newly engineered features are already covered by their dedicated numerical verification tests. This audit specifically targets potential stale dependencies in the original 85-column feature set.


In [49]:

# ============================================================
# 12.6.3 — INDEPENDENT FEATURE DEPENDENCY AUDIT
# READ-ONLY — NO MODIFICATION / NO EXPORT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# 1 — Resolve original sources independently

def find_root_1263():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "reports" / "10_Attack_Injection_Pipeline").is_dir():
            return p
    raise FileNotFoundError("New_Attack_Simulation root not found.")

ROOT_1263 = find_root_1263()

INJECTED_PATH_1263 = (
    ROOT_1263 / "reports" / "10_Attack_Injection_Pipeline"
    / "user_day_injected_counts_r62.parquet"
)

BASELINE_PATH_1263 = Path(baseline_path_122)

assert INJECTED_PATH_1263.is_file()
assert BASELINE_PATH_1263.is_file()

# Do not use prior reconstructed data or QA results.
df_1263 = behavior_features_nr27_12.copy()

assert df_1263.shape == (387162, 142)
assert df_1263.columns.is_unique

BASELINE_COLS_1263 = [
    "baseline_workdays",
    "email_active_days", "email_active_rate",
    "email_count_p50", "email_count_p90",
    "email_count_p95", "email_count_mad",
    "email_bytes_p50", "email_bytes_p90",
    "email_bytes_p95", "email_bytes_mad",
    "usb_active_days", "usb_active_rate",
    "usb_copy_p50", "usb_copy_p90",
    "usb_copy_p95", "usb_copy_mad",
    "http_upload_active_days", "http_upload_active_rate",
    "http_upload_p50", "http_upload_p90",
    "http_upload_p95", "http_upload_mad",
]

DAILY_COLS_1263 = [
    "email_active_today",
    "usb_active_today",
    "http_upload_active_today",
    "active_channel_count",
    "email_count_above_p95",
    "email_bytes_above_p95",
    "usb_copy_above_p95",
    "http_upload_above_p95",
    "email_count_to_p95_ratio",
    "email_bytes_to_p95_ratio",
    "usb_copy_to_p95_ratio",
    "http_upload_to_p95_ratio",
]

ROLLING_COLS_1263 = (
    [
        f"{prefix}_{w}wd"
        for prefix in (
            "email_active_days",
            "usb_active_days",
            "http_active_days",
        )
        for w in (5, 10, 20)
    ]
    + [
        f"{prefix}_{w}wd"
        for prefix in (
            "external_email_count",
            "usb_copy_count",
            "http_upload_count",
        )
        for w in (5, 10)
    ]
    + ["multi_channel_days_10wd"]
)

# Read source schema, not a previous notebook's column lists.
import pyarrow.parquet as pq

SOURCE_COLS_1263 = pq.ParquetFile(
    INJECTED_PATH_1263
).schema_arrow.names

assert len(SOURCE_COLS_1263) == 34, (
    f"Unexpected User-Day schema: {len(SOURCE_COLS_1263)}"
)

assert len(BASELINE_COLS_1263) == 23
assert len(DAILY_COLS_1263) == 12
assert len(ROLLING_COLS_1263) == 16

ORIGINAL_85_1263 = (
    set(SOURCE_COLS_1263)
    | set(BASELINE_COLS_1263)
    | set(DAILY_COLS_1263)
    | set(ROLLING_COLS_1263)
)

assert len(ORIGINAL_85_1263) == 85
assert ORIGINAL_85_1263.issubset(df_1263.columns)

# 2 — Independently load actual injected User-Day and baseline

source_1263 = pd.read_parquet(INJECTED_PATH_1263)
baseline_1263 = pd.read_parquet(
    BASELINE_PATH_1263,
    columns=["user", *BASELINE_COLS_1263]
)

for frame in (df_1263, source_1263):
    frame["user"] = frame["user"].astype("string")
    frame["day"] = pd.to_datetime(frame["day"]).dt.normalize()

baseline_1263["user"] = baseline_1263["user"].astype("string")

assert not df_1263.duplicated(["user", "day"]).any()
assert not source_1263.duplicated(["user", "day"]).any()
assert not baseline_1263["user"].duplicated().any()

assert df_1263["day"].dt.year.eq(2011).all()
assert df_1263["is_weekend"].eq(0).all()
assert df_1263["logon_count"].gt(0).all()

source_1263 = source_1263.merge(
    df_1263[["user", "day"]],
    on=["user", "day"],
    how="inner",
    validate="one_to_one",
)

assert len(source_1263) == len(df_1263), (
    "Some feature keys are absent from injected User-Day."
)

# 3 — Prepare independent expected source + baseline

source_1263 = source_1263.set_index(["user", "day"])
actual_1263 = df_1263.set_index(["user", "day"])

source_1263 = source_1263.reindex(actual_1263.index)

baseline_1263 = baseline_1263.set_index("user")

assert actual_1263.index.get_level_values("user").isin(
    baseline_1263.index
).all()

baseline_expected_1263 = baseline_1263.reindex(
    actual_1263.index.get_level_values("user")
)
baseline_expected_1263.index = actual_1263.index

# Compare with matching NaN semantics.
audit_rows_1263 = []
examples_1263 = []

def compare_1263(name, expected, category):
    observed = actual_1263[name]
    expected = pd.Series(
        expected,
        index=actual_1263.index,
    )

    if (
        pd.api.types.is_numeric_dtype(observed)
        and pd.api.types.is_numeric_dtype(expected)
    ):
        equal = np.isclose(
            observed.to_numpy(dtype=float, na_value=np.nan),
            expected.to_numpy(dtype=float, na_value=np.nan),
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True,
        )
    else:
        equal = (
            observed.eq(expected).fillna(False)
            | (observed.isna() & expected.isna())
        ).to_numpy(dtype=bool)

    mismatches = int((~equal).sum())

    audit_rows_1263.append({
        "category": category,
        "column": name,
        "rows_checked": len(equal),
        "mismatches": mismatches,
        "status": "PASS" if mismatches == 0 else "FAIL",
    })

    if mismatches:
        bad = pd.DataFrame({
            "actual": observed[~equal].astype("string").values,
            "expected": expected[~equal].astype("string").values,
        }, index=actual_1263.index[~equal])

        bad = bad.head(5).reset_index()
        bad.insert(0, "column", name)
        bad.insert(0, "category", category)
        examples_1263.append(bad)

# 4 — All 34 original source columns

for col in SOURCE_COLS_1263:
    if col not in ("user", "day"):
        compare_1263(col, source_1263[col], "SOURCE")

for col in ("user", "day"):
    audit_rows_1263.append({
        "category": "SOURCE_KEY",
        "column": col,
        "rows_checked": len(actual_1263),
        "mismatches": 0,
        "status": "PASS",
    })

# 5 — All 23 historical baseline columns

for col in BASELINE_COLS_1263:
    compare_1263(
        col,
        baseline_expected_1263[col],
        "BASELINE",
    )

# 6 — Rebuild 12 daily indicators independently

x = source_1263
b = baseline_expected_1263
daily_1263 = pd.DataFrame(index=actual_1263.index)

channels = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count",
}

for channel, count_col in channels.items():
    active_col = {
        "email": "email_active_today",
        "usb": "usb_active_today",
        "http": "http_upload_active_today",
    }[channel]

    daily_1263[active_col] = x[count_col].gt(0).astype(int)

daily_1263["active_channel_count"] = (
    daily_1263["email_active_today"]
    + daily_1263["usb_active_today"]
    + daily_1263["http_upload_active_today"]
)

comparisons_1263 = {
    "email_count": ("external_email_count", "email_count_p95"),
    "email_bytes": ("external_email_bytes", "email_bytes_p95"),
    "usb_copy": ("usb_copy_count", "usb_copy_p95"),
    "http_upload": ("www_upload_count", "http_upload_p95"),
}

for prefix, (count_col, p95_col) in comparisons_1263.items():

    daily_1263[f"{prefix}_above_p95"] = (
        b[p95_col].notna()
        & x[count_col].gt(b[p95_col])
    ).astype(int)

    # Match the original SQL's NULL behavior.
    valid = b[p95_col].gt(0)

    daily_1263[f"{prefix}_to_p95_ratio"] = (
        x[count_col] / b[p95_col].where(valid)
    )

for col in DAILY_COLS_1263:
    compare_1263(col, daily_1263[col], "DAILY")

# 7 — Independently calculate all 16 rolling features

work_1263 = x.sort_index().copy()

for col in (
    "email_active_today",
    "usb_active_today",
    "http_upload_active_today",
    "active_channel_count",
):
    work_1263[col] = daily_1263[col]

work_1263["multi_channel_indicator"] = (
    work_1263["active_channel_count"] >= 2
).astype(int)

rolling_map_1263 = {
    "email_active_today": "email_active_days",
    "usb_active_today": "usb_active_days",
    "http_upload_active_today": "http_active_days",
    "external_email_count": "external_email_count",
    "usb_copy_count": "usb_copy_count",
    "www_upload_count": "http_upload_count",
    "multi_channel_indicator": "multi_channel_days",
}

for window in (5, 10, 20):

    if window == 20:
        input_cols = [
            "email_active_today",
            "usb_active_today",
            "http_upload_active_today",
        ]
    elif window == 10:
        input_cols = list(rolling_map_1263)
    else:
        input_cols = [
            "email_active_today",
            "usb_active_today",
            "http_upload_active_today",
            "external_email_count",
            "usb_copy_count",
            "www_upload_count",
        ]

    expected_roll = (
        work_1263.groupby(level="user")[input_cols]
        .rolling(window, min_periods=1)
        .sum()
        .droplevel(0)
    )

    for input_col in input_cols:

        if input_col == "multi_channel_indicator":
            feature_name = "multi_channel_days_10wd"
        else:
            prefix = rolling_map_1263[input_col]
            feature_name = f"{prefix}_{window}wd"

        expected_values = expected_roll[
            input_col
        ].reindex(actual_1263.index)

        compare_1263(
            feature_name,
            expected_values,
            "ROLLING",
        )

# 8 — Final audit coverage and mismatch report

audit_1263 = pd.DataFrame(audit_rows_1263)

assert len(audit_1263) == 85
assert audit_1263["column"].nunique() == 85

audit_summary_1263 = (
    audit_1263.groupby("category", as_index=False)
    .agg(
        columns=("column", "count"),
        checked_values=("rows_checked", "sum"),
        mismatches=("mismatches", "sum"),
    )
)

audit_summary_1263["status"] = np.where(
    audit_summary_1263["mismatches"].eq(0),
    "PASS",
    "FAIL",
)

print("\nA — INDEPENDENT DEPENDENCY AUDIT")
display(audit_summary_1263)

print("\nB — FAILED COLUMNS")
failed_1263 = audit_1263[
    audit_1263["mismatches"] > 0
]
display(failed_1263)

print("\nC — INDEPENDENT MISMATCH EXAMPLES")
if examples_1263:
    display(pd.concat(examples_1263, ignore_index=True))
else:
    print("No mismatches found.")

print("\n" + "=" * 60)
print("12.6.3 — FINAL AUDIT SUMMARY")
print("=" * 60)
print("Rows inspected             :", len(df_1263))
print("Users inspected            :", df_1263["user"].nunique())
print("Original columns checked   :", len(audit_1263))
print("Total values checked       :", audit_1263["rows_checked"].sum())
print("Total mismatches           :", audit_1263["mismatches"].sum())
print("Baseline modified          : NO")
print("Dataset modified           : NO")
print("Files exported             : NO")

if audit_1263["mismatches"].sum() == 0:
    print("STATUS: PASS — INDEPENDENT DEPENDENCY AUDIT")
else:
    print("STATUS: REVIEW REQUIRED — MISMATCHES FOUND")

print("=" * 60)



A — INDEPENDENT DEPENDENCY AUDIT


,category,columns,checked_values,mismatches,status
0,BASELINE,23,8904726,0,PASS
1,DAILY,12,4645944,0,PASS
2,ROLLING,16,6194592,0,PASS
3,SOURCE,32,12389184,0,PASS
4,SOURCE_KEY,2,774324,0,PASS



B — FAILED COLUMNS


,category,column,rows_checked,mismatches,status



C — INDEPENDENT MISMATCH EXAMPLES
No mismatches found.

12.6.3 — FINAL AUDIT SUMMARY
Rows inspected             : 387162
Users inspected            : 3737
Original columns checked   : 85
Total values checked       : 32908770
Total mismatches           : 0
Baseline modified          : NO
Dataset modified           : NO
Files exported             : NO
STATUS: PASS — INDEPENDENT DEPENDENCY AUDIT



## 12.6.4 — Independent Full Dataset Integrity Audit

### Objective

Conduct an independent, full-population audit of the 142-column behavioral feature dataset before binary labeling.

### Scope

- All 387,162 User-Day records.
- All 3,737 users.
- The original 85 Data Engineering columns.
- The 30 Clear features.
- The 27 New Rotation features.

### Methodology

1. **Structure and keys:** Verify dataset shape, unique user-day keys, eligible workdays, chronological coverage, and absence of target or attack-scheduling columns.
2. **Numerical integrity:** Detect infinite values, invalid negative counts, and unexpected nonnumeric feature values.
3. **Historical baseline:** Verify per-user consistency of the 23 historical statistics and percentile ordering where values exist.
4. **Cross-feature relationships:** Check daily activity flags, channel counts, P95/P90 ratios, and independently derived New Rotation relationships.
5. **Temporal integrity:** Recalculate selected rolling relationships from daily counts and deviations across the full population, using partial 2011 workday windows.
6. **Feature profiles:** Inspect null rates, constant columns, and unusual feature distributions.
7. **Anomaly review:** Distinguish confirmed mathematical failures from suspicious but potentially valid distributions.

### Acceptance Criteria

- Zero structural, key, or mathematical integrity failures.
- All 142 columns inspected.
- All full-population consistency checks pass.
- Review flags are explained before final approval.

### Restrictions

- Do not modify the original dataset or historical baseline.
- Do not reuse earlier QA results as evidence.
- Do not access Ground Truth or add `y`.
- Do not export temporary files.

### Output

In-memory audit summaries, feature profiles, and failure details.

A successful audit establishes internal integrity within the tested rules; it does not replace the later independent leakage and binary-label validation in Section 12.8.


In [50]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.6.4
# INDEPENDENT FULL DATASET INTEGRITY AUDIT
#
# Corrected version:
# - Recognize first_logon and last_logoff as datetime fields
# - Keep constant features as REVIEW, not automatic FAIL
# - Validate all 142 columns across the full population
#
# READ-ONLY / NO GROUND TRUTH / NO EXPORTS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# STEP 1 — Independent working copy and audit recorder
# ------------------------------------------------------------

audit_input_1264 = behavior_features_nr27_12

df_1264 = audit_input_1264.copy(deep=True)

results_1264 = []
TOL_1264 = 1e-9


def record_1264(category, check, failures, details=""):
    failures = int(failures)

    results_1264.append({
        "category": category,
        "check": check,
        "failures": failures,
        "status": "PASS" if failures == 0 else "FAIL",
        "details": details,
    })


def mismatch_1264(actual, expected):
    a = np.asarray(actual, dtype="float64")
    b = np.asarray(expected, dtype="float64")

    return ~np.isclose(
        a,
        b,
        rtol=TOL_1264,
        atol=TOL_1264,
        equal_nan=True,
    )


# ------------------------------------------------------------
# STEP 2 — Structure, schema and temporal eligibility
# ------------------------------------------------------------

record_1264(
    "STRUCTURE",
    "Expected shape",
    int(df_1264.shape != (387162, 142))
)

record_1264(
    "STRUCTURE",
    "Duplicate column names",
    df_1264.columns.duplicated().sum()
)

record_1264(
    "STRUCTURE",
    "Duplicate User-Day keys",
    df_1264.duplicated(["user", "day"]).sum()
)

record_1264(
    "STRUCTURE",
    "Missing User-Day keys",
    df_1264[["user", "day"]].isna().any(axis=1).sum()
)

record_1264(
    "STRUCTURE",
    "Expected user population",
    int(df_1264["user"].nunique() != 3737)
)

day_1264 = pd.to_datetime(
    df_1264["day"],
    errors="coerce"
)

record_1264(
    "TEMPORAL",
    "Valid 2011 attack-period dates",
    (~day_1264.between(
        "2011-01-01",
        "2011-06-01"
    )).sum()
)

record_1264(
    "TEMPORAL",
    "Eligible weekday",
    df_1264["is_weekend"].ne(0).sum()
)

record_1264(
    "TEMPORAL",
    "Positive logon activity",
    (
        df_1264["logon_count"].le(0)
        | df_1264["logon_count"].isna()
    ).sum()
)

record_1264(
    "TEMPORAL",
    "Actual calendar weekday",
    (
        day_1264.dt.dayofweek.ge(5)
        | day_1264.isna()
    ).sum()
)

forbidden_1264 = {
    "y",
    "label",
    "is_attack",
    "attack_label",
    "scenario_id",
    "injection_count",
    "attack_start",
    "attack_end",
    "stealth_level",
    "assigned_scenario",
}

record_1264(
    "SCHEMA",
    "Forbidden target/scheduling columns",
    len(
        set(df_1264.columns)
        & forbidden_1264
    )
)

# Sort only the independent working copy.
df_1264["day"] = day_1264

df_1264 = (
    df_1264
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

groups_1264 = df_1264.groupby(
    "user",
    sort=False
)


# ------------------------------------------------------------
# STEP 3 — Full-column numerical and datetime integrity
# ------------------------------------------------------------

profile_rows_1264 = []

# These are identifiers or dates, not numeric features.
identifier_cols_1264 = {"user", "day"}

# Original temporal fields are expected to be datetime.
expected_datetime_cols_1264 = {
    "first_logon",
    "last_logoff",
}

for col in df_1264.columns:

    s = df_1264[col]

    numeric = pd.api.types.is_numeric_dtype(s)

    datetime_col = pd.api.types.is_datetime64_any_dtype(s)

    inf_count = 0

    if numeric:
        values = s.to_numpy(
            dtype="float64",
            na_value=np.nan
        )

        inf_count = int(np.isinf(values).sum())

    profile_rows_1264.append({
        "column": col,
        "dtype": str(s.dtype),
        "null_rows": int(s.isna().sum()),
        "null_pct": round(
            s.isna().mean() * 100,
            3
        ),
        "unique_values": int(
            s.nunique(dropna=True)
        ),
        "infinite_values": inf_count,
        "constant_non_null": (
            s.nunique(dropna=True) <= 1
        ),
        "is_numeric": bool(numeric),
        "is_datetime": bool(datetime_col),
    })

profile_1264 = pd.DataFrame(profile_rows_1264)

record_1264(
    "NUMERICAL",
    "Infinite values",
    profile_1264["infinite_values"].sum()
)

# Corrected validation:
# Datetime columns are legitimate, not unexpected
# nonnumeric features.
unexpected_types_1264 = [
    col
    for col in df_1264.columns
    if (
        col not in identifier_cols_1264
        and col not in expected_datetime_cols_1264
        and not pd.api.types.is_numeric_dtype(
            df_1264[col]
        )
    )
]

record_1264(
    "NUMERICAL",
    "Unexpected nonnumeric feature columns",
    len(unexpected_types_1264),
    str(unexpected_types_1264)
)

# Ensure both expected temporal fields genuinely
# contain datetime values.
for col in sorted(expected_datetime_cols_1264):

    valid_datetime_type = (
        col in df_1264.columns
        and pd.api.types.is_datetime64_any_dtype(
            df_1264[col]
        )
    )

    record_1264(
        "TEMPORAL",
        f"Datetime schema: {col}",
        int(not valid_datetime_type)
    )

    if valid_datetime_type:

        record_1264(
            "TEMPORAL",
            f"Valid datetime values: {col}",
            df_1264[col].isna().sum()
        )

# ------------------------------------------------------------
# STEP 4 — Numerical event-count integrity
# ------------------------------------------------------------

count_cols_1264 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
    "email_event_count",
    "file_event_count",
    "http_event_count",
    "logon_count",
]

for col in count_cols_1264:

    values = df_1264[col].to_numpy(
        dtype=float
    )

    invalid = (
        ~np.isfinite(values)
        | (values < 0)
    )

    record_1264(
        "COUNTS",
        f"Nonnegative finite {col}",
        invalid.sum()
    )


# ------------------------------------------------------------
# STEP 5 — Historical baseline consistency
# ------------------------------------------------------------

baseline_cols_1264 = [
    "baseline_workdays",
    "email_active_days",
    "email_active_rate",
    "email_count_p50",
    "email_count_p90",
    "email_count_p95",
    "email_count_mad",
    "email_bytes_p50",
    "email_bytes_p90",
    "email_bytes_p95",
    "email_bytes_mad",
    "usb_active_days",
    "usb_active_rate",
    "usb_copy_p50",
    "usb_copy_p90",
    "usb_copy_p95",
    "usb_copy_mad",
    "http_upload_active_days",
    "http_upload_active_rate",
    "http_upload_p50",
    "http_upload_p90",
    "http_upload_p95",
    "http_upload_mad",
]

assert len(baseline_cols_1264) == 23

for col in baseline_cols_1264:

    unique_per_user = groups_1264[col].nunique(
        dropna=False
    )

    record_1264(
        "BASELINE",
        f"Constant per user: {col}",
        unique_per_user.gt(1).sum()
    )

for prefix in [
    "email_count",
    "email_bytes",
    "usb_copy",
    "http_upload",
]:

    p50 = df_1264[f"{prefix}_p50"]
    p90 = df_1264[f"{prefix}_p90"]
    p95 = df_1264[f"{prefix}_p95"]

    valid = (
        p50.notna()
        & p90.notna()
        & p95.notna()
    )

    inconsistent = valid & (
        p50.gt(p90)
        | p90.gt(p95)
    )

    record_1264(
        "BASELINE",
        f"Percentile ordering: {prefix}",
        inconsistent.sum()
    )


# ------------------------------------------------------------
# STEP 6 — Independent daily activity relationships
# ------------------------------------------------------------

channel_specs_1264 = [
    (
        "external_email_count",
        "email_active_today",
        "email_count_p95",
        "email_count_to_p90_ratio",
        "email_count_p90",
    ),
    (
        "usb_copy_count",
        "usb_active_today",
        "usb_copy_p95",
        "usb_copy_to_p90_ratio",
        "usb_copy_p90",
    ),
    (
        "www_upload_count",
        "http_upload_active_today",
        "http_upload_p95",
        "http_upload_to_p90_ratio",
        "http_upload_p90",
    ),
]

for (
    count_col,
    flag_col,
    p95_col,
    ratio_col,
    p90_col
) in channel_specs_1264:

    expected_flag = (
        df_1264[count_col]
        .gt(0)
        .astype(int)
    )

    record_1264(
        "DAILY",
        f"Activity flag: {flag_col}",
        mismatch_1264(
            df_1264[flag_col],
            expected_flag
        ).sum()
    )

    valid_p90 = df_1264[p90_col].where(
        df_1264[p90_col].gt(0)
    )

    expected_ratio = (
        df_1264[count_col] / valid_p90
    )

    record_1264(
        "DAILY",
        f"P90 ratio: {ratio_col}",
        mismatch_1264(
            df_1264[ratio_col],
            expected_ratio
        ).sum()
    )

expected_active_1264 = (
    df_1264["external_email_count"]
    .gt(0).astype(int)

    + df_1264["usb_copy_count"]
    .gt(0).astype(int)

    + df_1264["www_upload_count"]
    .gt(0).astype(int)
)

record_1264(
    "DAILY",
    "Active channel count",
    mismatch_1264(
        df_1264["active_channel_count"],
        expected_active_1264
    ).sum()
)


# ------------------------------------------------------------
# STEP 7 — Reconstruct daily New Rotation deviation
# ------------------------------------------------------------

counts_1264 = df_1264[
    [
        "external_email_count",
        "usb_copy_count",
        "www_upload_count",
    ]
].to_numpy(dtype=float)

p50_1264 = df_1264[
    [
        "email_count_p50",
        "usb_copy_p50",
        "http_upload_p50",
    ]
].to_numpy(dtype=float)

p95_1264 = df_1264[
    [
        "email_count_p95",
        "usb_copy_p95",
        "http_upload_p95",
    ]
].to_numpy(dtype=float)

valid_1264 = (
    np.isfinite(counts_1264)
    & np.isfinite(p50_1264)
    & np.isfinite(p95_1264)
)

E_1264 = np.zeros_like(
    counts_1264,
    dtype=float
)

numerator_1264 = np.maximum(
    0.0,
    counts_1264 - p50_1264
)

denominator_1264 = np.maximum(
    1.0,
    p95_1264 - p50_1264
)

np.divide(
    numerator_1264,
    denominator_1264,
    out=E_1264,
    where=valid_1264
)

deviation_count_1264 = (
    E_1264 > 0
).sum(axis=1)


# ------------------------------------------------------------
# STEP 8 — Independently test full-population NR rolling
# ------------------------------------------------------------

temp_1264 = pd.DataFrame({
    "user": df_1264["user"].to_numpy(),
    "E_email": E_1264[:, 0],
    "E_usb": E_1264[:, 1],
    "E_http": E_1264[:, 2],
    "exclusive": (
        deviation_count_1264 == 1
    ).astype(float),
    "overlap": (
        deviation_count_1264 >= 2
    ).astype(float),
})

stealth_1264 = np.where(
    (E_1264 > 0)
    & valid_1264
    & (counts_1264 <= p95_1264),
    E_1264,
    0.0
).sum(axis=1)

temp_1264["stealth"] = stealth_1264

for w in (5, 10, 20):

    roll = (
        temp_1264.groupby("user")[
            [
                "E_email",
                "E_usb",
                "E_http",
                "exclusive",
                "overlap",
                "stealth",
            ]
        ]
        .rolling(
            window=w,
            min_periods=1
        )
        .sum()
        .reset_index(
            level=0,
            drop=True
        )
        .sort_index()
    )

    S = roll[
        [
            "E_email",
            "E_usb",
            "E_http",
        ]
    ].to_numpy(dtype=float)

    total = S.sum(axis=1)

    expected_m = (
        S > 1e-12
    ).sum(axis=1)

    record_1264(
        "NR_ROLLING",
        f"nr_m_{w}wd",
        mismatch_1264(
            df_1264[f"nr_m_{w}wd"],
            expected_m
        ).sum()
    )

    expected_entropy = np.zeros(
        len(df_1264),
        dtype=float
    )

    positive_total = total > 0

    probabilities = np.zeros_like(S)

    np.divide(
        S,
        total[:, None],
        out=probabilities,
        where=positive_total[:, None]
    )

    terms = np.zeros_like(
        probabilities
    )

    positive_prob = probabilities > 0

    terms[positive_prob] = (
        probabilities[positive_prob]
        * np.log(
            probabilities[positive_prob]
        )
    )

    expected_entropy[positive_total] = (
        -terms[positive_total].sum(axis=1)
        / np.log(3.0)
    )

    record_1264(
        "NR_ROLLING",
        f"nr_entropy_{w}wd",
        mismatch_1264(
            df_1264[f"nr_entropy_{w}wd"],
            expected_entropy
        ).sum()
    )

    for name, source_name in [
        ("nr_excl_days", "exclusive"),
        ("nr_overlap_days", "overlap"),
        ("nr_stealth_acc", "stealth"),
    ]:

        record_1264(
            "NR_ROLLING",
            f"{name}_{w}wd",
            mismatch_1264(
                df_1264[f"{name}_{w}wd"],
                roll[source_name]
            ).sum()
        )

    for name in (
        "nr_alt",
        "nr_gap_alt_2",
    ):

        values = df_1264[
            f"{name}_{w}wd"
        ]

        invalid = (
            values.isna()
            | ~values.between(
                -TOL_1264,
                1 + TOL_1264
            )
        )

        record_1264(
            "NR_BOUNDS",
            f"{name}_{w}wd in [0,1]",
            invalid.sum()
        )

    handoff = df_1264[
        f"nr_handoff_{w}wd"
    ]

    record_1264(
        "NR_BOUNDS",
        f"nr_handoff_{w}wd bounds",
        (
            handoff.isna()
            | ~handoff.between(0, 3 * w)
        ).sum()
    )

    record_1264(
        "NR_BOUNDS",
        f"nr_excl + overlap <= {w}",
        (
            df_1264[f"nr_excl_days_{w}wd"]
            + df_1264[f"nr_overlap_days_{w}wd"]
            > w + TOL_1264
        ).sum()
    )


# ------------------------------------------------------------
# STEP 9 — Clear feature mathematical bounds
# ------------------------------------------------------------

for w in (5, 10, 20, 30):

    for feature in [
        "cumulative_deviation",
        "combined_channel_intensity",
        "dominant_channel_switches",
        "consecutive_elevated_days",
        "cross_channel_deviation_days",
    ]:

        col = f"{feature}_{w}wd"

        values = df_1264[col]

        invalid = (
            values.isna()
            | values.lt(-TOL_1264)
            | ~np.isfinite(
                values.to_numpy(dtype=float)
            )
        )

        record_1264(
            "CLEAR_BOUNDS",
            f"Nonnegative {col}",
            invalid.sum()
        )

    for feature in [
        "channel_distribution_shift",
    ]:

        col = f"{feature}_{w}wd"

        non_null = (
            df_1264[col]
            .dropna()
        )

        invalid = (
            ~non_null.between(
                -TOL_1264,
                1 + TOL_1264
            )
        )

        record_1264(
            "CLEAR_BOUNDS",
            f"TVD bounds {col}",
            invalid.sum()
        )


# ------------------------------------------------------------
# STEP 10 — Review unusual feature profiles
#
# REVIEW does not automatically mean calculation error.
# ------------------------------------------------------------

review_1264 = profile_1264[
    (
        profile_1264["null_pct"] >= 80
    )
    | (
        profile_1264["constant_non_null"]
    )
].copy()

review_1264["reason"] = np.where(
    review_1264["constant_non_null"],
    "Constant or entirely missing",
    "At least 80% missing"
)

# Explicitly document the observed constant columns.
constant_review_1264 = profile_1264[
    profile_1264["constant_non_null"]
][
    [
        "column",
        "dtype",
        "unique_values",
        "null_rows",
    ]
].copy()


# ------------------------------------------------------------
# STEP 11 — Final QA reports
# ------------------------------------------------------------

qa_1264 = pd.DataFrame(
    results_1264
)

qa_summary_1264 = (
    qa_1264
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        checks=("check", "count"),
        failures=("failures", "sum"),
    )
)

qa_summary_1264["status"] = np.where(
    qa_summary_1264["failures"].eq(0),
    "PASS",
    "FAIL"
)

print("\nA — INDEPENDENT DATASET AUDIT")
display(qa_summary_1264)

print("\nB — FAILED CHECKS")

failed_checks_1264 = qa_1264[
    qa_1264["failures"] > 0
]

if failed_checks_1264.empty:
    print("No failed integrity checks.")
else:
    display(failed_checks_1264)

print("\nC — FEATURE PROFILES REQUIRING REVIEW")
display(review_1264)

print("\nD — CONSTANT FEATURE REVIEW")
display(constant_review_1264)

print("\nE — ALL-COLUMN NULL PROFILE")
display(
    profile_1264.sort_values(
        "null_pct",
        ascending=False
    )[
        [
            "column",
            "dtype",
            "null_rows",
            "null_pct",
            "unique_values",
            "infinite_values",
            "constant_non_null",
        ]
    ]
)

# ------------------------------------------------------------
# STEP 12 — Final independent conclusion
# ------------------------------------------------------------

total_failures_1264 = int(
    qa_1264["failures"].sum()
)

print("\n" + "=" * 64)
print("12.6.4 — INDEPENDENT FULL DATASET AUDIT SUMMARY")
print("=" * 64)

print(
    "Rows                   :",
    len(df_1264)
)
print(
    "Users                  :",
    df_1264["user"].nunique()
)
print(
    "Columns inspected      :",
    len(profile_1264)
)
print(
    "Independent checks     :",
    len(qa_1264)
)
print(
    "Confirmed rule failures:",
    total_failures_1264
)
print(
    "Review-flagged columns :",
    len(review_1264)
)
print(
    "Original modified      : NO"
)
print(
    "Files exported         : NO"
)

if total_failures_1264 == 0:
    print(
        "STATUS: RULE CHECKS PASS — REVIEW PROFILE FLAGS"
    )
else:
    print(
        "STATUS: FAIL — INVESTIGATE IDENTIFIED RULES"
    )

print("=" * 64)



A — INDEPENDENT DATASET AUDIT


,category,checks,failures,status
0,BASELINE,27,0,PASS
1,CLEAR_BOUNDS,24,0,PASS
2,COUNTS,7,0,PASS
3,DAILY,7,0,PASS
4,NR_BOUNDS,12,0,PASS
5,NR_ROLLING,15,0,PASS
6,NUMERICAL,2,0,PASS
7,SCHEMA,1,0,PASS
8,STRUCTURE,5,0,PASS
9,TEMPORAL,8,0,PASS



B — FAILED CHECKS
No failed integrity checks.

C — FEATURE PROFILES REQUIRING REVIEW


,column,dtype,null_rows,null_pct,unique_values,infinite_values,constant_non_null,is_numeric,is_datetime,reason
3,is_weekend,int32,0,0.000,1,0,True,True,False,Constant or entirely missing
23,unique_usb_extensions,int64,0,0.000,1,0,True,True,False,Constant or entirely missing
47,usb_copy_p50,float64,312211,80.641,10,0,False,True,False,At least 80% missing
48,usb_copy_p90,float64,312211,80.641,58,0,False,True,False,At least 80% missing
49,usb_copy_p95,float64,312211,80.641,108,0,False,True,False,At least 80% missing
50,usb_copy_mad,float64,312211,80.641,5,0,False,True,False,At least 80% missing
53,http_upload_p50,float64,328257,84.785,17,0,False,True,False,At least 80% missing
54,http_upload_p90,float64,328257,84.785,51,0,False,True,False,At least 80% missing
55,http_upload_p95,float64,328257,84.785,95,0,False,True,False,At least 80% missing
56,http_upload_mad,float64,328257,84.785,5,0,False,True,False,At least 80% missing



D — CONSTANT FEATURE REVIEW


,column,dtype,unique_values,null_rows
3,is_weekend,int32,1,0
23,unique_usb_extensions,int64,1,0



E — ALL-COLUMN NULL PROFILE


,column,dtype,null_rows,null_pct,unique_values,infinite_values,constant_non_null
141,http_upload_to_p90_ratio,float64,328257,84.785,420,0,False
53,http_upload_p50,float64,328257,84.785,17,0,False
68,http_upload_to_p95_ratio,float64,328257,84.785,674,0,False
56,http_upload_mad,float64,328257,84.785,5,0,False
55,http_upload_p95,float64,328257,84.785,95,0,False
...,...,...,...,...,...,...,...
135,nr_stealth_acc_20wd,float64,0,0.000,12163,0,False
137,nr_gap_alt_2_10wd,float64,0,0.000,29,0,False
136,nr_gap_alt_2_5wd,float64,0,0.000,7,0,False
139,email_count_to_p90_ratio,float64,0,0.000,326,0,False



12.6.4 — INDEPENDENT FULL DATASET AUDIT SUMMARY
Rows                   : 387162
Users                  : 3737
Columns inspected      : 142
Independent checks     : 108
Confirmed rule failures: 0
Review-flagged columns : 14
Original modified      : NO
Files exported         : NO
STATUS: RULE CHECKS PASS — REVIEW PROFILE FLAGS



## 12.7 — Feature Integration & Binary Labeling

### Objective
Integrate the approved User-Day Ground Truth from Notebook 11 with the validated behavioral feature dataset and assign binary classification labels.

### Input Datasets
- **Behavioral Features:** `behavior_features_nr27_12` — 387,162 rows × 142 columns.
- **User-Day Ground Truth:** `ground_truth_user_days_r62.parquet` — 9,080 verified attack User-Days.

### Labeling Method
- `y = 1`: A User-Day containing verified synthetic attack injection.
- `y = 0`: A User-Day without synthetic attack injection.
- Labels are assigned through exact matching of `user` and `day`.
- Attacker identity alone does not determine the label; non-injected days remain `y = 0`.

### Validation
1. Verify Ground Truth uniqueness and positive injected-event counts.
2. Confirm all 9,080 attack User-Days exist in the feature dataset.
3. Preserve all 387,162 records and the original 142 columns.
4. Ensure that only the binary target `y` is added.
5. Verify label distribution and exact Ground Truth reconciliation.
6. Prevent attack metadata and scheduling information from entering the feature columns.

### Expected Output
`behavior_features_labeled_12` — 387,162 rows × 143 columns.

### Restrictions
- No historical baseline recalculation.
- No modification of original source datasets.
- No feature recalculation.
- No file exports.

### Next Step
**12.8 — Independent Label & Leakage Validation**


In [51]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.7
# FEATURE INTEGRATION & BINARY LABELING
#
# Authoritative GT: Notebook 11 User-Day Ground Truth
# Label granularity: User + Day
# y=1: Verified injected User-Day
# y=0: No synthetic injection on that User-Day
#
# READ-ONLY SOURCES — NO EXPORTS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal
from IPython.display import display


def require_127(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.7 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Resolve approved sources
# ------------------------------------------------------------

ROOT_127 = Path.cwd().resolve()

for candidate in [ROOT_127, *ROOT_127.parents]:
    if (
        candidate / "reports"
        / "11_Ground_Truth_and_Quality_Validation"
    ).is_dir():
        ROOT_127 = candidate
        break
else:
    raise FileNotFoundError(
        "New_Attack_Simulation project root not found."
    )

GT_PATH_127 = (
    ROOT_127
    / "reports"
    / "11_Ground_Truth_and_Quality_Validation"
    / "ground_truth_user_days_r62.parquet"
)

require_127(
    GT_PATH_127.is_file(),
    f"Approved Ground Truth not found: {GT_PATH_127}"
)

# Approved 142-column behavioral dataset.
features_127 = behavior_features_nr27_12.copy(deep=True)

# Read only keys and actual injection evidence.
gt_127 = pd.read_parquet(
    GT_PATH_127,
    columns=[
        "user",
        "day",
        "total_injected_events",
    ],
)

require_127(
    features_127.shape == (387_162, 142),
    "Behavioral feature dimensions are invalid."
)

require_127(
    gt_127.shape == (9_080, 3),
    "Unexpected User-Day Ground Truth dimensions."
)

require_127(
    "y" not in features_127.columns,
    "Target column already exists."
)


# ------------------------------------------------------------
# STEP 2 — Normalize and validate User-Day keys
# ------------------------------------------------------------

for frame in (features_127, gt_127):
    frame["user"] = frame["user"].astype("string")
    frame["day"] = pd.to_datetime(
        frame["day"]
    ).dt.normalize()

require_127(
    not features_127[["user", "day"]].isna().any().any()
    and not gt_127[["user", "day"]].isna().any().any(),
    "Null User-Day keys detected."
)

require_127(
    not features_127.duplicated(["user", "day"]).any(),
    "Duplicate feature User-Day keys."
)

require_127(
    not gt_127.duplicated(["user", "day"]).any(),
    "Duplicate Ground Truth User-Day keys."
)

require_127(
    features_127["user"].nunique() == 3_737,
    "Unexpected ML user population."
)

require_127(
    gt_127["user"].nunique() == 400,
    "Expected 400 unique attackers."
)


# ------------------------------------------------------------
# STEP 3 — Validate actual injection evidence
# ------------------------------------------------------------

injected_events_127 = pd.to_numeric(
    gt_127["total_injected_events"],
    errors="coerce",
)

require_127(
    injected_events_127.notna().all()
    and np.isfinite(injected_events_127).all()
    and injected_events_127.gt(0).all(),
    "Ground Truth contains days without positive injection."
)

require_127(
    (injected_events_127 % 1 == 0).all(),
    "Injected event counts must be integers."
)

require_127(
    int(injected_events_127.sum()) == 29_007,
    "Ground Truth event total does not match Notebook 11."
)


# ------------------------------------------------------------
# STEP 4 — Verify all GT keys exist in ML population
# ------------------------------------------------------------

key_check_127 = gt_127[
    ["user", "day"]
].merge(
    features_127[["user", "day"]],
    on=["user", "day"],
    how="left",
    indicator=True,
    validate="one_to_one",
)

require_127(
    key_check_127["_merge"].eq("both").all(),
    "Some attacked User-Days are absent from ML features."
)


# ------------------------------------------------------------
# STEP 5 — Generate binary labels
#
# Only keys are used for the labeling join.
# No scenario, schedule or injection-count metadata
# enters the feature dataset.
# ------------------------------------------------------------

attack_keys_127 = (
    gt_127[["user", "day"]]
    .copy()
    .assign(y=1)
)

labeled_127 = features_127.merge(
    attack_keys_127,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
    sort=False,
)

labeled_127["y"] = (
    labeled_127["y"]
    .fillna(0)
    .astype("int8")
)

behavior_features_labeled_12 = labeled_127


# ------------------------------------------------------------
# STEP 6 — Strict validation
# ------------------------------------------------------------

require_127(
    behavior_features_labeled_12.shape == (387_162, 143),
    "Expected 387,162 rows and 143 columns."
)

require_127(
    set(behavior_features_labeled_12["y"].unique())
    == {0, 1},
    "Target y must be strictly binary."
)

require_127(
    int(behavior_features_labeled_12["y"].sum()) == 9_080,
    "Positive labels do not match attack User-Days."
)

require_127(
    int(behavior_features_labeled_12["y"].eq(0).sum())
    == 378_082,
    "Unexpected negative-label count."
)

require_127(
    behavior_features_labeled_12.loc[
        behavior_features_labeled_12["y"].eq(1),
        "user"
    ].nunique() == 400,
    "Labeled attacker population is incorrect."
)

# Verify original 142 columns are unchanged, including
# column order, nulls, dtypes, and all feature values.
assert_frame_equal(
    behavior_features_labeled_12[
        features_127.columns
    ],
    features_127,
    check_dtype=True,
    check_exact=True,
    check_like=False,
)

require_127(
    not behavior_features_labeled_12.duplicated(
        ["user", "day"]
    ).any(),
    "Duplicate keys appeared after labeling."
)

require_127(
    list(behavior_features_labeled_12.columns)
    == [*features_127.columns, "y"],
    "Unexpected feature columns introduced during labeling."
)

# Independently recover all positive keys.
positive_keys_127 = pd.MultiIndex.from_frame(
    behavior_features_labeled_12.loc[
        behavior_features_labeled_12["y"].eq(1),
        ["user", "day"]
    ]
)

gt_keys_127 = pd.MultiIndex.from_frame(
    gt_127[["user", "day"]]
)

require_127(
    positive_keys_127.is_unique
    and positive_keys_127.symmetric_difference(
        gt_keys_127
    ).empty,
    "Positive labels differ from authoritative Ground Truth."
)


# ------------------------------------------------------------
# STEP 7 — Final labeling reports
# ------------------------------------------------------------

label_distribution_127 = (
    behavior_features_labeled_12["y"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("y")
    .reset_index(name="user_days")
)

label_distribution_127["percentage"] = (
    label_distribution_127["user_days"]
    / len(behavior_features_labeled_12)
    * 100
).round(3)

print("\nA — BINARY LABEL DISTRIBUTION")
display(label_distribution_127)

print("\nB — GROUND TRUTH RECONCILIATION")
display(pd.DataFrame([
    {
        "check": "Ground Truth attack User-Days",
        "result": len(gt_127),
        "status": "PASS",
    },
    {
        "check": "Ground Truth attackers",
        "result": gt_127["user"].nunique(),
        "status": "PASS",
    },
    {
        "check": "Ground Truth injected events",
        "result": int(injected_events_127.sum()),
        "status": "PASS",
    },
    {
        "check": "Matched attack keys",
        "result": len(positive_keys_127),
        "status": "PASS",
    },
    {
        "check": "Original feature modifications",
        "result": 0,
        "status": "PASS",
    },
    {
        "check": "Unexpected target columns",
        "result": 0,
        "status": "PASS",
    },
]))

print("\n" + "=" * 60)
print("12.7 — FINAL BINARY LABELING SUMMARY")
print("=" * 60)
print("Users                  :", features_127["user"].nunique())
print("Total rows             :", len(behavior_features_labeled_12))
print("Original columns       :", len(features_127.columns))
print("Final columns          :", behavior_features_labeled_12.shape[1])
print("Attack User-Days (y=1) :", int(behavior_features_labeled_12["y"].sum()))
print("Other User-Days (y=0)  :", int(behavior_features_labeled_12["y"].eq(0).sum()))
print("Ground Truth users     :", gt_127["user"].nunique())
print("Original 142 preserved : YES")
print("Historical baseline    : UNCHANGED")
print("Files exported         : NO")
print("=" * 60)

print("12.7 STATUS: PASS — BINARY LABELS ASSIGNED")
print("Next: 12.8 — Independent Label & Leakage Validation")



A — BINARY LABEL DISTRIBUTION


,y,user_days,percentage
0,0,378082,97.655
1,1,9080,2.345



B — GROUND TRUTH RECONCILIATION


,check,result,status
0,Ground Truth attack User-Days,9080,PASS
1,Ground Truth attackers,400,PASS
2,Ground Truth injected events,29007,PASS
3,Matched attack keys,9080,PASS
4,Original feature modifications,0,PASS
5,Unexpected target columns,0,PASS



12.7 — FINAL BINARY LABELING SUMMARY
Users                  : 3737
Total rows             : 387162
Original columns       : 142
Final columns          : 143
Attack User-Days (y=1) : 9080
Other User-Days (y=0)  : 378082
Ground Truth users     : 400
Original 142 preserved : YES
Historical baseline    : UNCHANGED
Files exported         : NO
12.7 STATUS: PASS — BINARY LABELS ASSIGNED
Next: 12.8 — Independent Label & Leakage Validation



## 12.8 — Independent Label & Leakage Validation

### Objective

Independently verify binary target integrity, source consistency, feature preservation, and potential information leakage before exporting the final ML dataset.

### Independent Evidence

- Final behavioral dataset: `behavior_features_labeled_12`.
- Authoritative User-Day Ground Truth from Notebook 11.
- Independent Injection Execution Log from Notebook 10.
- Original validated 142-column feature dataset.

### Validation

1. Verify target values, population, unique keys, and class distribution.
2. Reconstruct attacked User-Days directly from positive execution-log increments.
3. Reconcile reconstructed attack keys and event totals with Ground Truth.
4. Verify every positive and negative label against independently reconstructed injection evidence.
5. Confirm that all original 142 feature columns remain unchanged.
6. Detect direct Ground Truth, attack-scheduling, and injection-provenance columns.
7. Flag suspicious exact target duplicates among feature columns.
8. Independently check selected causal rolling-count calculations.

### Acceptance Criteria

- Exactly 387,162 rows and 143 columns.
- Exactly 9,080 positive User-Days and 378,082 negative User-Days.
- Exactly 400 users with positive labels.
- Exactly 10,124 executed channel actions and 29,007 added events.
- Zero label, execution, Ground Truth, or feature-preservation mismatches.
- No direct target or simulation-metadata leakage.

### Interpretation

A passing result establishes correctness against the tested sources and rules. It does not establish the absence of all statistical leakage, especially leakage related to user identity, data splitting, or model preprocessing.

### Restrictions

Read-only validation. No feature recalculation, baseline modification, label modification, or file exports.


In [52]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.8
# INDEPENDENT LABEL & LEAKAGE VALIDATION
#
# Independent source: Notebook 10 execution log
# Target source: Notebook 11 User-Day Ground Truth
# No modifications, no exports
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pandas.testing import assert_frame_equal
from IPython.display import display


def require_128(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.8 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Locate independent source files
# ------------------------------------------------------------

ROOT_128 = None

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (
        candidate / "reports" /
        "11_Ground_Truth_and_Quality_Validation"
    ).is_dir():
        ROOT_128 = candidate
        break

require_128(
    ROOT_128 is not None,
    "New_Attack_Simulation root not found."
)

GT_PATH_128 = (
    ROOT_128 / "reports" /
    "11_Ground_Truth_and_Quality_Validation" /
    "ground_truth_user_days_r62.parquet"
)

require_128(
    GT_PATH_128.is_file(),
    "Notebook 11 Ground Truth not found."
)

# ------------------------------------------------------------
# Select the authoritative Notebook 10 Execution Log
# Explicit filename — no ambiguous schema discovery
# ------------------------------------------------------------

LOG_PATH_128 = (
    ROOT_128
    / "reports"
    / "10_Attack_Injection_Pipeline"
    / "injection_execution_log_r62.parquet"
)

require_128(
    LOG_PATH_128.is_file(),
    f"Authoritative execution log not found: {LOG_PATH_128}"
)

required_log_fields_128 = {
    "user",
    "day",
    "channel",
    "count_added",
}

actual_log_fields_128 = set(
    pq.ParquetFile(LOG_PATH_128).schema_arrow.names
)

require_128(
    required_log_fields_128.issubset(actual_log_fields_128),
    "Execution log is missing required columns: "
    + str(sorted(required_log_fields_128 - actual_log_fields_128))
)

print("EXECUTION SOURCE:", LOG_PATH_128)
print("GROUND TRUTH SOURCE:", GT_PATH_128)


# ------------------------------------------------------------
# STEP 2 — Load independent evidence
# ------------------------------------------------------------

labeled_128 = behavior_features_labeled_12.copy(deep=True)
original_128 = behavior_features_nr27_12.copy(deep=True)

gt_128 = pd.read_parquet(
    GT_PATH_128,
    columns=["user", "day", "total_injected_events"]
)

if LOG_PATH_128.suffix.lower() == ".parquet":
    log_128 = pd.read_parquet(
        LOG_PATH_128,
        columns=["user", "day", "channel", "count_added"]
    )
else:
    log_128 = pd.read_csv(
        LOG_PATH_128,
        usecols=["user", "day", "channel", "count_added"]
    )

for frame in (labeled_128, original_128, gt_128, log_128):
    frame["user"] = frame["user"].astype("string")
    frame["day"] = pd.to_datetime(
        frame["day"], errors="raise"
    ).dt.normalize()

require_128(
    labeled_128.shape == (387162, 143),
    "Unexpected labeled dataset shape."
)

require_128(
    original_128.shape == (387162, 142),
    "Unexpected feature dataset shape."
)

require_128(
    not labeled_128.duplicated(["user", "day"]).any(),
    "Duplicate labeled User-Day keys."
)

require_128(
    not gt_128.duplicated(["user", "day"]).any(),
    "Duplicate Ground Truth User-Day keys."
)

require_128(
    len(log_128) == 10124,
    "Expected 10,124 executed channel actions."
)


# ------------------------------------------------------------
# STEP 3 — Independent execution evidence validation
# ------------------------------------------------------------

log_128["count_added"] = pd.to_numeric(
    log_128["count_added"],
    errors="raise"
)

require_128(
    log_128[["user", "day", "channel", "count_added"]]
    .notna().all().all(),
    "Null values in execution evidence."
)

require_128(
    np.isfinite(log_128["count_added"]).all()
    and log_128["count_added"].gt(0).all()
    and (log_128["count_added"] % 1 == 0).all(),
    "Invalid executed injection increments."
)

require_128(
    int(log_128["count_added"].sum()) == 29007,
    "Executed event total does not equal 29,007."
)

require_128(
    log_128["user"].nunique() == 400,
    "Expected 400 users in execution log."
)

# Reconstruct attack evidence without using GT labels.
executed_days_128 = (
    log_128
    .groupby(["user", "day"], as_index=False)
    .agg(
        execution_actions=("channel", "size"),
        executed_events=("count_added", "sum"),
    )
)

require_128(
    len(executed_days_128) == 9080,
    "Execution log does not reconstruct 9,080 attack days."
)


# ------------------------------------------------------------
# STEP 4 — Independently reconcile GT and execution
# ------------------------------------------------------------

gt_128["total_injected_events"] = pd.to_numeric(
    gt_128["total_injected_events"],
    errors="raise"
)

require_128(
    len(gt_128) == 9080
    and gt_128["user"].nunique() == 400,
    "Unexpected GT population."
)

gt_execution_128 = gt_128.merge(
    executed_days_128,
    on=["user", "day"],
    how="outer",
    indicator=True,
    validate="one_to_one",
)

gt_missing_128 = int(
    gt_execution_128["_merge"].ne("both").sum()
)

gt_event_mismatch_128 = int(
    (~np.isclose(
        gt_execution_128["total_injected_events"],
        gt_execution_128["executed_events"],
        rtol=0,
        atol=0,
        equal_nan=True
    )).sum()
)

require_128(
    gt_missing_128 == 0,
    "GT and execution action keys do not match."
)

require_128(
    gt_event_mismatch_128 == 0,
    "GT injected event counts differ from actual execution."
)


# ------------------------------------------------------------
# STEP 5 — Reconstruct y from execution ONLY
# ------------------------------------------------------------

expected_labels_128 = (
    executed_days_128[["user", "day"]]
    .assign(expected_y=1)
)

comparison_128 = labeled_128[
    ["user", "day", "y"]
].merge(
    expected_labels_128,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
)

comparison_128["expected_y"] = (
    comparison_128["expected_y"]
    .fillna(0)
    .astype("int8")
)

label_mismatches_128 = int(
    comparison_128["y"].ne(
        comparison_128["expected_y"]
    ).sum()
)

require_128(
    label_mismatches_128 == 0,
    "Labeled y differs from independently reconstructed execution."
)

require_128(
    labeled_128["y"].notna().all()
    and set(labeled_128["y"].unique()) == {0, 1},
    "Invalid binary target."
)

require_128(
    labeled_128["y"].sum() == 9080
    and labeled_128["y"].eq(0).sum() == 378082,
    "Unexpected class distribution."
)

require_128(
    labeled_128.loc[
        labeled_128["y"].eq(1), "user"
    ].nunique() == 400,
    "Unexpected positive-label user population."
)


# ------------------------------------------------------------
# STEP 6 — Verify preservation of original 142 columns
# ------------------------------------------------------------

require_128(
    list(labeled_128.columns) ==
    [*original_128.columns, "y"],
    "Label integration changed feature column structure."
)

original_sorted_128 = (
    original_128
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

labeled_sorted_128 = (
    labeled_128
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

assert_frame_equal(
    original_sorted_128,
    labeled_sorted_128[original_sorted_128.columns],
    check_exact=True,
    check_dtype=True,
)

# ------------------------------------------------------------
# STEP 7 — Direct leakage and suspicious target duplication
# ------------------------------------------------------------

# Identifiers and historical references require separate
# model-input selection decisions; they are not automatically
# injection metadata.
metadata_pattern_128 = re.compile(
    r"(ground_truth|scenario|injection|injected|"
    r"attack_label|attack_day|is_attack|"
    r"stealth_level|stealth_multiplier|"
    r"scheduled_channel|switch_bitmask|"
    r"formula_id|count_added|count_before|count_after)",
    flags=re.IGNORECASE,
)

predictor_cols_128 = [
    c for c in labeled_128.columns
    if c not in ("user", "day", "y")
]

direct_leaks_128 = [
    c for c in predictor_cols_128
    if metadata_pattern_128.search(c)
]

# Check for an exact numeric copy (or inversion) of y.
# This is a targeted warning, not a complete leakage proof.
target_duplicates_128 = []

y_array_128 = labeled_128["y"].to_numpy(dtype=float)

for col in predictor_cols_128:

    if not pd.api.types.is_numeric_dtype(labeled_128[col]):
        continue

    x = labeled_128[col].to_numpy(
        dtype=float,
        na_value=np.nan
    )

    if not np.isfinite(x).all():
        continue

    if np.array_equal(x, y_array_128):
        target_duplicates_128.append((col, "exact y"))

    elif np.array_equal(x, 1.0 - y_array_128):
        target_duplicates_128.append((col, "inverse y"))

require_128(
    not direct_leaks_128,
    f"Direct simulation metadata detected: {direct_leaks_128}"
)

require_128(
    not target_duplicates_128,
    f"Target-duplicate features found: {target_duplicates_128}"
)


# ------------------------------------------------------------
# STEP 8 — Independent causal rolling-count audit
# ------------------------------------------------------------

rolling_rows_128 = []

source_count_map_128 = {
    "external_email_count": "external_email_count",
    "usb_copy_count": "usb_copy_count",
    "http_upload_count": "www_upload_count",
}

for feature_prefix, source_col in source_count_map_128.items():

    for w in (5, 10, 20, 30):

        name = f"{feature_prefix}_{w}wd"

        # Independently recompute using only present and
        # previous eligible workdays from 2011.
        expected = (
            labeled_sorted_128.groupby("user")[source_col]
            .transform(
                lambda s: s.rolling(
                    window=w,
                    min_periods=1
                ).sum()
            )
        )

        actual = labeled_sorted_128[name]

        differences = ~np.isclose(
            actual.to_numpy(dtype=float),
            expected.to_numpy(dtype=float),
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True
        )

        rolling_rows_128.append({
            "feature": name,
            "checked": len(actual),
            "mismatches": int(differences.sum()),
            "status": "PASS" if not differences.any() else "FAIL",
        })

rolling_qa_128 = pd.DataFrame(rolling_rows_128)

require_128(
    rolling_qa_128["mismatches"].sum() == 0,
    "A causal rolling-count feature has inconsistent values."
)


# ------------------------------------------------------------
# STEP 9 — Produce final QA summaries
# ------------------------------------------------------------

qa_128 = pd.DataFrame([
    ["Execution actions", len(log_128), "PASS"],
    ["Executed injected events", int(log_128["count_added"].sum()), "PASS"],
    ["Reconstructed attack days", len(executed_days_128), "PASS"],
    ["GT key mismatches", gt_missing_128, "PASS"],
    ["GT event mismatches", gt_event_mismatch_128, "PASS"],
    ["Binary label mismatches", label_mismatches_128, "PASS"],
    ["Original feature modifications", 0, "PASS"],
    ["Direct metadata columns", len(direct_leaks_128), "PASS"],
    ["Exact target duplicates", len(target_duplicates_128), "PASS"],
    ["Causal rolling mismatches", int(rolling_qa_128["mismatches"].sum()), "PASS"],
], columns=["check", "result", "status"])

label_counts_128 = (
    labeled_128["y"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("y")
    .reset_index(name="user_days")
)

print("\nA — INDEPENDENT LABEL DISTRIBUTION")
display(label_counts_128)

print("\nB — EXECUTION & GROUND TRUTH RECONCILIATION")
display(qa_128)

print("\nC — CAUSAL ROLLING CHECK")
display(rolling_qa_128)

print("\nD — LEAKAGE REVIEW")
display(pd.DataFrame([
    {
        "item": "Direct GT/scheduling metadata",
        "result": "NONE DETECTED" if not direct_leaks_128 else str(direct_leaks_128),
    },
    {
        "item": "Exact target duplication",
        "result": "NONE DETECTED" if not target_duplicates_128 else str(target_duplicates_128),
    },
    {
        "item": "Identifiers requiring model-input review",
        "result": "user, day",
    },
    {
        "item": "Baseline reference columns",
        "result": "Require explicit model feature selection",
    },
]))

print("\n" + "=" * 60)
print("12.8 — INDEPENDENT LABEL & LEAKAGE QA")
print("=" * 60)
print("Rows                      :", len(labeled_128))
print("Columns                   :", labeled_128.shape[1])
print("Users                     :", labeled_128["user"].nunique())
print("Positive User-Days        :", int(labeled_128["y"].sum()))
print("Negative User-Days        :", int(labeled_128["y"].eq(0).sum()))
print("GT vs execution           : EXACT MATCH")
print("Independent label errors  :", label_mismatches_128)
print("Original features modified: NO")
print("Direct metadata leakage   : NOT DETECTED")
print("Full statistical leakage  : NOT PROVEN")
print("Files exported            : NO")
print("=" * 60)
print("12.8 STATUS: PASS — TESTED RULES VERIFIED")
print("Next: 12.9 — Final Export & Integrity")


EXECUTION SOURCE: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\10_Attack_Injection_Pipeline\injection_execution_log_r62.parquet
GROUND TRUTH SOURCE: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\11_Ground_Truth_and_Quality_Validation\ground_truth_user_days_r62.parquet

A — INDEPENDENT LABEL DISTRIBUTION


,y,user_days
0,0,378082
1,1,9080



B — EXECUTION & GROUND TRUTH RECONCILIATION


,check,result,status
0,Execution actions,10124,PASS
1,Executed injected events,29007,PASS
2,Reconstructed attack days,9080,PASS
3,GT key mismatches,0,PASS
4,GT event mismatches,0,PASS
5,Binary label mismatches,0,PASS
6,Original feature modifications,0,PASS
7,Direct metadata columns,0,PASS
8,Exact target duplicates,0,PASS
9,Causal rolling mismatches,0,PASS



C — CAUSAL ROLLING CHECK


,feature,checked,mismatches,status
0,external_email_count_5wd,387162,0,PASS
1,external_email_count_10wd,387162,0,PASS
2,external_email_count_20wd,387162,0,PASS
3,external_email_count_30wd,387162,0,PASS
4,usb_copy_count_5wd,387162,0,PASS
5,usb_copy_count_10wd,387162,0,PASS
6,usb_copy_count_20wd,387162,0,PASS
7,usb_copy_count_30wd,387162,0,PASS
8,http_upload_count_5wd,387162,0,PASS
9,http_upload_count_10wd,387162,0,PASS



D — LEAKAGE REVIEW


,item,result
0,Direct GT/scheduling metadata,NONE DETECTED
1,Exact target duplication,NONE DETECTED
2,Identifiers requiring model-input review,"user, day"
3,Baseline reference columns,Require explicit model feature selection



12.8 — INDEPENDENT LABEL & LEAKAGE QA
Rows                      : 387162
Columns                   : 143
Users                     : 3737
Positive User-Days        : 9080
Negative User-Days        : 378082
GT vs execution           : EXACT MATCH
Independent label errors  : 0
Original features modified: NO
Direct metadata leakage   : NOT DETECTED
Full statistical leakage  : NOT PROVEN
Files exported            : NO
12.8 STATUS: PASS — TESTED RULES VERIFIED
Next: 12.9 — Final Export & Integrity



## 12.9.1 — Final Dataset & Export Path Preflight

### Objective

Verify the final labeled behavioral dataset, output structure, destination paths, and export readiness before generating the final deliverables.

### Input

`behavior_features_labeled_12`

Expected dimensions: 387,162 User-Day records × 143 columns.

### Validation

1. Verify dataset dimensions, user population, and unique User-Day keys.
2. Confirm the original 85 columns, 30 additional behavioral features, and 27 New Rotation features.
3. Verify the binary target `y` and expected label distribution.
4. Check schema integrity and absence of infinite numerical values.
5. Identify the intended export directory and inspect existing files.
6. Block accidental replacement of existing deliverables.

### Planned Deliverables

- Final labeled behavioral dataset (Parquet).
- Complete feature data dictionary (CSV).
- Feature methodology and design decisions (Markdown).
- Final feature QA report (Markdown).
- Export manifest and SHA-256 checksums.

### Documentation Requirements

- Partial 2011-only workday windows and immutable historical baselines.
- Feature-specific definitions of Dominant Activity Channel and Dominant Deviation Channel, including tie handling.
- Missing historical P90 statistics and corresponding undefined ratios.
- Binary target semantics and evaluation limitations.

### Restrictions

No source modification, no directory creation, no file export, and no overwrite in this section.

### Expected Status

`PASS — READY FOR DOCUMENTATION AND EXPORT PREPARATION`


In [53]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.9.1
# FINAL DATASET & EXPORT PATH PREFLIGHT
#
# READ-ONLY — NO EXPORTS — NO OVERWRITES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display


def require_1291(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.9.1 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Locate project root
# ------------------------------------------------------------

ROOT_1291 = None

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (
        candidate / "reports"
        / "11_Ground_Truth_and_Quality_Validation"
    ).is_dir():
        ROOT_1291 = candidate
        break

require_1291(
    ROOT_1291 is not None,
    "New_Attack_Simulation root not found."
)

REPORT_1291 = (
    ROOT_1291
    / "reports"
    / "12_Feature_Engineering_and_Validation"
)

# Intended output names. No file creation.
OUTPUT_NAMES_1291 = [
    "behavior_features_r62_ml_final_new.parquet",
    "feature_dictionary_r62_final.csv",
    "feature_methodology_and_decisions.md",
    "final_feature_qa_report.md",
    "export_manifest.json",
]

output_paths_1291 = {
    name: REPORT_1291 / name
    for name in OUTPUT_NAMES_1291
}


# ------------------------------------------------------------
# STEP 2 — Validate final dataset
# ------------------------------------------------------------

require_1291(
    "behavior_features_labeled_12" in globals(),
    "Final labeled dataset is missing from memory."
)

final_1291 = behavior_features_labeled_12

require_1291(
    final_1291.shape == (387_162, 143),
    f"Unexpected dataset dimensions: {final_1291.shape}"
)

require_1291(
    final_1291["user"].nunique() == 3_737,
    "Unexpected user population."
)

require_1291(
    final_1291.columns.is_unique,
    "Duplicate column names detected."
)

require_1291(
    not final_1291.duplicated(["user", "day"]).any(),
    "Duplicate User-Day keys detected."
)

require_1291(
    not final_1291[["user", "day"]].isna().any().any(),
    "Missing User-Day keys."
)


# ------------------------------------------------------------
# STEP 3 — Verify feature groups
# ------------------------------------------------------------

original_1291 = list(original_columns_122)
clear_1291 = list(official_names_1251)
nr_1291 = list(NR_NAMES_1261)

require_1291(
    len(original_1291) == 85
    and len(set(original_1291)) == 85,
    "Original feature group is invalid."
)

require_1291(
    len(clear_1291) == 30
    and len(set(clear_1291)) == 30,
    "Additional 30-feature group is invalid."
)

require_1291(
    len(nr_1291) == 27
    and len(set(nr_1291)) == 27,
    "New Rotation feature group is invalid."
)

all_features_1291 = (
    original_1291
    + clear_1291
    + nr_1291
)

require_1291(
    len(all_features_1291) == 142
    and len(set(all_features_1291)) == 142,
    "Feature groups contain duplicate names."
)

require_1291(
    set(all_features_1291)
    == set(final_1291.columns) - {"y"},
    "Final feature schema differs from approved groups."
)

require_1291(
    list(final_1291.columns)
    == [*all_features_1291, "y"],
    "Unexpected final feature column order."
)


# ------------------------------------------------------------
# STEP 4 — Verify target distribution
# ------------------------------------------------------------

require_1291(
    final_1291["y"].notna().all()
    and set(final_1291["y"].unique()) == {0, 1},
    "Target y is not strictly binary."
)

positive_1291 = int(final_1291["y"].eq(1).sum())
negative_1291 = int(final_1291["y"].eq(0).sum())

require_1291(
    positive_1291 == 9_080,
    "Unexpected positive-label count."
)

require_1291(
    negative_1291 == 378_082,
    "Unexpected negative-label count."
)

require_1291(
    final_1291.loc[
        final_1291["y"].eq(1), "user"
    ].nunique() == 400,
    "Unexpected attacker population."
)


# ------------------------------------------------------------
# STEP 5 — Schema and numerical integrity
# ------------------------------------------------------------

numeric_1291 = final_1291.select_dtypes(
    include="number"
)

infinite_1291 = int(
    np.isinf(
        numeric_1291.to_numpy(dtype="float64")
    ).sum()
)

require_1291(
    infinite_1291 == 0,
    "Infinite values detected."
)

require_1291(
    pd.api.types.is_datetime64_any_dtype(
        final_1291["first_logon"]
    ),
    "first_logon is not datetime."
)

require_1291(
    pd.api.types.is_datetime64_any_dtype(
        final_1291["last_logoff"]
    ),
    "last_logoff is not datetime."
)

# Preserve missing values for downstream preprocessing.
null_columns_1291 = int(
    final_1291.isna().any().sum()
)


# ------------------------------------------------------------
# STEP 6 — Check destination without writing files
# ------------------------------------------------------------

path_rows_1291 = []

for name, path in output_paths_1291.items():
    path_rows_1291.append({
        "filename": name,
        "exists": path.exists(),
        "status": (
            "BLOCKED — FILE EXISTS"
            if path.exists()
            else "AVAILABLE"
        ),
    })

path_audit_1291 = pd.DataFrame(path_rows_1291)

existing_files_1291 = path_audit_1291[
    path_audit_1291["exists"]
]["filename"].tolist()

require_1291(
    not existing_files_1291,
    "Existing destination files would be overwritten: "
    + ", ".join(existing_files_1291)
)


# ------------------------------------------------------------
# STEP 7 — Display reports
# ------------------------------------------------------------

group_audit_1291 = pd.DataFrame([
    {
        "group": "Original Data Engineering",
        "columns": len(original_1291),
        "status": "PASS",
    },
    {
        "group": "Additional Behavioral Features",
        "columns": len(clear_1291),
        "status": "PASS",
    },
    {
        "group": "New Rotation Features",
        "columns": len(nr_1291),
        "status": "PASS",
    },
    {
        "group": "Binary Target",
        "columns": 1,
        "status": "PASS",
    },
])

label_audit_1291 = pd.DataFrame([
    {
        "label": 0,
        "user_days": negative_1291,
        "percentage": round(
            negative_1291 / len(final_1291) * 100, 3
        ),
    },
    {
        "label": 1,
        "user_days": positive_1291,
        "percentage": round(
            positive_1291 / len(final_1291) * 100, 3
        ),
    },
])

print("\nA — FINAL DATASET STRUCTURE")
display(group_audit_1291)

print("\nB — BINARY LABEL DISTRIBUTION")
display(label_audit_1291)

print("\nC — EXPORT DESTINATION INVENTORY")
display(path_audit_1291)

print("\n" + "=" * 60)
print("12.9.1 — FINAL EXPORT PREFLIGHT")
print("=" * 60)
print("Project root         :", ROOT_1291)
print("Export directory     :", REPORT_1291)
print("Directory exists     :", REPORT_1291.is_dir())
print("Final rows           :", len(final_1291))
print("Final columns        :", final_1291.shape[1])
print("Users                :", final_1291["user"].nunique())
print("Attack User-Days     :", positive_1291)
print("Other User-Days      :", negative_1291)
print("Columns with NaN     :", null_columns_1291)
print("Infinite values      :", infinite_1291)
print("Existing destinations:", len(existing_files_1291))
print("Original data changed: NO")
print("Files exported       : NO")
print("=" * 60)
print("12.9.1 STATUS: PASS — READY FOR EXPORT PREPARATION")
print("Next: 12.9.2 — Final Feature Data Dictionary")


AssertionError: 12.9.1 FAILED: Existing destination files would be overwritten: behavior_features_r62_ml_final_new.parquet, feature_dictionary_r62_final.csv, feature_methodology_and_decisions.md, final_feature_qa_report.md, export_manifest.json


## 12.9.2 — Final Feature Data Dictionary

### Objective

Construct a complete, source-grounded data dictionary for the 143-column labeled behavioral dataset.

### Authoritative References

- Original Data Engineering feature dictionary for the 85 source columns.
- Approved 30-feature dictionary for additional behavioral features.
- Feature Generation Contract for the 27 rotation and deviation features.
- Binary labeling definition from Notebook 11.

### Method

1. Preserve original feature descriptions and calculations from their authoritative dictionaries.
2. Document the effective 2011-only partial-window policy.
3. Distinguish activity-based and deviation-based dominant-channel definitions at the feature level.
4. Document historical baseline immutability, missing-value semantics, and target-label meaning.
5. Verify exact coverage of all 143 dataset columns.
6. Flag missing or conflicting documentation instead of inventing feature definitions.

### Expected Output

`final_feature_dictionary_12` — 143 documented columns, ordered exactly as the final dataset.

### Restrictions

No dataset modification, baseline recalculation, directory creation, or file export.

Final dictionary approval requires complete coverage and no unresolved documentation gaps.


In [ ]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.9.2
# FINAL FEATURE DATA DICTIONARY — CORRECTED
#
# 85 Original + 30 Behavioral + 27 New Rotation + y
#
# Corrected:
# - Search dictionaries across the main FYP project
# - Preserve authoritative source definitions
# - Document 2011-only partial windows
# - Document feature-specific dominant-channel definitions
#
# IN-MEMORY ONLY — NO EXPORTS
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display


def require_1292(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.9.2 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Verify approved final dataset
# ------------------------------------------------------------

require_1292(
    "final_1291" in globals()
    and final_1291.shape == (387162, 143),
    "Run the approved 12.9.1 preflight first."
)

dataset_1292 = final_1291

require_1292(
    dataset_1292.columns.is_unique,
    "Final dataset contains duplicate column names."
)

require_1292(
    "y" in dataset_1292.columns,
    "Final dataset is missing the binary target."
)


# ------------------------------------------------------------
# STEP 2 — Locate authoritative dictionaries
#
# CORRECTED:
# New_Attack_Simulation is a subfolder of the main
# FYP project. Search its sibling folders as well.
# ------------------------------------------------------------

PROJECT_ROOT_1292 = ROOT_1291.parent

require_1292(
    PROJECT_ROOT_1292.is_dir(),
    "Main FYP project directory not found."
)


def locate_dictionary_1292(filename, preferred_path):

    matches = sorted(
        p for p in PROJECT_ROOT_1292.rglob(filename)
        if p.is_file()
    )

    preferred = [
        p for p in matches
        if preferred_path.lower() in str(p).lower()
    ]

    candidates = preferred if preferred else matches

    print(f"\nSearching for: {filename}")

    if candidates:
        for p in candidates:
            print("FOUND:", p)
    else:
        print("NOT FOUND")

    require_1292(
        len(candidates) == 1,
        f"Expected exactly one authoritative {filename}; "
        f"found {len(candidates)}: "
        f"{[str(p) for p in candidates]}"
    )

    return candidates[0]


ORIGINAL_DICT_PATH_1292 = locate_dictionary_1292(
    "feature_dictionary_r62.csv",
    "Data_Engineering"
)

BEHAVIORAL_DICT_PATH_1292 = locate_dictionary_1292(
    "new_features_dictionary_30_r62.csv",
    "Clear_Attack_Simulation"
)

original_dict_1292 = pd.read_csv(
    ORIGINAL_DICT_PATH_1292,
    keep_default_na=False
)

behavioral_dict_1292 = pd.read_csv(
    BEHAVIORAL_DICT_PATH_1292,
    keep_default_na=False
)


# ------------------------------------------------------------
# STEP 3 — Normalize original dictionary schemas
# ------------------------------------------------------------

def normalize_dictionary_1292(frame, source):

    frame = frame.copy()
    frame.columns = frame.columns.str.strip()

    aliases = {
        "feature_name": [
            "feature_name",
            "column_name",
            "column",
            "feature",
            "name",
        ],
        "description": [
            "description",
            "definition",
        ],
        "calculation": [
            "calculation",
            "formula",
            "calculation_rule",
        ],
        "source_columns": [
            "source_columns",
            "source_fields",
            "inputs",
        ],
        "time_scope": [
            "time_scope",
            "window",
            "temporal_scope",
        ],
        "notes": [
            "notes",
            "remarks",
        ],
        "category": [
            "category",
            "feature_category",
        ],
    }

    lower_lookup = {
        str(c).lower(): c
        for c in frame.columns
    }

    normalized = pd.DataFrame(index=frame.index)

    for target, options in aliases.items():

        found = next(
            (
                lower_lookup[option.lower()]
                for option in options
                if option.lower() in lower_lookup
            ),
            None
        )

        if found is not None:
            normalized[target] = (
                frame[found]
                .astype("string")
                .fillna("")
            )
        else:
            normalized[target] = ""

    normalized["feature_name"] = (
        normalized["feature_name"]
        .str.strip()
    )

    require_1292(
        normalized["feature_name"].ne("").all(),
        f"{source}: missing feature names."
    )

    require_1292(
        normalized["feature_name"].is_unique,
        f"{source}: duplicate feature names."
    )

    normalized["documentation_source"] = source

    return normalized


original_normalized_1292 = normalize_dictionary_1292(
    original_dict_1292,
    "Data Engineering feature dictionary"
)

behavioral_normalized_1292 = normalize_dictionary_1292(
    behavioral_dict_1292,
    "Approved behavioral feature dictionary"
)

require_1292(
    len(original_normalized_1292) == 85
    and set(original_normalized_1292["feature_name"])
    == set(original_1291),
    "Original dictionary does not match all 85 columns."
)

require_1292(
    len(behavioral_normalized_1292) == 30
    and set(behavioral_normalized_1292["feature_name"])
    == set(clear_1291),
    "Behavioral dictionary does not match all 30 columns."
)


# ------------------------------------------------------------
# STEP 4 — Document the 27 New Rotation features
#
# Definitions follow the Feature Generation Contract.
# ------------------------------------------------------------

NR_DEFINITIONS_1292 = {

    "nr_m": (
        "Number of channels with positive cumulative "
        "deviation within the window.",
        "S_c = SUM(E_c) within W; "
        "count channels where S_c > 0.",
        "Daily counts; historical P50 and P95",
    ),

    "nr_entropy": (
        "Normalized entropy of cumulative channel deviations.",
        "If S>0, -SUM(p_c*ln(p_c))/ln(3), "
        "p_c=S_c/S; otherwise 0.",
        "Daily channel deviations E",
    ),

    "nr_excl_days": (
        "Number of window days with exactly one "
        "deviating channel.",
        "SUM(1 when exactly one E_c(d)>0).",
        "Daily channel deviations E",
    ),

    "nr_overlap_days": (
        "Number of window days with at least two "
        "deviating channels.",
        "SUM(1 when at least two E_c(d)>0).",
        "Daily channel deviations E",
    ),

    "nr_alt": (
        "Dominant deviation-channel alternation among "
        "adjacent eligible workdays with positive deviation.",
        "Changed dominant-deviation channel pairs divided by "
        "valid adjacent deviation-day pairs; "
        "0 if no pairs.",
        "Daily deviations E; dominant deviation channel",
    ),

    "nr_handoff": (
        "Number of newly deviating channels relative "
        "to the preceding eligible workday.",
        "For each day in W, count channels with E_c(d)>0 "
        "and E_c(d-1)=0, only when any channel "
        "deviated on d-1; sum across W.",
        "Daily channel deviations E; "
        "previous eligible workday",
    ),

    "nr_stealth_acc": (
        "Accumulated deviations that remain at or below "
        "their respective historical P95 thresholds.",
        "SUM(E_c(d) where E_c(d)>0 "
        "and Count_c(d)<=P95_c).",
        "Daily counts; historical P50 and P95",
    ),

    "nr_gap_alt_2": (
        "Dominant deviation-channel alternation allowing "
        "up to two intervening quiet eligible workdays.",
        "Among consecutive deviation days in W, "
        "retain pairs with at most 2 quiet workdays; "
        "divide dominant-channel changes by eligible pairs; "
        "0 if no pairs.",
        "Daily deviations E; dominant deviation channel",
    ),
}

NR_DAILY_FORMULAS_1292 = {

    "email_count_to_p90_ratio": (
        "Daily Email event count divided by historical Email P90.",
        "external_email_count / email_count_p90 "
        "if P90>0; else NaN.",
        "external_email_count; email_count_p90",
    ),

    "usb_copy_to_p90_ratio": (
        "Daily USB copy count divided by historical USB P90.",
        "usb_copy_count / usb_copy_p90 "
        "if P90>0; else NaN.",
        "usb_copy_count; usb_copy_p90",
    ),

    "http_upload_to_p90_ratio": (
        "Daily HTTP upload count divided by historical HTTP P90.",
        "www_upload_count / http_upload_p90 "
        "if P90>0; else NaN.",
        "www_upload_count; http_upload_p90",
    ),
}

nr_rows_1292 = []

for name in nr_1291:

    if name in NR_DAILY_FORMULAS_1292:

        description, calculation, inputs = (
            NR_DAILY_FORMULAS_1292[name]
        )

        category = "Daily P90 Ratio"
        time_scope = "Current eligible workday"

    else:

        match = re.fullmatch(
            r"(nr_.+)_(5|10|20)wd",
            name
        )

        require_1292(
            match is not None,
            f"Unrecognized New Rotation feature: {name}"
        )

        family, window = match.groups()

        require_1292(
            family in NR_DEFINITIONS_1292,
            f"Missing documented definition: {family}"
        )

        description, calculation, inputs = (
            NR_DEFINITIONS_1292[family]
        )

        category = "New Rotation"

        time_scope = (
            f"Last {window} eligible workdays; "
            "partial windows from 2011"
        )

    notes = (
        "Deviation E_c(d)=max(0,Count_c(d)-P50_c)"
        "/max(1,P95_c-P50_c). "
        "Missing baseline produces E=0. "
        "Baseline fixed from 2010."
    )

    if name.startswith(("nr_alt_", "nr_gap_alt_2_")):
        notes += (
            " Dominant Deviation Channel = argmax(E_c). "
            "Exact ties resolve EMAIL, USB, HTTP."
        )

    if name.startswith("nr_handoff_"):
        notes += (
            " The preceding eligible workday may be "
            "outside the current window, but must "
            "belong to the available 2011 history."
        )

    if name in NR_DAILY_FORMULAS_1292:
        notes = (
            "NaN when historical P90 is missing or "
            "nonpositive; missing values are preserved."
        )

    nr_rows_1292.append({
        "feature_name": name,
        "description": description,
        "calculation": calculation,
        "source_columns": inputs,
        "time_scope": time_scope,
        "notes": notes,
        "category": category,
        "documentation_source":
            "Feature Generation Contract",
    })

nr_normalized_1292 = pd.DataFrame(nr_rows_1292)

require_1292(
    len(nr_normalized_1292) == 27
    and set(nr_normalized_1292["feature_name"])
    == set(nr_1291),
    "New Rotation dictionary coverage mismatch."
)


# ------------------------------------------------------------
# STEP 5 — Document binary target y
# ------------------------------------------------------------

target_1292 = pd.DataFrame([{
    "feature_name": "y",

    "description": (
        "Binary indicator of verified synthetic "
        "attack injection on the User-Day."
    ),

    "calculation": (
        "1 if (user, day) matches an approved User-Day "
        "with positive injected event count; otherwise 0."
    ),

    "source_columns": (
        "user; day; authoritative User-Day Ground Truth"
    ),

    "time_scope": "Current User-Day",

    "notes": (
        "Evaluation target only. Never use as a predictor. "
        "y=0 means no synthetic injection, not a guarantee "
        "that all underlying activity is benign."
    ),

    "category": "Binary Target",

    "documentation_source":
        "Approved User-Day Ground Truth",
}])


# ------------------------------------------------------------
# STEP 6 — Combine and align with final dataset
# ------------------------------------------------------------

combined_1292 = pd.concat(
    [
        original_normalized_1292,
        behavioral_normalized_1292,
        nr_normalized_1292,
        target_1292,
    ],
    ignore_index=True,
)

require_1292(
    len(combined_1292) == 143
    and combined_1292["feature_name"].is_unique,
    "Dictionary contains missing or duplicated feature names."
)

require_1292(
    set(combined_1292["feature_name"])
    == set(dataset_1292.columns),
    "Dictionary feature names do not match the final dataset."
)

dictionary_1292 = (
    combined_1292
    .set_index("feature_name")
    .loc[list(dataset_1292.columns)]
    .reset_index()
)

dictionary_1292.insert(
    0,
    "column_order",
    np.arange(1, 144),
)

dictionary_1292["data_type"] = [
    str(dataset_1292[c].dtype)
    for c in dictionary_1292["feature_name"]
]

dictionary_1292["null_rows"] = [
    int(dataset_1292[c].isna().sum())
    for c in dictionary_1292["feature_name"]
]

dictionary_1292["nullable"] = np.where(
    dictionary_1292["null_rows"] > 0,
    "YES",
    "NO",
)


# ------------------------------------------------------------
# STEP 7 — Assign column roles
# ------------------------------------------------------------

dictionary_1292["column_role"] = "Behavioral Feature"

dictionary_1292.loc[
    dictionary_1292["feature_name"].isin(
        ["user", "day"]
    ),
    "column_role"
] = "Identifier"

dictionary_1292.loc[
    dictionary_1292["feature_name"].isin(
        baseline_cols_1264
    ),
    "column_role"
] = "Historical Baseline Reference"

dictionary_1292.loc[
    dictionary_1292["feature_name"].eq("y"),
    "column_role"
] = "Target — Not Predictor"


# ------------------------------------------------------------
# STEP 8 — Document effective partial-window policy
#
# Preserve original source definitions separately.
# Clearly identify the approved methodological override.
# ------------------------------------------------------------

rolling_1292 = (
    dictionary_1292["feature_name"]
    .str.contains(
        r"_(?:5|10|20|30)wd$",
        regex=True
    )
)

dictionary_1292["effective_window_policy"] = ""

dictionary_1292.loc[
    rolling_1292,
    "effective_window_policy"
] = (
    "Partial eligible-workday windows from 2011 only; "
    "min_periods=1; current day included."
)

dictionary_1292["methodology_notes"] = ""

dictionary_1292.loc[
    rolling_1292,
    "methodology_notes"
] = (
    "Current dataset uses partial 2011-only windows "
    "instead of the original full-window startup convention. "
    "Historical 2010 baseline remains unchanged."
)


# ------------------------------------------------------------
# STEP 9 — Feature-specific dominant-channel definitions
#
# No personal names or team ownership references.
# ------------------------------------------------------------

dictionary_1292["dominant_channel_definition"] = ""

activity_switches_1292 = (
    dictionary_1292["feature_name"]
    .str.startswith("dominant_channel_switches_")
)

dictionary_1292.loc[
    activity_switches_1292,
    "dominant_channel_definition"
] = (
    "Dominant Activity Channel: "
    "argmax(Count_c/P95_c) among valid P95 channels. "
    "Inactive days and approximately tied maxima "
    "(np.isclose, rtol=1e-9, atol=1e-12) are excluded."
)

deviation_switches_1292 = (
    dictionary_1292["feature_name"]
    .str.startswith(("nr_alt_", "nr_gap_alt_2_"))
)

dictionary_1292.loc[
    deviation_switches_1292,
    "dominant_channel_definition"
] = (
    "Dominant Deviation Channel: argmax(E_c), "
    "where E_c=max(0,Count_c-P50_c)/max(1,P95_c-P50_c). "
    "Ties resolve EMAIL, USB, HTTP. "
    "Days without deviation have no dominant channel."
)


# ------------------------------------------------------------
# STEP 10 — Documentation completeness validation
# ------------------------------------------------------------

required_text_1292 = [
    "description",
    "calculation",
    "source_columns",
    "documentation_source",
]

missing_fields_1292 = []

for field in required_text_1292:

    values = dictionary_1292[field].astype("string")

    missing = (
        values.isna()
        | values.str.strip().eq("").fillna(True)
    )

    for name in dictionary_1292.loc[
        missing,
        "feature_name"
    ]:
        missing_fields_1292.append({
            "feature": name,
            "missing_field": field,
        })

gaps_1292 = pd.DataFrame(
    missing_fields_1292,
    columns=["feature", "missing_field"],
)


# ------------------------------------------------------------
# STEP 11 — Source and feature coverage reports
# ------------------------------------------------------------

coverage_1292 = pd.DataFrame([
    {
        "group": "Original Data Engineering",
        "columns": 85,
        "source": str(ORIGINAL_DICT_PATH_1292),
    },
    {
        "group": "Additional Behavioral Features",
        "columns": 30,
        "source": str(BEHAVIORAL_DICT_PATH_1292),
    },
    {
        "group": "New Rotation Features",
        "columns": 27,
        "source": "Feature Generation Contract",
    },
    {
        "group": "Binary Target",
        "columns": 1,
        "source": "User-Day Ground Truth",
    },
])

print("\nA — DOCUMENTATION SOURCE COVERAGE")
display(coverage_1292)

print("\nB — DOCUMENTATION GAPS")

if gaps_1292.empty:
    print("No missing required documentation fields.")
else:
    display(gaps_1292)

print("\nC — DOMINANT CHANNEL DEFINITIONS")
display(
    dictionary_1292.loc[
        activity_switches_1292 | deviation_switches_1292,
        [
            "feature_name",
            "dominant_channel_definition",
        ]
    ]
)

print("\nD — FINAL DATA DICTIONARY PREVIEW")
display(dictionary_1292.head(12))


# ------------------------------------------------------------
# STEP 12 — Final validation
# ------------------------------------------------------------

exact_coverage_1292 = (
    list(dictionary_1292["feature_name"])
    == list(dataset_1292.columns)
)

print("\n" + "=" * 60)
print("12.9.2 — FINAL DICTIONARY SUMMARY")
print("=" * 60)

print("Project root           :", PROJECT_ROOT_1292)
print("Documented columns     :", len(dictionary_1292))
print("Original definitions   :", 85)
print("Additional features    :", 30)
print("Rotation features      :", 27)
print("Target labels          :", 1)
print("Missing documentation  :", len(gaps_1292))
print("Exact dataset coverage :", exact_coverage_1292)
print("Historical baseline    : UNCHANGED")
print("Files exported         : NO")
print("=" * 60)

require_1292(
    gaps_1292.empty,
    "Required documentation is incomplete."
)

require_1292(
    exact_coverage_1292,
    "Dictionary does not match the final dataset schema."
)

final_feature_dictionary_12 = dictionary_1292.copy()

print("12.9.2 STATUS: PASS — DICTIONARY READY")
print("Next: 12.9.3 — Methodology & Final QA Documentation")



Searching for: feature_dictionary_r62.csv
FOUND: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\results\feature_dictionary_r62.csv

Searching for: new_features_dictionary_30_r62.csv
FOUND: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\documentation\dictionary\new_features_dictionary_30_r62.csv

A — DOCUMENTATION SOURCE COVERAGE


,group,columns,source
0,Original Data Engineering,85,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
1,Additional Behavioral Features,30,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
2,New Rotation Features,27,Feature Generation Contract
3,Binary Target,1,User-Day Ground Truth



B — DOCUMENTATION GAPS
No missing required documentation fields.

C — DOMINANT CHANNEL DEFINITIONS


,feature_name,dominant_channel_definition
93,dominant_channel_switches_5wd,Dominant Activity Channel: argmax(Count_c/P95_...
94,dominant_channel_switches_10wd,Dominant Activity Channel: argmax(Count_c/P95_...
95,dominant_channel_switches_20wd,Dominant Activity Channel: argmax(Count_c/P95_...
96,dominant_channel_switches_30wd,Dominant Activity Channel: argmax(Count_c/P95_...
127,nr_alt_5wd,"Dominant Deviation Channel: argmax(E_c), where..."
128,nr_alt_10wd,"Dominant Deviation Channel: argmax(E_c), where..."
129,nr_alt_20wd,"Dominant Deviation Channel: argmax(E_c), where..."
136,nr_gap_alt_2_5wd,"Dominant Deviation Channel: argmax(E_c), where..."
137,nr_gap_alt_2_10wd,"Dominant Deviation Channel: argmax(E_c), where..."
138,nr_gap_alt_2_20wd,"Dominant Deviation Channel: argmax(E_c), where..."



D — FINAL DATA DICTIONARY PREVIEW


,column_order,feature_name,description,calculation,source_columns,time_scope,notes,category,documentation_source,data_type,null_rows,nullable,column_role,effective_window_policy,methodology_notes,dominant_channel_definition
0,1,user,Unique identifier of the user.,Original user identifier.,user,User level,,Identifier,Data Engineering feature dictionary,string,0,NO,Identifier,,,
1,2,day,Calendar day used to aggregate the user's acti...,CAST(timestamp AS DATE),date / timestamp,Daily,,Calendar,Data Engineering feature dictionary,datetime64[s],0,NO,Identifier,,,
2,3,day_of_week,"Numeric day of the week, where Monday = 1 and ...","strftime(day, '%u')",day,Daily,,Calendar,Data Engineering feature dictionary,int32,0,NO,Behavioral Feature,,,
3,4,is_weekend,Indicates whether the day is Saturday or Sunday.,"1 if day_of_week in (6,7), else 0",day_of_week,Daily,The behavior dataset contains weekday workdays...,Calendar,Data Engineering feature dictionary,int32,0,NO,Behavioral Feature,,,
4,5,email_event_count,Total number of email events generated by the ...,COUNT(email events),email.csv,Daily,,Email,Data Engineering feature dictionary,int64,0,NO,Behavioral Feature,,,
5,6,email_send_count,Number of email Send events during the day.,COUNT(activity = 'Send'),email.activity,Daily,,Email,Data Engineering feature dictionary,float64,0,NO,Behavioral Feature,,,
6,7,email_view_count,Number of email View events during the day.,COUNT(activity = 'View'),email.activity,Daily,,Email,Data Engineering feature dictionary,float64,0,NO,Behavioral Feature,,,
7,8,external_email_count,Number of sent emails containing at least one ...,COUNT(external Send events),"to, cc, bcc, activity",Daily,Recipients are parsed individually and empty r...,Email,Data Engineering feature dictionary,float64,0,NO,Behavioral Feature,,,
8,9,internal_email_count,Number of sent emails classified as internal.,email_send_count - external_email_count,"to, cc, bcc, activity",Daily,,Email,Data Engineering feature dictionary,float64,0,NO,Behavioral Feature,,,
9,10,total_sent_email_bytes,Total size in bytes of emails sent by the user...,SUM(size) for Send events,email.size,Daily,,Email,Data Engineering feature dictionary,float64,0,NO,Behavioral Feature,,,



12.9.2 — FINAL DICTIONARY SUMMARY
Project root           : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration
Documented columns     : 143
Original definitions   : 85
Additional features    : 30
Rotation features      : 27
Target labels          : 1
Missing documentation  : 0
Exact dataset coverage : True
Historical baseline    : UNCHANGED
Files exported         : NO
12.9.2 STATUS: PASS — DICTIONARY READY
Next: 12.9.3 — Methodology & Final QA Documentation



## 12.9.3 — Methodology & Final QA Documentation

### Objective

Prepare the final methodology and quality-assurance documentation for the labeled behavioral feature dataset.

### Deliverables

**1. Feature Methodology and Design Decisions**

Document:
- Dataset population, structure, and temporal scope.
- Immutable historical behavioral baselines.
- Partial rolling windows beginning in 2011.
- Feature-specific dominant-channel definitions.
- Missing baseline and undefined-ratio handling.
- Binary target semantics.
- Model-input and evaluation limitations.

**2. Final Feature Quality Assurance Report**

Consolidate:
- Feature reconstruction and numerical validation.
- Independent feature dependency verification.
- Full-dataset integrity checks.
- Missing-value investigation.
- Ground Truth and execution-log reconciliation.
- Independent binary label validation.
- Direct metadata leakage checks.
- Remaining modeling considerations.

### Verification Requirements

- All required validation results must exist in the current Jupyter session.
- Zero unexplained numerical, labeling, or source mismatches.
- Correct final dataset dimensions and label counts.
- No unapproved modifications to source data.
- Document limitations separately from successful tests.

### Output

- `feature_methodology_md_12`
- `final_feature_qa_md_12`

Both outputs remain in memory.

### Restrictions

No files exported, no directory creation, and no source modification.

### Next Step

**12.9.4 — Final Dataset and Documentation Export**


In [ ]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.9.3
# METHODOLOGY & FINAL QA DOCUMENTATION
#
# Source-grounded and validated from current Jupyter results
# IN-MEMORY ONLY — NO FILE EXPORTS
# ============================================================

from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display


def require_1293(condition, message):
    if not bool(condition):
        raise AssertionError(f"12.9.3 FAILED: {message}")


# ------------------------------------------------------------
# STEP 1 — Verify required results exist
# ------------------------------------------------------------

required_objects_1293 = [
    "behavior_features_labeled_12",
    "final_feature_dictionary_12",
    "independent_qa_1253",
    "qa_1262",
    "report_126B",
    "audit_1263",
    "qa_1264",
    "qa_128",
    "rolling_qa_128",
    "log_128",
    "gt_128",
]

missing_objects_1293 = [
    name for name in required_objects_1293
    if name not in globals()
]

require_1293(
    not missing_objects_1293,
    "Missing validation results: "
    + ", ".join(missing_objects_1293)
)

df_1293 = behavior_features_labeled_12
dictionary_1293 = final_feature_dictionary_12

require_1293(
    df_1293.shape == (387162, 143),
    "Unexpected final dataset dimensions."
)

require_1293(
    len(dictionary_1293) == 143
    and list(dictionary_1293["feature_name"])
    == list(df_1293.columns),
    "Data dictionary is incomplete or misaligned."
)

# ------------------------------------------------------------
# STEP 2 — Verify QA evidence before documentation
# ------------------------------------------------------------

require_1293(
    len(independent_qa_1253) == 30
    and independent_qa_1253["status"].eq("PASS").all()
    and independent_qa_1253["value_mismatches"].sum() == 0
    and independent_qa_1253["nan_mismatches"].sum() == 0,
    "30-feature independent verification is incomplete."
)

require_1293(
    len(qa_1262) == 27
    and qa_1262["status"].eq("PASS").all()
    and qa_1262["value_mismatches"].sum() == 0
    and qa_1262["nan_mismatches"].sum() == 0,
    "27-feature independent verification is incomplete."
)

require_1293(
    len(audit_1263) == 85
    and audit_1263["mismatches"].sum() == 0,
    "Original feature dependency audit did not pass."
)

require_1293(
    len(qa_1264) == 108
    and qa_1264["failures"].sum() == 0,
    "Full dataset integrity audit did not pass."
)

require_1293(
    len(qa_128) == 10
    and qa_128["status"].eq("PASS").all(),
    "Independent label/metadata QA is incomplete."
)

require_1293(
    len(rolling_qa_128) == 12
    and rolling_qa_128["mismatches"].sum() == 0,
    "Independent rolling audit did not pass."
)

require_1293(
    report_126B["classification_complete"].all()
    and report_126B["expected_inactivity_pattern"].all(),
    "Missing-P90 historical activity audit did not pass."
)

require_1293(
    len(log_128) == 10124
    and int(log_128["count_added"].sum()) == 29007
    and len(gt_128) == 9080,
    "Ground Truth or injection evidence mismatch."
)

require_1293(
    int(df_1293["y"].eq(1).sum()) == 9080
    and int(df_1293["y"].eq(0).sum()) == 378082
    and df_1293["user"].nunique() == 3737,
    "Final target distribution mismatch."
)

require_1293(
    not direct_leaks_128
    and not target_duplicates_128,
    "Direct metadata or exact target duplication detected."
)


# ------------------------------------------------------------
# STEP 3 — Extract verified QA metrics
# ------------------------------------------------------------

metrics_1293 = {
    "rows": len(df_1293),
    "columns": df_1293.shape[1],
    "users": df_1293["user"].nunique(),
    "positive": int(df_1293["y"].sum()),
    "negative": int(df_1293["y"].eq(0).sum()),
    "attackers": df_1293.loc[
        df_1293["y"].eq(1), "user"
    ].nunique(),
    "injection_actions": len(log_128),
    "injected_events": int(log_128["count_added"].sum()),
    "original_checked": len(audit_1263),
    "original_values": int(
        audit_1263["rows_checked"].sum()
    ),
    "behavioral_checked": len(independent_qa_1253),
    "behavioral_values": int(
        independent_qa_1253["checked_values"].sum()
    ),
    "rotation_checked": len(qa_1262),
    "rotation_values": int(
        qa_1262["tested_values"].sum()
    ),
    "integrity_checks": len(qa_1264),
    "label_checks": len(qa_128),
    "rolling_checks": len(rolling_qa_128),
    "columns_with_nan": int(
        df_1293.isna().any().sum()
    ),
}

# Historical P90 missingness, from actual dataset.
p90_missing_1293 = {
    name: int(df_1293[name].isna().sum())
    for name in [
        "email_count_p90",
        "usb_copy_p90",
        "http_upload_p90",
    ]
}

# ------------------------------------------------------------
# STEP 4 — Feature-specific dominant-channel definitions
# ------------------------------------------------------------

dominant_rows_1293 = []

dominant_features_1293 = dictionary_1293.loc[
    dictionary_1293[
        "dominant_channel_definition"
    ].astype("string").str.strip().ne(""),
    ["feature_name", "dominant_channel_definition"]
]

require_1293(
    len(dominant_features_1293) == 10,
    "Expected 10 features with explicit dominant "
    "channel definitions."
)

for _, row in dominant_features_1293.iterrows():
    dominant_rows_1293.append(
        f"| `{row['feature_name']}` | "
        f"{row['dominant_channel_definition']} |"
    )

dominant_table_1293 = "\n".join(dominant_rows_1293)


# ------------------------------------------------------------
# STEP 5 — Build methodology document in memory
# ------------------------------------------------------------

feature_methodology_md_12 = f"""# MUBSIR — Feature Methodology and Design Decisions

## 1. Dataset Overview

- Dataset: CERT r6.2, behavioral features following validated synthetic event-count injection.
- Granularity: One eligible user-workday per row.
- Users: {metrics_1293['users']:,}.
- Records: {metrics_1293['rows']:,}.
- Columns: {metrics_1293['columns']} (including the binary target).
- Original Data Engineering columns: 85.
- Additional behavioral features: 30.
- New Rotation features: 27.
- Binary target: `y`.

The dataset preserves user identifiers, temporal fields,
historical reference statistics, and behavioral measurements.
Not all columns are intended as model predictors.

## 2. Historical Baseline

The historical behavioral baseline is calculated from the
2010 reference period and remains immutable throughout
2011 feature generation and synthetic attack injection.

Historical percentile, active-rate, and related reference
statistics are reused without recalculation.

A missing historical statistic is not automatically replaced
with zero.

## 3. Workday and Rolling-Window Policy

An eligible workday has `is_weekend = 0` and
`logon_count > 0`.

Rolling windows use the last eligible workdays for each user,
including the current day.

The approved windows are:

- Original and additional engineered features: 5, 10, 20,
  and 30 eligible workdays where defined.
- New Rotation features: 5, 10, and 20 eligible workdays.

**Effective window policy:**

- Rolling history starts in 2011.
- Incomplete initial windows are calculated using available
  eligible workdays (`min_periods=1`).
- No activity from 2010 enters rolling-window calculations.
- No future activity is used.

The original source dictionaries may document an earlier
full-window startup policy. The effective policy above
supersedes that startup convention for this dataset.

The historical 2010 baseline remains unchanged.

## 4. Dominant Channel Definitions

The dataset uses two distinct operational definitions
of dominant channel. They describe different behavioral
measurements and must not be treated as interchangeable.

### Feature-specific definitions

| Feature | Dominant-channel definition |
|---|---|
{dominant_table_1293}

**Dominant Activity Channel**

The channel with the highest normalized activity
`Count_c / P95_c` among channels with valid P95.

Days without a positive normalized maximum and days
with approximately tied maxima are excluded from
the switching sequence.

Approximate equality uses `np.isclose` with
`rtol=1e-9` and `atol=1e-12`.

**Dominant Deviation Channel**

The channel with the highest positive deviation:

`E_c(d) = max(0, Count_c(d) - P50_c) / max(1, P95_c - P50_c)`

Missing count or baseline values produce `E=0`.
A historical `P50=0` is permitted when present.

Ties are resolved in the order EMAIL, USB, HTTP.
Days with no positive deviation have no dominant channel.

### Interpretation

The activity-based definition measures changes in the
channel showing the greatest relative activity.

The deviation-based definition measures changes in the
channel showing the greatest deviation from historical behavior.

These features provide different perspectives on temporal
channel changes. Their predictive usefulness must be
evaluated during model training.

## 5. Missing Historical Statistics

Missing P90 values were investigated independently.

| Channel | User-Day rows with missing P90 |
|---|---:|
| EMAIL | {p90_missing_1293['email_count_p90']:,} |
| USB | {p90_missing_1293['usb_copy_p90']:,} |
| HTTP | {p90_missing_1293['http_upload_p90']:,} |

The historical activity audit established that users
with missing P90 statistics in the examined channels
had zero recorded historical active days in those channels.

Daily P90 ratios are defined only when P90 is positive.
Otherwise, the ratio remains NaN.

Missing values are preserved for explicit handling
during model preprocessing, rather than automatically
being interpreted as zero activity in 2011.

## 6. Binary Label Definition

- `y=1`: Verified positive synthetic injection on the User-Day.
- `y=0`: No synthetic injection on the User-Day.

Labels are assigned through exact `(user, day)` matching
with approved User-Day Ground Truth.

A user's assigned attack status is not used to label
all of that user's days as attacks.

The label describes synthetic injection and does not
independently certify that all original activities
on non-injected days are benign.

## 7. Feature Selection and Leakage Considerations

- `y` is the target and must never be a predictor.
- Ground Truth, planned schedules, and execution
  metadata must not enter predictor columns.
- `user` and `day` require explicit model-input decisions.
- Historical reference columns are needed for feature
  engineering but are not automatically approved predictors.
- Constant columns, including `is_weekend` and
  `unique_usb_extensions`, require feature-selection review.
- Missing-value handling must be fitted using training
  data only, following the approved train/test protocol.

## 8. Implementation Boundaries

The dataset represents synthetic count-based injection
and behavioral feature construction, not real file transfers.

Successful QA confirms tested computational consistency;
it does not establish model detection performance,
attack realism, or complete absence of statistical leakage.

Production deployment in another environment requires
behavioral baseline recalibration.

## 9. Documentation Sources

- Original Data Engineering feature dictionary.
- Approved 30-feature calculation dictionary.
- Feature Generation Contract for the 27 additional features.
- Notebook 10 injection execution evidence.
- Notebook 11 validated User-Day Ground Truth.
- Notebook 12 feature generation and validation results.
"""


# ------------------------------------------------------------
# STEP 6 — Build QA report from verified results
# ------------------------------------------------------------

qa_report_rows_1293 = [
    (
        "30 additional behavioral features",
        f"{metrics_1293['behavioral_checked']} features; "
        f"{metrics_1293['behavioral_values']:,} sampled values",
        "0 numerical/NaN mismatches",
    ),
    (
        "27 rotation/deviation features",
        f"{metrics_1293['rotation_checked']} features; "
        f"{metrics_1293['rotation_values']:,} sampled values",
        "0 numerical/NaN mismatches",
    ),
    (
        "Original feature dependency audit",
        f"{metrics_1293['original_checked']} columns; "
        f"{metrics_1293['original_values']:,} values",
        "0 mismatches",
    ),
    (
        "Full dataset integrity",
        f"{metrics_1293['integrity_checks']} checks",
        "0 rule failures",
    ),
    (
        "Independent label and source validation",
        f"{metrics_1293['label_checks']} checks",
        "All passed",
    ),
    (
        "Independent rolling-count verification",
        f"{metrics_1293['rolling_checks']} features",
        "0 mismatches",
    ),
    (
        "Final feature dictionary",
        f"{len(dictionary_1293)} documented columns",
        "Complete schema coverage",
    ),
]

qa_table_1293 = "\n".join(
    f"| {name} | {scope} | {result} |"
    for name, scope, result in qa_report_rows_1293
)

final_feature_qa_md_12 = f"""# MUBSIR — Final Feature Quality Assurance Report

## 1. Dataset Summary

| Metric | Verified result |
|---|---:|
| User-Day records | {metrics_1293['rows']:,} |
| Users | {metrics_1293['users']:,} |
| Columns including target | {metrics_1293['columns']} |
| Positive User-Days (`y=1`) | {metrics_1293['positive']:,} |
| Non-injected User-Days (`y=0`) | {metrics_1293['negative']:,} |
| Users with positive labels | {metrics_1293['attackers']:,} |
| Executed channel actions | {metrics_1293['injection_actions']:,} |
| Total added synthetic events | {metrics_1293['injected_events']:,} |

## 2. Validated QA Evidence

| Validation | Scope | Result |
|---|---|---|
{qa_table_1293}

## 3. Ground Truth and Execution Reconciliation

The independent execution log was used to reconstruct
the positive User-Day population.

- Execution actions: {metrics_1293['injection_actions']:,}.
- Injected synthetic events: {metrics_1293['injected_events']:,}.
- Reconstructed positive User-Days: {metrics_1293['positive']:,}.
- Ground Truth key mismatches: 0.
- Ground Truth event-count mismatches: 0.
- Binary label mismatches: 0.

The 142 pre-labeling columns were preserved during
binary target integration.

## 4. Historical Baseline and Missing Values

Historical reference fields remained unchanged.

Missing historical P90 values were investigated
against recorded historical active-day counts.

All examined missing-P90 users had zero recorded
historical active days in the corresponding channel.

Undefined ratios remain NaN.

Columns with at least one missing value:
{metrics_1293['columns_with_nan']}.

## 5. Constant-Column Review

The following columns were constant in the inspected
feature population:

- `is_weekend`: zero because only eligible weekdays
  are present.
- `unique_usb_extensions`: zero for every record
  in the current dataset.

The latter matched its approved source, but a
source match alone does not establish its usefulness
or independently validate upstream raw-event extraction.

Both columns require appropriate model feature-selection
decisions.

## 6. Leakage Validation

The tested checks found:

- No direct Ground Truth or attack-scheduling metadata
  columns within the feature dataset.
- No numerical feature exactly equal to `y` or `1-y`.
- No mismatches in the independently checked causal
  rolling-count features.
- No modifications to the original 142 columns
  during labeling.

**Limitation:** These tests do not prove the absence
of every possible form of statistical or temporal leakage.

Model development must also validate:

- Predictor selection and exclusion of identifiers
  where appropriate.
- Training, validation, and test split isolation.
- Preprocessing fitted on training data only.
- Handling of missing values and constant features.
- Class imbalance and appropriate evaluation metrics.

## 7. Class Distribution

The positive class constitutes approximately
{metrics_1293['positive']/metrics_1293['rows']*100:.3f}%
of User-Day records.

Model performance should therefore be evaluated using
appropriate metrics including precision, recall,
F1-score, and precision-recall AUC, rather than
accuracy alone.

## 8. Final Assessment

**Status: PASS — TESTED DATASET AND LABELING RULES VERIFIED.**

The validated dataset is ready for controlled final
export and independent post-export integrity checks.

This status applies to tested computational and data
integrity conditions, not to model detection accuracy,
real-world attack realism, or complete leakage elimination.

No files were exported by this documentation stage.
"""


# ------------------------------------------------------------
# STEP 7 — Documentation checks
# ------------------------------------------------------------

require_1293(
    len(feature_methodology_md_12.strip()) > 2000,
    "Methodology document appears incomplete."
)

require_1293(
    len(final_feature_qa_md_12.strip()) > 1000,
    "Final QA report appears incomplete."
)

require_1293(
    all(
        name in feature_methodology_md_12
        for name in dominant_features_1293["feature_name"]
    ),
    "Feature-specific dominant-channel definitions missing."
)

# No team-member attribution in scientific documentation.
for text_doc in (
    feature_methodology_md_12,
    final_feature_qa_md_12,
):
    require_1293(
        "Clear_Attack_Simulation" not in text_doc
        and "Reef" not in text_doc,
        "Internal implementation names found in documentation."
    )

documentation_audit_1293 = pd.DataFrame([
    {
        "document": "feature_methodology_and_decisions.md",
        "characters": len(feature_methodology_md_12),
        "status": "READY — IN MEMORY",
    },
    {
        "document": "final_feature_qa_report.md",
        "characters": len(final_feature_qa_md_12),
        "status": "READY — IN MEMORY",
    },
])

print("\nA — DOCUMENTATION READINESS")
display(documentation_audit_1293)

print("\nB — QA EVIDENCE INCLUDED")
display(pd.DataFrame(
    qa_report_rows_1293,
    columns=["validation", "scope", "result"],
))

print("\nC — FEATURE-SPECIFIC DOMINANT DEFINITIONS")
display(dominant_features_1293)

print("\nD — METHODOLOGY DOCUMENT PREVIEW")
print(feature_methodology_md_12[:1800])

print("\nE — QA REPORT PREVIEW")
print(final_feature_qa_md_12[:1500])

print("\n" + "=" * 60)
print("12.9.3 — DOCUMENTATION SUMMARY")
print("=" * 60)
print("Methodology document      : READY")
print("Final QA report           : READY")
print("Documented feature columns:", len(dictionary_1293))
print("Original dataset modified : NO")
print("Historical baseline       : UNCHANGED")
print("Files exported            : NO")
print("=" * 60)
print("12.9.3 STATUS: PASS — DOCUMENTATION PREPARED")
print("Next: 12.9.4 — Controlled Final Export")



A — DOCUMENTATION READINESS


,document,characters,status
0,feature_methodology_and_decisions.md,7599,READY — IN MEMORY
1,final_feature_qa_report.md,3778,READY — IN MEMORY



B — QA EVIDENCE INCLUDED


,validation,scope,result
0,30 additional behavioral features,"30 features; 107,970 sampled values",0 numerical/NaN mismatches
1,27 rotation/deviation features,"27 features; 226,314 sampled values",0 numerical/NaN mismatches
2,Original feature dependency audit,"85 columns; 32,908,770 values",0 mismatches
3,Full dataset integrity,108 checks,0 rule failures
4,Independent label and source validation,10 checks,All passed
5,Independent rolling-count verification,12 features,0 mismatches
6,Final feature dictionary,143 documented columns,Complete schema coverage



C — FEATURE-SPECIFIC DOMINANT DEFINITIONS


,feature_name,dominant_channel_definition
93,dominant_channel_switches_5wd,Dominant Activity Channel: argmax(Count_c/P95_...
94,dominant_channel_switches_10wd,Dominant Activity Channel: argmax(Count_c/P95_...
95,dominant_channel_switches_20wd,Dominant Activity Channel: argmax(Count_c/P95_...
96,dominant_channel_switches_30wd,Dominant Activity Channel: argmax(Count_c/P95_...
127,nr_alt_5wd,"Dominant Deviation Channel: argmax(E_c), where..."
128,nr_alt_10wd,"Dominant Deviation Channel: argmax(E_c), where..."
129,nr_alt_20wd,"Dominant Deviation Channel: argmax(E_c), where..."
136,nr_gap_alt_2_5wd,"Dominant Deviation Channel: argmax(E_c), where..."
137,nr_gap_alt_2_10wd,"Dominant Deviation Channel: argmax(E_c), where..."
138,nr_gap_alt_2_20wd,"Dominant Deviation Channel: argmax(E_c), where..."



D — METHODOLOGY DOCUMENT PREVIEW
# MUBSIR — Feature Methodology and Design Decisions

## 1. Dataset Overview

- Dataset: CERT r6.2, behavioral features following validated synthetic event-count injection.
- Granularity: One eligible user-workday per row.
- Users: 3,737.
- Records: 387,162.
- Columns: 143 (including the binary target).
- Original Data Engineering columns: 85.
- Additional behavioral features: 30.
- New Rotation features: 27.
- Binary target: `y`.

The dataset preserves user identifiers, temporal fields,
historical reference statistics, and behavioral measurements.
Not all columns are intended as model predictors.

## 2. Historical Baseline

The historical behavioral baseline is calculated from the
2010 reference period and remains immutable throughout
2011 feature generation and synthetic attack injection.

Historical percentile, active-rate, and related reference
statistics are reused without recalculation.

A missing historical statistic is not automatically replaced


## 12.9.4 — Controlled Final Export

### Objective

Export the approved labeled behavioral dataset, feature dictionary, methodology documentation, final QA report, and integrity manifest.

### Input Artifacts

- `behavior_features_labeled_12`
- `final_feature_dictionary_12`
- `feature_methodology_md_12`
- `final_feature_qa_md_12`

### Export Strategy

1. Verify all required in-memory artifacts.
2. Block export if the final destination already exists.
3. Write the deliverables to a temporary staging directory.
4. Generate SHA-256 checksums for the four primary artifacts.
5. Create the export manifest containing filenames, sizes, checksums, and dataset metadata.
6. Verify the staging contents.
7. Commit the complete staging directory to the final destination.
8. Preserve all original datasets, historical baselines, and previously approved reports.

### Deliverables

- `behavior_features_r62_ml_final_new.parquet`
- `feature_dictionary_r62_final.csv`
- `feature_methodology_and_decisions.md`
- `final_feature_qa_report.md`
- `export_manifest.json`

### Acceptance Criteria

- Final dataset contains 387,162 rows and 143 columns.
- Data dictionary documents all 143 columns.
- Both documentation files are complete.
- All five export artifacts are present.
- File sizes and SHA-256 hashes are recorded.
- No pre-existing deliverables are replaced.

### Next Step

**12.9.5 — Independent Post-Export Integrity Verification**

Reopen every exported artifact and verify the saved files against the approved in-memory results and recorded checksums.


In [ ]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.9.4
# CONTROLLED FINAL EXPORT
#
# Outputs:
# 1. Final labeled dataset (Parquet)
# 2. Feature dictionary (CSV)
# 3. Methodology document (Markdown)
# 4. Final QA report (Markdown)
# 5. Export manifest (JSON)
#
# NO OVERWRITE — STAGED EXPORT — SHA-256
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import tempfile

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


def require_1294(condition, message):
    if not bool(condition):
        raise RuntimeError(
            f"12.9.4 EXPORT BLOCKED: {message}"
        )


def sha256_1294(path):
    """Calculate SHA-256 without loading the full file."""
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# STEP 1 — Verify all approved in-memory artifacts
# ------------------------------------------------------------

required_1294 = [
    "behavior_features_labeled_12",
    "final_feature_dictionary_12",
    "feature_methodology_md_12",
    "final_feature_qa_md_12",
    "REPORT_1291",
]

missing_1294 = [
    name for name in required_1294
    if name not in globals()
]

require_1294(
    not missing_1294,
    "Missing in-memory artifacts: "
    + ", ".join(missing_1294)
)

dataset_1294 = behavior_features_labeled_12
dictionary_1294 = final_feature_dictionary_12

require_1294(
    dataset_1294.shape == (387162, 143),
    "Invalid final dataset dimensions."
)

require_1294(
    dataset_1294.columns.is_unique
    and not dataset_1294.duplicated(
        ["user", "day"]
    ).any(),
    "Invalid final dataset schema or keys."
)

require_1294(
    dataset_1294["user"].nunique() == 3737,
    "Unexpected user population."
)

require_1294(
    int(dataset_1294["y"].eq(1).sum()) == 9080
    and int(dataset_1294["y"].eq(0).sum()) == 378082
    and dataset_1294["y"].notna().all(),
    "Unexpected binary label distribution."
)

require_1294(
    len(dictionary_1294) == 143
    and dictionary_1294["feature_name"].is_unique,
    "Invalid feature dictionary."
)

require_1294(
    list(dictionary_1294["feature_name"])
    == list(dataset_1294.columns),
    "Dictionary and dataset column order differ."
)

require_1294(
    isinstance(feature_methodology_md_12, str)
    and len(feature_methodology_md_12.strip()) > 2000,
    "Methodology documentation is incomplete."
)

require_1294(
    isinstance(final_feature_qa_md_12, str)
    and len(final_feature_qa_md_12.strip()) > 1000,
    "Final QA documentation is incomplete."
)


# ------------------------------------------------------------
# STEP 2 — Validate final destination
# ------------------------------------------------------------

REPORT_1294 = Path(REPORT_1291).resolve()

REPORTS_PARENT_1294 = REPORT_1294.parent

require_1294(
    REPORTS_PARENT_1294.is_dir(),
    f"Reports parent directory missing: "
    f"{REPORTS_PARENT_1294}"
)

# Do not overwrite an existing directory,
# even if it appears empty.
require_1294(
    not REPORT_1294.exists(),
    f"Destination already exists: {REPORT_1294}"
)

FILENAMES_1294 = {
    "dataset":
        "behavior_features_r62_ml_final_new.parquet",
    "dictionary":
        "feature_dictionary_r62_final.csv",
    "methodology":
        "feature_methodology_and_decisions.md",
    "qa_report":
        "final_feature_qa_report.md",
    "manifest":
        "export_manifest.json",
}

require_1294(
    len(set(FILENAMES_1294.values())) == 5,
    "Duplicate output filenames."
)


# ------------------------------------------------------------
# STEP 3 — Create isolated staging directory
#
# Temporary files are placed on the same filesystem
# as the final destination.
# ------------------------------------------------------------

STAGING_1294 = Path(
    tempfile.mkdtemp(
        prefix=".nb12_final_export_staging_",
        dir=REPORTS_PARENT_1294,
    )
)

export_committed_1294 = False

try:

    # --------------------------------------------------------
    # STEP 4 — Write the four primary artifacts to staging
    # --------------------------------------------------------

    dataset_path_1294 = (
        STAGING_1294 / FILENAMES_1294["dataset"]
    )

    dictionary_path_1294 = (
        STAGING_1294 / FILENAMES_1294["dictionary"]
    )

    methodology_path_1294 = (
        STAGING_1294 / FILENAMES_1294["methodology"]
    )

    qa_report_path_1294 = (
        STAGING_1294 / FILENAMES_1294["qa_report"]
    )

    dataset_1294.to_parquet(
        dataset_path_1294,
        engine="pyarrow",
        compression="zstd",
        index=False,
    )

    dictionary_1294.to_csv(
        dictionary_path_1294,
        index=False,
        encoding="utf-8",
    )

    methodology_path_1294.write_text(
        feature_methodology_md_12,
        encoding="utf-8",
    )

    qa_report_path_1294.write_text(
        final_feature_qa_md_12,
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # STEP 5 — Verify staged content before committing
    # --------------------------------------------------------

    primary_paths_1294 = {
        key: STAGING_1294 / filename
        for key, filename in FILENAMES_1294.items()
        if key != "manifest"
    }

    for role, path in primary_paths_1294.items():
        require_1294(
            path.is_file() and path.stat().st_size > 0,
            f"Missing or empty staged file: {role}"
        )

    parquet_metadata_1294 = pq.read_metadata(
        dataset_path_1294
    )

    require_1294(
        parquet_metadata_1294.num_rows == 387162
        and parquet_metadata_1294.num_columns == 143,
        "Staged Parquet dimensions are incorrect."
    )

    dictionary_check_1294 = pd.read_csv(
        dictionary_path_1294,
        usecols=["column_order", "feature_name"],
        keep_default_na=False,
    )

    require_1294(
        len(dictionary_check_1294) == 143
        and dictionary_check_1294["feature_name"].tolist()
        == list(dataset_1294.columns)
        and dictionary_check_1294["column_order"].tolist()
        == list(range(1, 144)),
        "Staged dictionary does not match dataset schema."
    )

    require_1294(
        methodology_path_1294.read_text(
            encoding="utf-8"
        ) == feature_methodology_md_12,
        "Staged methodology text mismatch."
    )

    require_1294(
        qa_report_path_1294.read_text(
            encoding="utf-8"
        ) == final_feature_qa_md_12,
        "Staged QA report text mismatch."
    )


    # --------------------------------------------------------
    # STEP 6 — Generate SHA-256 checksums and manifest
    # --------------------------------------------------------

    file_records_1294 = []

    for role, path in primary_paths_1294.items():

        file_records_1294.append({
            "role": role,
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_1294(path),
        })

    manifest_1294 = {
        "project": "MUBSIR",
        "dataset_version": "CERT r6.2",
        "notebook": "12 — Feature Engineering and Validation",
        "export_time_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "dataset": {
            "rows": 387162,
            "columns": 143,
            "users": 3737,
            "positive_user_days": 9080,
            "negative_user_days": 378082,
            "positive_users": 400,
            "granularity": "Eligible User-Day",
            "target_column": "y",
        },
        "methodology": {
            "rolling_policy":
                "Partial eligible-workday windows from 2011",
            "historical_baseline":
                "Fixed historical 2010 baseline",
            "ground_truth":
                "Verified synthetic User-Day injection",
        },
        "files": file_records_1294,
        "checksum_algorithm": "SHA-256",
        "note": (
            "Checksums cover the four primary artifacts. "
            "The manifest itself does not contain its own hash."
        ),
    }

    manifest_path_1294 = (
        STAGING_1294 / FILENAMES_1294["manifest"]
    )

    manifest_path_1294.write_text(
        json.dumps(
            manifest_1294,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    require_1294(
        manifest_path_1294.is_file(),
        "Export manifest was not created."
    )


    # --------------------------------------------------------
    # STEP 7 — Independently verify staged checksums
    # --------------------------------------------------------

    loaded_manifest_1294 = json.loads(
        manifest_path_1294.read_text(
            encoding="utf-8"
        )
    )

    for item in loaded_manifest_1294["files"]:

        staged_path = STAGING_1294 / item["filename"]

        require_1294(
            staged_path.is_file(),
            f"Missing staged artifact: {item['filename']}"
        )

        require_1294(
            staged_path.stat().st_size == item["size_bytes"],
            f"Size mismatch: {item['filename']}"
        )

        require_1294(
            sha256_1294(staged_path) == item["sha256"],
            f"SHA-256 mismatch: {item['filename']}"
        )

    require_1294(
        set(p.name for p in STAGING_1294.iterdir())
        == set(FILENAMES_1294.values()),
        "Unexpected or missing files in staging."
    )


    # --------------------------------------------------------
    # STEP 8 — Commit complete staged directory
    #
    # One directory rename, not five separate file moves.
    # Destination must not exist.
    # --------------------------------------------------------

    require_1294(
        not REPORT_1294.exists(),
        "Destination appeared during export. "
        "Existing files were not replaced."
    )

    STAGING_1294.rename(REPORT_1294)

    export_committed_1294 = True

finally:

    # Clean up only this operation's own staging folder.
    # Never delete or overwrite an existing output folder.
    if not export_committed_1294 and STAGING_1294.exists():
        shutil.rmtree(STAGING_1294)


# ------------------------------------------------------------
# STEP 9 — Final committed-file inventory
# ------------------------------------------------------------

require_1294(
    export_committed_1294
    and REPORT_1294.is_dir(),
    "Final export was not committed."
)

export_rows_1294 = []

for role, name in FILENAMES_1294.items():

    path = REPORT_1294 / name

    export_rows_1294.append({
        "role": role,
        "filename": name,
        "exists": path.is_file(),
        "size_mb": round(
            path.stat().st_size / 1024**2, 3
        ) if path.is_file() else None,
    })

export_inventory_1294 = pd.DataFrame(
    export_rows_1294
)

print("\nA — FINAL EXPORTED ARTIFACTS")
display(export_inventory_1294)

print("\nB — SHA-256 MANIFEST RECORDS")
display(
    pd.DataFrame(file_records_1294)
)

print("\n" + "=" * 60)
print("12.9.4 — CONTROLLED FINAL EXPORT")
print("=" * 60)
print("Export directory :", REPORT_1294)
print("Files exported   :", len(export_inventory_1294))
print("Dataset rows     :", dataset_1294.shape[0])
print("Dataset columns  :", dataset_1294.shape[1])
print("Dictionary rows  :", len(dictionary_1294))
print("Manifest records :", len(file_records_1294))
print("Original modified: NO")
print("Overwrite used   : NO")
print("=" * 60)

print("12.9.4 STATUS: PASS — EXPORT COMMITTED")
print("Next: 12.9.5 — Independent Post-Export Verification")



A — FINAL EXPORTED ARTIFACTS


,role,filename,exists,size_mb
0,dataset,behavior_features_r62_ml_final_new.parquet,True,20.779
1,dictionary,feature_dictionary_r62_final.csv,True,0.078
2,methodology,feature_methodology_and_decisions.md,True,0.007
3,qa_report,final_feature_qa_report.md,True,0.004
4,manifest,export_manifest.json,True,0.002



B — SHA-256 MANIFEST RECORDS


,role,filename,size_bytes,sha256
0,dataset,behavior_features_r62_ml_final_new.parquet,21788560,446047aa1276f154e845e92c01bfd4fe2bf8cfc7fc97e7...
1,dictionary,feature_dictionary_r62_final.csv,81288,442708d3f68beee95481011fcf5e30b22d7cb9f8beff09...
2,methodology,feature_methodology_and_decisions.md,7785,52fa079df13799c10172b4d8ba6fdef3750b6e16638aab...
3,qa_report,final_feature_qa_report.md,3903,0492cd81cafe0417afb1e3a72ce940415104c5faddb746...



12.9.4 — CONTROLLED FINAL EXPORT
Export directory : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\12_Feature_Engineering_and_Validation
Files exported   : 5
Dataset rows     : 387162
Dataset columns  : 143
Dictionary rows  : 143
Manifest records : 4
Original modified: NO
Overwrite used   : NO
12.9.4 STATUS: PASS — EXPORT COMMITTED
Next: 12.9.5 — Independent Post-Export Verification



## 12.9.5 — Independent Post-Export Integrity Verification

### Objective

Independently verify the five finalized Notebook 12 deliverables after export by reading their actual contents from disk.

### Verification Scope

**1. Final Behavioral Dataset**
- Reopen the complete Parquet file.
- Verify all 387,162 rows and 143 columns.
- Compare every saved value against the approved in-memory dataset.
- Verify column order, data types, missing values, and User-Day keys.
- Reconfirm binary target distribution.

**2. Feature Data Dictionary**
- Reopen the exported CSV.
- Confirm complete documentation of all 143 columns.
- Compare every dictionary field with the approved in-memory version.
- Verify exact feature names and ordering.

**3. Methodology and QA Documentation**
- Reopen both Markdown files.
- Compare their entire text against the approved documentation.

**4. Manifest and SHA-256**
- Validate the manifest structure and declared dataset metadata.
- Recalculate SHA-256 for all four primary artifacts.
- Verify file sizes, checksums, names, and directory contents.

### Acceptance Criteria

- Zero dataset value, schema, or dtype mismatches.
- Zero dictionary mismatches.
- Zero documentation text mismatches.
- Zero SHA-256 or file-size mismatches.
- All five expected files exist, without additional artifacts.
- No source or exported file modifications.

### Expected Status

`PASS — FINAL DATASET AND ALL ARTIFACTS VERIFIED`

### Completion

Successful verification completes Notebook 12 and establishes the validated final dataset package for model-development handoff.


In [54]:

# ============================================================
# NOTEBOOK 12 — SECTION 12.9.5
# INDEPENDENT POST-EXPORT INTEGRITY VERIFICATION
# CORRECTED VERSION
#
# Fix:
# Normalize only the 'day' datetime storage unit
# for exact in-memory vs exported dataset comparison.
#
# Full Parquet value verification
# Full dictionary and documentation comparison
# Independent SHA-256 verification
#
# STRICTLY READ-ONLY — NO EXPORTS / NO MODIFICATIONS
# ============================================================

from pathlib import Path
from io import StringIO
import json
import hashlib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.testing import assert_frame_equal
from IPython.display import display


# ------------------------------------------------------------
# STEP 0 — Validation helpers
# ------------------------------------------------------------

def require_1295(condition, message):
    if not bool(condition):
        raise RuntimeError(
            f"12.9.5 VERIFICATION FAILED: {message}"
        )


def sha256_1295(path):
    """Calculate SHA-256 from the actual exported file."""

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


checks_1295 = []


def record_1295(category, check, passed, details=""):

    checks_1295.append({
        "category": category,
        "check": check,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": str(details),
    })


# ------------------------------------------------------------
# STEP 1 — Resolve final export directory
# ------------------------------------------------------------

required_memory_1295 = [
    "REPORT_1291",
    "behavior_features_labeled_12",
    "final_feature_dictionary_12",
    "feature_methodology_md_12",
    "final_feature_qa_md_12",
]

missing_memory_1295 = [
    name for name in required_memory_1295
    if name not in globals()
]

require_1295(
    not missing_memory_1295,
    "Required in-memory objects are missing: "
    + ", ".join(missing_memory_1295)
)

REPORT_1295 = Path(REPORT_1291).resolve()

require_1295(
    REPORT_1295.is_dir(),
    f"Final export directory not found: {REPORT_1295}"
)

NAMES_1295 = {
    "dataset":
        "behavior_features_r62_ml_final_new.parquet",
    "dictionary":
        "feature_dictionary_r62_final.csv",
    "methodology":
        "feature_methodology_and_decisions.md",
    "qa_report":
        "final_feature_qa_report.md",
    "manifest":
        "export_manifest.json",
}

PATHS_1295 = {
    role: REPORT_1295 / filename
    for role, filename in NAMES_1295.items()
}


# ------------------------------------------------------------
# STEP 2 — Independent directory inventory
# ------------------------------------------------------------

actual_names_1295 = {
    p.name for p in REPORT_1295.iterdir()
}

expected_names_1295 = set(NAMES_1295.values())

record_1295(
    "DIRECTORY",
    "Exact five-file inventory",
    actual_names_1295 == expected_names_1295,
    (
        f"Missing: "
        f"{sorted(expected_names_1295 - actual_names_1295)}; "
        f"Extra: "
        f"{sorted(actual_names_1295 - expected_names_1295)}"
    )
)

for role, path in PATHS_1295.items():

    valid_file = (
        path.is_file()
        and not path.is_symlink()
        and path.stat().st_size > 0
    )

    record_1295(
        "DIRECTORY",
        f"File exists: {role}",
        valid_file,
    )

require_1295(
    all(
        path.is_file()
        for path in PATHS_1295.values()
    ),
    "One or more required export files are missing."
)


# ------------------------------------------------------------
# STEP 3 — Reopen the complete exported Parquet dataset
# ------------------------------------------------------------

approved_1295 = behavior_features_labeled_12

saved_1295 = pd.read_parquet(
    PATHS_1295["dataset"],
    engine="pyarrow",
)

record_1295(
    "DATASET",
    "Final dimensions",
    saved_1295.shape == (387162, 143),
    f"Actual shape: {saved_1295.shape}"
)

record_1295(
    "DATASET",
    "Column names and order",
    list(saved_1295.columns)
    == list(approved_1295.columns)
)

record_1295(
    "DATASET",
    "Unique User-Day keys",
    not saved_1295.duplicated(
        ["user", "day"]
    ).any()
)

record_1295(
    "DATASET",
    "User population",
    saved_1295["user"].nunique() == 3737
)

record_1295(
    "DATASET",
    "Binary target values",
    saved_1295["y"].notna().all()
    and set(saved_1295["y"].unique()) == {0, 1}
)

record_1295(
    "DATASET",
    "Positive and negative labels",
    int(saved_1295["y"].eq(1).sum()) == 9080
    and int(saved_1295["y"].eq(0).sum()) == 378082
)

record_1295(
    "DATASET",
    "Positive-label users",
    saved_1295.loc[
        saved_1295["y"].eq(1),
        "user"
    ].nunique() == 400
)


# ------------------------------------------------------------
# STEP 4 — Controlled datetime-unit normalization
#
# Only 'day' is normalized.
# Original objects and exported files stay unchanged.
# ------------------------------------------------------------

approved_compare_1295 = approved_1295.copy(deep=True)
saved_compare_1295 = saved_1295.copy(deep=True)

day_dtype_valid_1295 = (
    "day" in approved_compare_1295.columns
    and "day" in saved_compare_1295.columns
    and pd.api.types.is_datetime64_any_dtype(
        approved_compare_1295["day"]
    )
    and pd.api.types.is_datetime64_any_dtype(
        saved_compare_1295["day"]
    )
)

record_1295(
    "DATASET",
    "Datetime schema for day",
    day_dtype_valid_1295,
    (
        f"Memory: {approved_1295['day'].dtype}; "
        f"Disk: {saved_1295['day'].dtype}"
    )
)

require_1295(
    day_dtype_valid_1295,
    "'day' must be a datetime column in both datasets."
)

# Normalize only the temporal storage resolution.
# No date rounding or flooring is performed.
approved_compare_1295["day"] = (
    approved_compare_1295["day"]
    .astype("datetime64[ns]")
)

saved_compare_1295["day"] = (
    saved_compare_1295["day"]
    .astype("datetime64[ns]")
)

# Verify exact dates after normalization.
day_values_match_1295 = (
    approved_compare_1295["day"]
    .equals(saved_compare_1295["day"])
)

record_1295(
    "DATASET",
    "Exact day values after unit normalization",
    day_values_match_1295
)


# ------------------------------------------------------------
# STEP 5 — Full DataFrame exact comparison
#
# Every value, row, column, dtype and NaN must match.
# Only the already verified day datetime unit is normalized.
# ------------------------------------------------------------

dataset_comparison_error_1295 = ""

try:

    assert_frame_equal(
        approved_compare_1295,
        saved_compare_1295,
        check_dtype=True,
        check_exact=True,
        check_index_type=True,
        check_column_type=True,
        check_like=False,
    )

    full_dataset_match_1295 = True

except AssertionError as exc:

    full_dataset_match_1295 = False

    dataset_comparison_error_1295 = str(exc)[:1000]

record_1295(
    "DATASET",
    "Full DataFrame exact comparison",
    full_dataset_match_1295,
    dataset_comparison_error_1295
)


# ------------------------------------------------------------
# STEP 6 — Strict dtype verification
#
# All original dtypes must match except day,
# whose unit difference is separately validated.
# ------------------------------------------------------------

dtype_mismatches_1295 = [
    col
    for col in approved_1295.columns
    if (
        str(approved_1295[col].dtype)
        != str(saved_1295[col].dtype)
        and col != "day"
    )
]

dtype_match_1295 = (
    len(dtype_mismatches_1295) == 0
    and day_values_match_1295
    and full_dataset_match_1295
)

record_1295(
    "DATASET",
    "All dtypes preserved (day unit normalized)",
    dtype_match_1295,
    dtype_mismatches_1295
)

# Independently compare every missing-value position.
null_positions_match_1295 = (
    approved_1295.isna().to_numpy()
    == saved_1295.isna().to_numpy()
).all()

record_1295(
    "DATASET",
    "All missing-value positions preserved",
    bool(null_positions_match_1295)
)


# ------------------------------------------------------------
# STEP 7 — Verify physical Parquet metadata
# ------------------------------------------------------------

parquet_info_1295 = pq.read_metadata(
    PATHS_1295["dataset"]
)

record_1295(
    "DATASET",
    "Parquet physical metadata",
    (
        parquet_info_1295.num_rows == 387162
        and parquet_info_1295.num_columns == 143
    )
)


# ------------------------------------------------------------
# STEP 8 — Independently reopen CSV dictionary
#
# Compare normalized CSV round-trip representations,
# avoiding false mismatches from CSV type inference.
# ------------------------------------------------------------

approved_dictionary_1295 = final_feature_dictionary_12

csv_buffer_1295 = StringIO()

approved_dictionary_1295.to_csv(
    csv_buffer_1295,
    index=False,
)

csv_buffer_1295.seek(0)

expected_dictionary_1295 = pd.read_csv(
    csv_buffer_1295,
    dtype=str,
    keep_default_na=False,
)

saved_dictionary_1295 = pd.read_csv(
    PATHS_1295["dictionary"],
    dtype=str,
    keep_default_na=False,
)

record_1295(
    "DICTIONARY",
    "Dictionary dimensions",
    saved_dictionary_1295.shape
    == expected_dictionary_1295.shape
)

record_1295(
    "DICTIONARY",
    "143 documented columns",
    len(saved_dictionary_1295) == 143
    and saved_dictionary_1295["feature_name"].is_unique
)

record_1295(
    "DICTIONARY",
    "Dataset column alignment",
    saved_dictionary_1295["feature_name"].tolist()
    == saved_1295.columns.tolist()
)

dictionary_comparison_error_1295 = ""

try:

    assert_frame_equal(
        expected_dictionary_1295,
        saved_dictionary_1295,
        check_exact=True,
        check_dtype=True,
        check_like=False,
    )

    dictionary_match_1295 = True

except AssertionError as exc:

    dictionary_match_1295 = False

    dictionary_comparison_error_1295 = str(exc)[:1000]

record_1295(
    "DICTIONARY",
    "Complete dictionary content comparison",
    dictionary_match_1295,
    dictionary_comparison_error_1295
)


# ------------------------------------------------------------
# STEP 9 — Reopen and verify Markdown documents
# ------------------------------------------------------------

saved_methodology_1295 = (
    PATHS_1295["methodology"]
    .read_text(encoding="utf-8")
)

saved_qa_report_1295 = (
    PATHS_1295["qa_report"]
    .read_text(encoding="utf-8")
)

record_1295(
    "DOCUMENTATION",
    "Exact methodology document",
    saved_methodology_1295
    == feature_methodology_md_12
)

record_1295(
    "DOCUMENTATION",
    "Exact QA report",
    saved_qa_report_1295
    == final_feature_qa_md_12
)

required_methodology_1295 = [
    "Historical Baseline",
    "Workday and Rolling-Window Policy",
    "Dominant Channel Definitions",
    "Missing Historical Statistics",
    "Binary Label Definition",
    "Leakage Considerations",
]

record_1295(
    "DOCUMENTATION",
    "Required methodology sections",
    all(
        section in saved_methodology_1295
        for section in required_methodology_1295
    )
)

required_qa_1295 = [
    "Ground Truth and Execution Reconciliation",
    "Leakage Validation",
    "Final Assessment",
]

record_1295(
    "DOCUMENTATION",
    "Required QA report sections",
    all(
        section in saved_qa_report_1295
        for section in required_qa_1295
    )
)


# ------------------------------------------------------------
# STEP 10 — Independently reopen export manifest
# ------------------------------------------------------------

with open(
    PATHS_1295["manifest"],
    "r",
    encoding="utf-8",
) as f:
    manifest_1295 = json.load(f)

manifest_dataset_1295 = manifest_1295["dataset"]

metadata_expected_1295 = {
    "rows": 387162,
    "columns": 143,
    "users": 3737,
    "positive_user_days": 9080,
    "negative_user_days": 378082,
    "positive_users": 400,
    "target_column": "y",
}

record_1295(
    "MANIFEST",
    "Dataset metadata consistency",
    all(
        manifest_dataset_1295.get(key) == value
        for key, value in metadata_expected_1295.items()
    )
)

record_1295(
    "MANIFEST",
    "Checksum algorithm",
    manifest_1295.get("checksum_algorithm")
    == "SHA-256"
)

manifest_records_1295 = manifest_1295["files"]

expected_primary_1295 = {
    role: filename
    for role, filename in NAMES_1295.items()
    if role != "manifest"
}

record_1295(
    "MANIFEST",
    "Four unique primary artifact records",
    (
        len(manifest_records_1295) == 4
        and len({
            item.get("filename")
            for item in manifest_records_1295
        }) == 4
        and {
            (item.get("role"), item.get("filename"))
            for item in manifest_records_1295
        } == set(expected_primary_1295.items())
    )
)


# ------------------------------------------------------------
# STEP 11 — Independently recalculate SHA-256
# ------------------------------------------------------------

hash_rows_1295 = []

for item in manifest_records_1295:

    filename = item["filename"]

    require_1295(
        filename in expected_primary_1295.values()
        and Path(filename).name == filename,
        f"Unexpected manifest filename: {filename}"
    )

    path = REPORT_1295 / filename

    actual_size = path.stat().st_size
    actual_hash = sha256_1295(path)

    expected_size = item["size_bytes"]
    expected_hash = item["sha256"]

    size_ok = actual_size == expected_size
    hash_ok = actual_hash == expected_hash

    hash_rows_1295.append({
        "filename": filename,
        "size_match": size_ok,
        "sha256_match": hash_ok,
        "status": (
            "PASS"
            if size_ok and hash_ok
            else "FAIL"
        ),
    })

hash_report_1295 = pd.DataFrame(
    hash_rows_1295
)

record_1295(
    "MANIFEST",
    "All recorded file sizes",
    hash_report_1295["size_match"].all()
)

record_1295(
    "MANIFEST",
    "All four SHA-256 hashes",
    hash_report_1295["sha256_match"].all()
)


# ------------------------------------------------------------
# STEP 12 — Final independent verification reports
# ------------------------------------------------------------

qa_1295 = pd.DataFrame(checks_1295)

summary_1295 = (
    qa_1295
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        checks=("check", "count"),
        failed=(
            "status",
            lambda x: x.eq("FAIL").sum()
        ),
    )
)

summary_1295["status"] = np.where(
    summary_1295["failed"].eq(0),
    "PASS",
    "FAIL"
)

print("\nA — POST-EXPORT VERIFICATION SUMMARY")
display(summary_1295)

print("\nB — FAILED CHECKS")

failed_1295 = qa_1295[
    qa_1295["status"].eq("FAIL")
]

if failed_1295.empty:
    print("No failed verification checks.")
else:
    display(failed_1295)

print("\nC — FILE CHECKSUM VERIFICATION")
display(hash_report_1295)

print("\nD — SAVED DATASET LABEL DISTRIBUTION")
display(
    saved_1295["y"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("y")
    .reset_index(name="user_days")
)


# ------------------------------------------------------------
# STEP 13 — Final completion status
# ------------------------------------------------------------

total_failures_1295 = int(
    qa_1295["status"].eq("FAIL").sum()
)

print("\n" + "=" * 64)
print("12.9.5 — INDEPENDENT POST-EXPORT INTEGRITY")
print("=" * 64)
print("Export directory       :", REPORT_1295)
print("Verified dataset rows  :", len(saved_1295))
print("Verified columns       :", saved_1295.shape[1])
print("Verified users         :", saved_1295["user"].nunique())
print("Dictionary entries     :", len(saved_dictionary_1295))
print("Files inspected        :", len(PATHS_1295))
print("SHA-256 checks         :", len(hash_report_1295))
print("Integrity checks       :", len(qa_1295))
print("Failed checks          :", total_failures_1295)
print("Day datetime normalized: YES — comparison only")
print("Dataset values match   :", full_dataset_match_1295)
print("Dictionary matches     :", dictionary_match_1295)
print("Files modified         : NO")
print("Files exported         : NO")
print("=" * 64)

if total_failures_1295 == 0:

    print(
        "12.9.5 STATUS: PASS — FINAL ARTIFACTS VERIFIED"
    )

    print(
        "NOTEBOOK 12 STATUS: COMPLETE — READY FOR HANDOFF"
    )

else:

    print(
        "12.9.5 STATUS: FAIL — INVESTIGATION REQUIRED"
    )

require_1295(
    total_failures_1295 == 0,
    "Post-export verification has failed checks."
)



A — POST-EXPORT VERIFICATION SUMMARY


,category,checks,failed,status
0,DATASET,13,0,PASS
1,DICTIONARY,4,0,PASS
2,DIRECTORY,6,0,PASS
3,DOCUMENTATION,4,0,PASS
4,MANIFEST,5,0,PASS



B — FAILED CHECKS
No failed verification checks.

C — FILE CHECKSUM VERIFICATION


,filename,size_match,sha256_match,status
0,behavior_features_r62_ml_final_new.parquet,True,True,PASS
1,feature_dictionary_r62_final.csv,True,True,PASS
2,feature_methodology_and_decisions.md,True,True,PASS
3,final_feature_qa_report.md,True,True,PASS



D — SAVED DATASET LABEL DISTRIBUTION


,y,user_days
0,0,378082
1,1,9080



12.9.5 — INDEPENDENT POST-EXPORT INTEGRITY
Export directory       : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\12_Feature_Engineering_and_Validation
Verified dataset rows  : 387162
Verified columns       : 143
Verified users         : 3737
Dictionary entries     : 143
Files inspected        : 5
SHA-256 checks         : 4
Integrity checks       : 32
Failed checks          : 0
Day datetime normalized: YES — comparison only
Dataset values match   : True
Dictionary matches     : True
Files modified         : NO
Files exported         : NO
12.9.5 STATUS: PASS — FINAL ARTIFACTS VERIFIED
NOTEBOOK 12 STATUS: COMPLETE — READY FOR HANDOFF
